Goals:
1. Transform a glossary (TB_PATHS) into spell-check dictionary (Hunspell .dic format with & without affix flags)
2. Filter out commons words available in standard Hunspell dictionary (HUNSPELL_PATHS).
3. Extend filtered dictionary with similar 

# HYPERPARAMETERS

# REFACTORING NOTES (2026-01-27)

**Coherency & Type Safety Improvements:**

1. **Imports Consolidated** ✅
   - Added `Dict` to typing imports for proper type annotations
   - Moved `re`, `glob`, and `itertools.product` to top-level imports
   - Removed duplicate imports within function definitions

2. **Complete Type Hints Added** ✅
   - All functions now have explicit return types: `Set[str]`, `List[str]`, `Tuple[...]`, `Dict[...]`
   - Parameter types fully annotated with proper generics
   - Internal helper functions typed for clarity
   - Return type strategy: `Set[str]` internally (fast, deduped), `List[str]` at export boundaries

3. **Preprocessing Functions Consolidated** ✅
   - Removed redundant `matches_time_pattern()` and `matches_digit_word_pattern()` - inlined into `tokenize_text()`
   - Kept single-purpose utilities: `remove_html_tags()`, `demorph_string()`, language-specific contractions
   - New utility: `normalize_language_code()` for standardizing language codes (es-es → es)

4. **Duplicate Code Removed** ✅
   - Fixed `process_xliff_file()`: removed duplicate WIP marker check
   - Consolidated `load_and_tokenize_terminology_base()` and `load_and_tokenize_terminology_base_to_file()` into single unified function with optional export

5. **Hyperparameter Integration** ✅
   - Updated `batch_filter_tokens_by_dictionary()` to use `OUTPUT_DIR` from hyperparameters when `output_folder` not specified
   - Added `INTERMEDIARY_DIR` usage recommendation for testing workflow
   - Language code normalization in batch processing for flexible input formats

6. **Documentation Enhanced** ✅
   - All functions have comprehensive docstrings with Args, Returns, Raises, and Examples
   - Processing workflow documented in function headers
   - Return type contracts clearly specified

7. **Consistency Verified** ✅
   - Function signatures aligned with confirmed hyperparameters (GAMES, TB_PATHS, HUNSPELL_PATHS, DIC_FOLDER, OUTPUT_DIR)
   - Language codes standardized to 2-letter lowercase format throughout pipeline
   - Filter options consistent across Excel and XLIFF processing

**Data Flow Architecture:**
```
Raw Input (Excel/XLIFF/TB_PATHS)
  ↓
[Preprocessing: HTML removal, morphological expansion, contractions]
  ↓ tokenize_text()
Set[str] Tokens (internal representation)
  ↓
[Filtering: Dictionary matching with affix rules]
  ↓ filter_tokens_by_dictionary_with_affixes()
List[str] Neologisms (exported, sorted)
  ↓
[Export: .txt flat list or .dic Hunspell format]
```

**Ready for Step 3-5 Implementation:** Flag generation, redundant form removal, and final assembly pipeline.

In [ ]:
import pandas as pd
import xml.etree.ElementTree as ET
import re
import os
import time
import glob
from pathlib import Path
from typing import Set, List, Tuple, Dict
from itertools import product
import html

GAMES = ["DOFUS", "WAKFU", "TOUCH", "WAVEN", "RETRO",
        "ONE_MORE_GATE", "ANKANIMATION"]
# Animation glossary not included by 1/2026

TB_PATHS = {
    "DOFUS" : "TB_ANK_202507/DOFUS_TB.csv",
    "WAKFU" : "TB_ANK_202507/WAKFU_TB.csv",
    "TOUCH" : "TB_ANK_202507/TOUCH_TB.csv",
    "WAVEN" : "TB_ANK_202507/WAVEN_TB.xlsx",
    "RETRO" : "TB_ANK_202507/Retro_TB.xlsx",
    "ONE_MORE_GATE" : "TB_ANK_202507/ONE_MORE_GATE_TB.xlsx",
    "ANKANIMATION": "TB_ANK_202507/Anim.xls"
}

# TM files have xliff extension, if not available or empty, let's use i18n_PATHS
TM_PATHS = {
    "DOFUS" : "",
    "WAKFU" : "",
    "TOUCH" : "",
    "WAVEN" : "",
    "RETRO" : "",
    "ONE_MORE_GATE" : "",
    "ANKANIMATION": ""
}

#In-game i18n strings https://github.com/dofusdude/dofus3-main/releases
i18n_PATHS = {
    "DOFUS" : {
        "fr": "i18n/DOFUS/fr.json",
        "en": "i18n/DOFUS/en.json",
        "es": "i18n/DOFUS/es.json",
        "pt": "i18n/DOFUS/pt.json",
        "de": "i18n/DOFUS/de.json"
    },
    "WAKFU" : {
        "fr": "i18n/WAKFU/texts_fr.properties",
        "en": "i18n/WAKFU/texts_en.properties",
        "es": "i18n/WAKFU/texts_es.properties",
        "pt": "i18n/WAKFU/texts_pt.properties"
    },
    "TOUCH" :  {
        "fr": "i18n/TOUCH/fr.json",
        "en": "i18n/TOUCH/en.json",
        "es": "i18n/TOUCH/es.json",
        "pt": "i18n/TOUCH/pt.json",
        "de": "i18n/TOUCH/de.json"
    },
    "WAVEN" : "",
    "RETRO" : "",
    "ONE_MORE_GATE" : "",
    "ANKANIMATION": ""
}

LANG_CODES = {
    "FRENCH" : "fr",
    "ENGLISH" : "en",
    "SPANISH" : "es",
    "PORTUGUESE" : "pt",
    "GERMAN" : "de"
}

DIC_FOLDER = "dics"
OUTPUT_DIR = "output_dics"          # for final dictionary folders
INTERMEDIARY_DIR = "processing_dics" # for intermediary files, cache and test outputs
OUTPUT_FLATLISTS_DIR = os.path.join(OUTPUT_DIR, "Flatlists")
OUTPUT_COMPRESSED_DIR = os.path.join(OUTPUT_DIR, "Compressed_dics")
OUTPUT_FULL_DIR = os.path.join(OUTPUT_DIR, "Full_dics")


HUNSPELL_PATHS = {
    "es": os.path.join(DIC_FOLDER, "es_dic", "es", "es_ES.dic"),
    "fr": os.path.join(DIC_FOLDER, "fr_dic", "fr.dic"),
    "pt": os.path.join(DIC_FOLDER, "pt_dic", "pt_BR", "pt_BR.dic"),
    "en": os.path.join(DIC_FOLDER, "en_dic", "en_GB.dic"),
    "de": os.path.join(DIC_FOLDER, "de_dic", "de_DE_frami.dic"),
}

# ── Proper-noun key patterns → sidecar category mapping ──────────────────────
# Maps a semantic category label to one or more regex patterns matched with
# re.search() against each comma-split TB / i18n key string.
#
# Pattern notes:
#   • D3 / RETRO / TOUCH  dot-notation    : monster.123.name
#   • WKF                 mixed notation  : Item.name.42 / Monster.name
#   • WAVEN               SCREAMING_SNAKE : ZONE_WORLD_1_2_NAME
#   • (?i) makes the match case-insensitive
#   • Only keys matching a pattern end up in the propernoun sidecar JSON.
#     Keys like "NPC.#.reply" are intentionally NOT listed → not tagged.
#
# Category labels consumed by:
#   • find_corpus_wordforms()    → entity categories trigger gender ghost forms
#   • munch_to_compressed_dic()  → category-specific mandatory flag assignment
#
# _ENTITY_CATEGORIES  : names of living entities → gender ghosts + possessive
# _AREA_CATEGORIES    : place names              → DE compound flags
PROPER_NOUN_KEY_PATTERNS: Dict[str, List[str]] = {
    "monster": [
        r"(?i)monster\.\d+\.name",                # D3, RETRO, TOUCH
        r"(?i)Monster\.name(?:\.\d+)?",           # WKF
    ],
    "npc": [
        r"(?i)NPC\.\d+\.name",                    # D3, TOUCH
        r"(?i)npc\.\d+\.name",                    # RETRO
        r"(?i)taxcollector\.\w+name\.\d+\.\w+",  # TOUCH tax collector
        r"(?i)document\.\d+\.author",             # D3 document authors
        r"(?i)Membre de clan\.name\.\d+",         # WKF clan members
        # WAVEN companions are NPCs
        r"COMPANION_\d+_NAME",                    # WAVEN
        r"GOD_\d+_NAME",                          # WAVEN
        r"CHARACTER_NAME_\d+_NAME",               # WAVEN
    ],
 
    "title": [
        r"(?i)title\.\d+\.(male|female)",         # D3 titles (gendered)
    ],
    "area": [
        r"(?i)area\.\d+\.name",                   # D3, RETRO
        r"(?i)subarea\.\d+\.name",                # D3, RETRO
        r"(?i)superarea\.\d+\.name",              # RETRO
        r"(?i)world\.subarea\.\d+\.name",         # D3
        r"(?i)world\.\w+\.\d+\.name",             # D3 (world.* catch-all)
        r"(?i)map\.name\.\d+\.name",              # D3
        #r"(?i)dungeon\.\d+\.name",                # RETRO
        r"(?i)hint\.\d+\.name",                   # D3, RETRO, TOUCH
        #r"(?i)interactive\.\d+\.name",            # D3
        r"(?i)Ambience Zone\.name\.\d+",          # WKF
        r"(?i)Territory_Zone\.name\.\d+",         # WKF
        r"(?i)Zaap\.\w+name\.\d+",               # WKF
        r"ZONE_\w+_\d+(?:_\d+)?_NAME",           # WAVEN
    ],

    "item": [
        r"(?i)Item\.name\.\d+",                   # WKF items
        r"(?i)object\.\d+\.name",                 # D3
    ],
}

# Semantic groupings used by munch_to_compressed_dic() for flag assignment
_ENTITY_CATEGORIES: Set[str] = {"monster", "npc", "title"}
_AREA_CATEGORIES:   Set[str] = {"area"}

# Pre-compiled patterns (built once at import time for speed)
import re as _re
_COMPILED_PROPER_NOUN_PATTERNS: Dict[str, list] = {
    cat: [_re.compile(p) for p in patterns]
    for cat, patterns in PROPER_NOUN_KEY_PATTERNS.items()
}

# ── Munch flag strategy per language ─────────────────────────────────────────
# mandatory     : assigned to ALL words unconditionally
# area_extra    : additional mandatory flags for area-category proper nouns only
#                 (e.g. DE COMPOUNDBEGIN/MID/END so place names can head compounds)
# entity_extra  : additional mandatory flags for entity-category proper nouns only
#                 (e.g. EN possessive M — not needed for place names)
# validation    : assigned when ≥ quorum fraction of flag-generated forms are
#                 in the confirmed corpus set (evaluated in _wordform_match_worker)
# verb          : like validation but only tested when add_verb_flags=True
MUNCH_FLAG_CONFIG: Dict = {
    "es": {
        "mandatory":    [],
        "area_extra":   [],
        "entity_extra": [],
        "validation":   ["S", "G"],
        "verb":         ["R", "E", "I"],
    },
    "de": {
        "mandatory":    ["S"],             # genitive -s for all DE proper nouns
        "area_extra":   ["x", "y", "z"],   # COMPOUNDBEGIN/MID/END only for place names
        "entity_extra": [],
        "validation":   ["F", "p", "P", "R", "N", "E"],
        "verb":         ["I", "X", "Y", "Z", "W"],
    },
    "en": {
        "mandatory":    [],
        "area_extra":   [],
        "entity_extra": ["M"],             # possessive 's only for entity names
        "validation":   ["S"],
        "verb":         ["D", "G", "d"],
    },
    "pt": {
        "mandatory":    [],
        "area_extra":   [],
        "entity_extra": [],
        "validation":   ["B", "D", "F"],
        "verb":         [],
    },
    "fr": {
        "mandatory":    [],
        "area_extra":   [],
        "entity_extra": [],
        "validation":   [],
        "verb":         [],
    },
}

basic_punct = (
    '.,;:¡!?"'           # standard punct (incl. straight double quote)
    '\u201c\u201d'       # " " (curly quotes)
    '\u2018\u2019'       # ' ' (single curly quotes)
    '()[]{}«»„\u201a'    # brackets and other punct
    '-+=*/@#$%^&|\\<>~`º°ª¿'
    '\u2026'             # … (ellipsis as single char)
)

unicode_dashes = '\u2014\u2013'  # em-dash and en-dash

# Workflow
Workflow to transform text input into a Hunspell dictionary

Input:
* Excel/CSV/TSV/TXT file with monolingual glossary or multilingual glossary;
* (Optional for each language) Monolingual corpus (JSON, TXT, EXCEL, CSV) or Bilingual corpus (JSON, XLIFF, EXCEL, CSV)
* Hunspell data folders with base dictionaries and affixes files

Output:
* Hunspell dictionary .dic (only new dic entries or combined with existing Hunspell dic) with affixes flags
* Flat list of words .txt (only neologisms or combination) without affixes flags

Languages to be supported:
* English (UK)
* Spanish (Spain)
* Portuguese (Brazil)
* French (France)
* German (Germany)

Steps per language:
1. Pre-process input files
2. Tokenize glossary
3. Filter out known words in tokenized_glossary from existing Hunspell dic --> Output flat list of words .txt
4. Find language-specific morphological affix flags from Hunspell AFF file within tokenized_glossary. If corpus available, find morph derivation of tokenized_glossary in corpus.



🔧 CORE FUNCTION 1: Enhanced Dictionary Generation

🔧 CORE FUNCTION 2: Redundant Form Removal

🔧 CORE FUNCTION 3: Flag Generation (FIXED VERSION)

🔧 CORE FUNCTION 4: Language-Specific Flag Identification

🔧 CORE FUNCTION 5: Enhanced Export with Cleanup



# FUNCTIONS

## 1. Pre-processing

In [2]:

# ==============================================================================
# LANGUAGE UTILITIES
# ==============================================================================

def normalize_language_code(code: str) -> str:
    """
    Normalize language codes to 2-letter lowercase format.
    
    Converts 'es-es', 'ES-ES', 'es_ES' -> 'es'; 'en-gb' -> 'en', etc.
    
    Args:
        code: Language code in any format (e.g., 'es-es', 'pt-br', 'en')
        
    Returns:
        str: 2-letter lowercase language code (e.g., 'es', 'pt', 'en')
        
    Raises:
        ValueError: If code cannot be normalized to a 2-letter code
    """
    if not code or not isinstance(code, str):
        raise ValueError(f"Invalid language code: {code}")
    
    # Normalize: lowercase, remove hyphens, underscores; take first 2 chars
    normalized = code.lower().replace('-', '').replace('_', '')
    if len(normalized) < 2:
        raise ValueError(f"Language code too short: {code}")
    
    return normalized[:2]


# ==============================================================================
# PRE-PROCESSING FUNCTIONS
# ==============================================================================

def remove_html_tags(text: str) -> str:
    """Remove HTML tags and decode HTML entities, with space insertion for br/p tags"""
    if not text:
        return text
    
    # First, replace br and p tags with spaces to prevent word concatenation
    # Handle both self-closing and regular br tags
    text = re.sub(r'&lt;/?br\s*/?&gt;', ' ', text, flags=re.IGNORECASE)
    text = re.sub(r'&lt;/?p\s*/?&gt;', ' ', text, flags=re.IGNORECASE)
    text = re.sub(r'&lt;p\s+[^&]*&gt;', ' ', text, flags=re.IGNORECASE)  # p with attributes
    text = re.sub(r'&lt;/p&gt;', ' ', text, flags=re.IGNORECASE)
    
    # Remove other HTML tags (without space insertion)
    text = re.sub(r'&lt;[^&]*&gt;', '', text)
    
    # Decode HTML entities
    text = html.unescape(text)
    
    # Clean up multiple spaces
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text

def process_english_contractions(text: str) -> str:
    """Process English contractions while preserving case"""
    if not text:
        return text
    
    # Comprehensive English contractions mapping
    contractions = {
        "ain't": "am not", "aren't": "are not", "can't": "cannot", "could've": "could have",
        "couldn't": "could not", "didn't": "did not", "doesn't": "does not", "don't": "do not",
        "hadn't": "had not", "hasn't": "has not", "haven't": "have not", "he'd": "he would",
        "he'll": "he will", "he's": "he is", "i'd": "i would", "i'll": "i will", "i'm": "i am",
        "i've": "i have", "isn't": "is not", "it'd": "it would", "it'll": "it will", "it's": "it is",
        "let's": "let us", "mustn't": "must not", "shan't": "shall not", "she'd": "she would",
        "she'll": "she will", "she's": "she is", "shouldn't": "should not", "that's": "that is",
        "there's": "there is", "they'd": "they would", "they'll": "they will", "they're": "they are",
        "they've": "they have", "we'd": "we would", "we're": "we are", "we've": "we have",
        "weren't": "were not", "what's": "what is", "where's": "where is", "who's": "who is",
        "won't": "will not", "wouldn't": "would not", "you'd": "you would", "you'll": "you will",
        "you're": "you are", "you've": "you have", "'cause": "because", "how's": "how is",
        "when's": "when is", "why's": "why is", "y'all": "you all", "would've": "would have",
        "should've": "should have", "might've": "might have", "must've": "must have"
    }
    
    def replace_contraction(match):
        contraction = match.group(0)
        lower_contraction = contraction.lower()
        
        if lower_contraction in contractions:
            replacement = contractions[lower_contraction]
            
            # Preserve case: if original was capitalized, capitalize the replacement
            if contraction[0].isupper():
                replacement = replacement.capitalize()
            
            return replacement
        return contraction
    
    # Use word boundaries to match contractions
    pattern = r"\b(?:" + "|".join(re.escape(cont) for cont in contractions.keys()) + r")\b"
    result = re.sub(pattern, replace_contraction, text, flags=re.IGNORECASE)
    
    return result

def process_portuguese_contractions(text: str) -> str:
    """Process Portuguese contractions and apostrophe patterns"""
    if not text:
        return text
    
    # Handle apostrophe contractions like d'Água -> de Água
    text = re.sub(r"\bd'([A-ZÁÉÍÓÚÂÊÔÀÇ])", r"de \1", text)
    text = re.sub(r"\bl'([A-ZÁÉÍÓÚÂÊÔÀÇ])", r"le \1", text)
    
    # Handle hyphenated pronouns like amá-lo -> amar lo
    text = re.sub(r"([aeiouáéíóúâêôàç])-([lm][eoasá]s?)\b", r"\1r \2", text)
    
    return text

def process_french_elisions(text: str) -> str:
    """Strip common French elided clitic prefixes before tokenization."""
    if not text:
        return text

    # Keep lexical apostrophes like A'geuse, but normalize l', s', qu' prefixes.
    return re.sub(
    r"\b(?:[cdjlmnst]|qu)['’](?=[AEIOUYÀÂÄÆÉÈÊËÎÏÔÖÙÛÜŒHaeiouyàâäæéèêëîïôöùûüœh])",
    "",
    text,
    flags=re.IGNORECASE
)

def has_wip_markers(text: str) -> bool:
    """Check if text contains WIP/translation markers"""
    if not text:
        return False
    if "[!]" in text:
        return True
    # Pattern to match markers like {WIP}, [NOTRAD], [no trad], {no_trad}, etc.
    pattern = r'[\[\{].*(wip|notrad|no trad|no_trad|no-trad).*[\]\}]'
    return bool(re.search(pattern, text, re.IGNORECASE))

def demorph_string(input_string: str) -> str:
    """
    Expand morphological patterns in localization strings.
    
    Supports two pattern types:
    1. Tilde patterns: {~X...} where X is a letter and ... is suffix
    2. Square bracket patterns: {[N*]?option1:option2} where N is a digit
    
    Args:
        input_string (str): String containing morphological patterns
        
    Returns:
        str: String with all variations joined by spaces
    """
    
    def extract_tilde_patterns(text: str) -> List[tuple]:
        """Extract all tilde morphological patterns from a word."""
        pattern_regex = r'\{~([^}]+)\}'
        matches = re.findall(pattern_regex, text)
        parsed_patterns = []
        for match in matches:
            # Split by ~ to handle multiple patterns in the same braces
            sub_patterns = match.split('~')
            for sub_pattern in sub_patterns:
                if len(sub_pattern) >= 1:
                    letter = sub_pattern[0]
                    suffix = sub_pattern[1:] if len(sub_pattern) > 1 else ""
                    parsed_patterns.append((letter, suffix))
        return parsed_patterns
    
    def extract_bracket_patterns(text: str) -> List[tuple]:
        """Extract all bracket patterns from a word."""
        # Pattern: {[N*]?option1:option2} — N may carry a comparison operator:
        #   {[>1]?a:o}   {[<1]?s:}   {[=1]?:s}   {[1*]?...}   {[~1]?...}
        pattern_regex = r'\{\[([~]?[<>=]?\d+\*?)\]\?([^:}]*):([^}]*)\}'
        matches = re.findall(pattern_regex, text)
        return matches
    
    def generate_tilde_variations(base_word: str, patterns: List[tuple]) -> List[str]:
        """Generate variations for tilde patterns."""
        # Remove patterns from base word to get the root
        root = re.sub(r'\{~[^}]+\}', '', base_word)
        
        # Check if root should be excluded (if 's' or 'm' patterns present)
        pattern_letters = [p[0] for p in patterns]
        exclude_root = 's' in pattern_letters or 'm' in pattern_letters
        
        # If no patterns, return the original word
        if not patterns:
            return [base_word]
        
        variations = []
        
        # Group patterns by type
        gender_patterns = [(letter, suffix) for letter, suffix in patterns if letter in 'mf']
        number_patterns = [(letter, suffix) for letter, suffix in patterns if letter in 'sp']
        
        # Handle gender+number combinations
        if gender_patterns and number_patterns:
            # We need all 4 combinations: masc sing, fem sing, masc plural, fem plural
            
            # 1. Masculine singular (root) - only if not excluded
            if not exclude_root:
                variations.append(root)

            # 2. Masculine singular with masculine suffix
            for g_letter, g_suffix in gender_patterns:
                if g_letter == 'm':
                    male_root = root + g_suffix
                    variations.append(male_root)

            # 3. Feminine singular (root + feminine suffix)
            for g_letter, g_suffix in gender_patterns:
                if g_letter == 'f':
                    variations.append(root + g_suffix)
            
            # 4. Masculine plural (root + plural suffix)  
            for n_letter, n_suffix in number_patterns:
                if n_letter == 'p':
                    variations.append(root + n_suffix)
            
            # 5. Feminine plural (root + feminine suffix + plural suffix)
            for (g_letter, g_suffix), (n_letter, n_suffix) in product(gender_patterns, number_patterns):
                if g_letter == 'f' and n_letter == 'p':
                    variations.append(root + g_suffix + n_suffix)
                    
        else:
            # Handle simple cases (no combinations needed)
            
            # If root should be included, add it first
            if not exclude_root:
                variations.append(root)
            
            # Add individual pattern variations
            for letter, suffix in patterns:
                variation = root + suffix
                variations.append(variation)
        
        # Remove duplicates while preserving order
        seen = set()
        unique_variations = []
        for var in variations:
            if var not in seen:
                seen.add(var)
                unique_variations.append(var)
        
        return unique_variations
    
    def generate_bracket_variations(base_word: str, bracket_patterns: List[tuple]) -> List[str]:
        """Generate variations for bracket patterns."""
        if not bracket_patterns:
            return [base_word]
        
        current_variations = [base_word]
        
        for pattern_match, option1, option2 in bracket_patterns:
            new_variations = []
            
            # Build the regex pattern correctly
            pattern_to_replace = r'\{\['  # {[
            pattern_to_replace += re.escape(pattern_match)  # pattern (escaped)
            pattern_to_replace += r'\]\?'  # ]?
            pattern_to_replace += re.escape(option1)  # option1 (escaped)
            pattern_to_replace += ':'  # :
            pattern_to_replace += re.escape(option2)  # option2 (escaped)
            pattern_to_replace += r'\}'  # }
            
            for current_var in current_variations:
                # For the pattern {[N*]?option1:option2}:
                # Generate variation 1: condition true -> use option1 (usually the base/unmarked form)
                # Use lambda replacement to avoid re.sub treating backslashes in option1/option2
                # as regex escape sequences (e.g. trailing \ in WAKFU strings causes "bad escape")
                var1 = re.sub(pattern_to_replace, lambda m, r=option1: r, current_var, count=1)
                if var1 not in new_variations:
                    new_variations.append(var1)
                
                # Generate variation 2: condition false -> use option2 (usually the marked form)
                var2 = re.sub(pattern_to_replace, lambda m, r=option2: r, current_var, count=1)
                if var2 not in new_variations:
                    new_variations.append(var2)
            
            current_variations = new_variations
        

        return current_variations
        
    # Find all words with patterns (both types); \S* at both ends to capture
    # any prefix/suffix chars like the trailing 's' in Ocult{[>1]?a:o}s
    word_pattern_regex = r'\S*\{[~\[][^}]+\}(?:\{[~\[][^}]+\})*\S*'
    
    def replace_word_patterns(match) -> str:
        word_with_patterns = match.group(0)
        
        # Check what type of patterns we have
        bracket_patterns = extract_bracket_patterns(word_with_patterns)
        tilde_patterns = extract_tilde_patterns(word_with_patterns)
        
        if bracket_patterns and not tilde_patterns:
            # Only bracket patterns
            variations = generate_bracket_variations(word_with_patterns, bracket_patterns)
        elif tilde_patterns and not bracket_patterns:
            # Only tilde patterns
            variations = generate_tilde_variations(word_with_patterns, tilde_patterns)
        elif bracket_patterns and tilde_patterns:
            # Both types - handle bracket first, then tilde
            bracket_variations = generate_bracket_variations(word_with_patterns, bracket_patterns)
            final_variations = []
            for var in bracket_variations:
                if extract_tilde_patterns(var):
                    tilde_vars = generate_tilde_variations(var, extract_tilde_patterns(var))
                    final_variations.extend(tilde_vars)
                else:
                    final_variations.append(var)
            variations = final_variations
        else:
            # No patterns found (shouldn't happen with our regex)
            variations = [word_with_patterns]
        
        return ' '.join(variations)
    
    # Replace all pattern words with their variations
    result = re.sub(word_pattern_regex, replace_word_patterns, input_string)
    
    return result

def tokenize_text(text: str, language: str = "default") -> Set[str]:
    """
    Enhanced tokenize function with language-specific processing and comprehensive filtering
    
    Args:
        text: Input text to tokenize
        language: Language for processing ("english", "portuguese", "french", or "default")
    
    Returns:
        Set[str]: Set of filtered tokens (deduplicated, no particular order)
    """
    if not text or not isinstance(text, str):
        return set()
    
    # Step 1: Remove HTML tags and decode entities
    text = remove_html_tags(text)

    # Step 1.25: Normalize escaped control sequences from i18n/properties sources
    # Example: "...\nUNA..." stored as literal backslash+n becomes "... nUNA ..."
    # and creates bad tokens like "nUNA". Convert to whitespace first.
    text = re.sub(r'\\+[nrt]', ' ', text)

    # Step 1.5: Expand morphological patterns if { or [ detected
    if '{' in text or '[' in text:
        text = demorph_string(text)
    
    # Step 2: Remove URLs and email addresses
    text = re.sub(r'http[s]?://(?:[a-zA-Z]|[0-9]|[$-_@.&+]|[!*\\(\\),]|(?:%[0-9a-fA-F][0-9a-fA-F]))+', '', text)
    text = re.sub(r'\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Z|a-z]{2,}\b', '', text)
    
    # Step 3: Language-specific contraction processing
    if language.lower() == "english":
        text = process_english_contractions(text)
    elif language.lower() == "portuguese":
        text = process_portuguese_contractions(text)
    elif language.lower() == "french":
        text = process_french_elisions(text)
    # For "default" or other languages, skip contraction processing
    
    # Step 4: Enhanced punctuation (including º character)
    punctuation = basic_punct + unicode_dashes
    
    # Step 5: Tokenize by whitespace and punctuation, preserving internal hyphens and apostrophes
    tokens = re.findall(r"[^\s" + re.escape(punctuation) + r"]+(?:[-'][^\s" + re.escape(punctuation) + r"]+)*", text)
    
    # Step 6: Clean and filter tokens
    filtered_tokens = set()
    for token in tokens:
        # Remove leading/trailing apostrophes and hyphens
        cleaned_token = token.strip("'-")

        # Remove English possessive 's (e.g., "Anear's" → "Anear")
        if language.lower() == "english":
            cleaned_token = re.sub(r"'[sS]$", "", cleaned_token)
        elif language.lower() == "french":
            cleaned_token = process_french_elisions(cleaned_token)
        
        # Skip if empty after cleaning
        if not cleaned_token:
            continue
        
        # Skip short tokens (< 3 characters)
        if len(cleaned_token) < 3:
            continue
        
        # Skip tokens that are chains of the same character
        if len(set(cleaned_token.lower())) == 1:
            continue
        
        # Skip tokens that are only digits
        if cleaned_token.isdigit():
            continue
        
        # Skip time patterns (e.g., "3PM", "10AM", "5PA", "12AL") - inlined for efficiency
        if re.match(r'^\d+(PM|AM|PA|AL|MP|AP|RW|BP|KT)$', cleaned_token, re.IGNORECASE):
            continue
        
        # Skip digit-word patterns (e.g., "123-neutral") - inlined for efficiency
        if re.match(r'^\d+-\w+$', cleaned_token):
            continue
        
        filtered_tokens.add(cleaned_token)
    
    return filtered_tokens

def detect_file_type(file_path: str) -> str:
    """
    Detect if file is Excel, CSV, or XLIFF based on extension.
    
    Args:
        file_path: Path to the file
        
    Returns:
        str: 'excel', 'csv', or 'xliff'
        
    Raises:
        ValueError: If file type is unsupported
    """
    file_path_lower = file_path.lower()
    if file_path_lower.endswith(('.xlsx', '.xls')):
        return 'excel'
    elif file_path_lower.endswith('.csv'):
        return 'csv'
    elif file_path_lower.endswith(('.tsv', '.txt')):
        return 'tsv'
    elif file_path_lower.endswith(('.xliff', '.xlf', '.xml')):
        return 'xliff'
    else:
        raise ValueError(f"Unsupported file type for: {file_path}")

def load_dataframe(file_path: str) -> pd.DataFrame:
    """
    Load a tabular file into a DataFrame, auto-detecting format (Excel, CSV, TSV).
    
    Args:
        file_path: Path to the file (.xlsx, .xls, .csv, .tsv, .txt)
        
    Returns:
        pd.DataFrame: Loaded data
        
    Raises:
        ValueError: If the file cannot be read or is unsupported
    """
    file_type = detect_file_type(file_path)
    
    if file_type == 'excel':
        return pd.read_excel(file_path)
    elif file_type == 'csv':
        # utf-8-sig strips BOM (\ufeff) that Excel adds when saving UTF-8 CSVs
        try:
            return pd.read_csv(file_path, encoding='utf-8-sig', sep=None, engine='python')
        except UnicodeDecodeError:
            return pd.read_csv(file_path, encoding='latin-1', sep=None, engine='python')
    elif file_type == 'tsv':
        try:
            return pd.read_csv(file_path, encoding='utf-8-sig', sep='\t')
        except UnicodeDecodeError:
            return pd.read_csv(file_path, encoding='latin-1', sep='\t')
    else:
        raise ValueError(f"Cannot load unsupported file type: {file_path}")

def process_excel_file(file_path: str, language_code: str, ignore_identical_translation: bool, 
                      tokenize_language: str, skip_square_brackets: bool, skip_all_caps: bool, 
                      skip_wip_markers: bool) -> Tuple[Set[str], int, int]:
    """
    Process Excel/CSV file and extract tokens with configurable filtering.
    
    Args:
        file_path: Path to Excel or CSV file
        language_code: Target language code (e.g., 'es', 'en')
        ignore_identical_translation: Skip entries where source == target
        tokenize_language: Language for tokenization ("english", "portuguese", or "default")
        skip_square_brackets: Skip entries with [brackets] in source
        skip_all_caps: Skip entries with ALL_CAPS in target
        skip_wip_markers: Skip entries with WIP markers
        
    Returns:
        Tuple[Set[str], int, int]: (tokens_set, processed_count, skipped_count)
    """
    file_type = detect_file_type(file_path)
    
    if file_type == 'excel':
        # Try to find the sheet with actual data for the language
        xl_file = pd.ExcelFile(file_path)
        df = None
        sheet_used = None
        
        for sheet_name in xl_file.sheet_names:
            temp_df = pd.read_excel(file_path, sheet_name=sheet_name)
            if language_code in temp_df.columns:
                non_null_count = temp_df[language_code].notna().sum()
                if non_null_count > 0:
                    df = temp_df
                    sheet_used = sheet_name
                    print(f"Using sheet '{sheet_name}' with {non_null_count} {language_code} values")
                    break
        
        if df is None:
            df = pd.read_excel(file_path)
            sheet_used = "default"
    else:
        # CSV / TSV
        df = load_dataframe(file_path)
        sheet_used = "n/a"
    
    print(f"Columns: {list(df.columns)}")
    print(f"Sheet used: {sheet_used}")
    
    if language_code not in df.columns:
        raise ValueError(f"Language code '{language_code}' not found in columns: {list(df.columns)}")
    # Initialize tracking
    print(f"Total rows to process: {len(df)}")
    
    # Initialize tracking
    tokens = set()
    processed_count = 0
    skipped_count = 0
    skip_reasons = {"identical": 0, "square_brackets": 0, "all_caps": 0, "wip_markers": 0, "empty_target": 0}
    
    for index, row in df.iterrows():
        source_text = str(row.iloc[1]) if len(row) > 1 else ""  # Assume source is second column
        
        # Check if target is NaN or empty BEFORE converting to string
        target_value = row[language_code]
        if pd.isna(target_value):
            skipped_count += 1
            skip_reasons["empty_target"] += 1
            continue
            
        target_text = str(target_value)
        
        # Skip if target is empty string after conversion
        if target_text.strip() == '':
            skipped_count += 1
            skip_reasons["empty_target"] += 1
            continue
        
        # Apply filters based on configuration
        should_skip = False
        skip_reason = None
        
        # Filter 1: Identical translation
        if ignore_identical_translation and source_text == target_text:
            should_skip = True
            skip_reason = "identical"
        
        # Filter 2: Square brackets in source
        elif skip_square_brackets and re.search(r'\[.+\]', source_text):
            should_skip = True
            skip_reason = "square_brackets"
        
        # Filter 3: All caps target
        elif skip_all_caps and target_text.isupper() and len(target_text) > 2:
            should_skip = True
            skip_reason = "all_caps"
        
        # Filter 4: WIP markers
        elif skip_wip_markers and has_wip_markers(target_text):
            should_skip = True
            skip_reason = "wip_markers"
        
        if should_skip:
            skipped_count += 1
            skip_reasons[skip_reason] += 1
            continue
        
        # Process the target text
        processed_count += 1
        text_tokens = tokenize_text(target_text, tokenize_language)
        tokens.update(text_tokens)
    
    # Print skip statistics
    print(f"Skip reasons breakdown:")
    for reason, count in skip_reasons.items():
        if count > 0:
            print(f"  - {reason}: {count}")
    
    return tokens, processed_count, skipped_count

def process_xliff_file(file_path: str, language_code: str, ignore_identical_translation: bool,
                      tokenize_language: str, skip_square_brackets: bool, skip_all_caps: bool,
                      skip_wip_markers: bool) -> Tuple[Set[str], int, int]:
    """
    Process XLIFF file and extract tokens with configurable filtering.
    
    Args:
        file_path: Path to XLIFF file
        language_code: Target language code (e.g., 'es', 'en')
        ignore_identical_translation: Skip entries where source == target
        tokenize_language: Language for tokenization ("english", "portuguese", or "default")
        skip_square_brackets: Skip entries with [brackets] in source
        skip_all_caps: Skip entries with ALL_CAPS in target
        skip_wip_markers: Skip entries with WIP markers
    
    Returns:
        Tuple[Set[str], int, int]: (tokens_set, processed_count, skipped_count)
    """
    
    tree = ET.parse(file_path)
    root = tree.getroot()
    
    # Find the namespace
    namespace = ''
    if root.tag.startswith('{'):
        namespace = root.tag.split('}')[0] + '}'
    
    # Find file element and check language attributes
    file_elem = root.find(f'.//{namespace}file')
    if file_elem is None:
        raise ValueError("No file element found in XLIFF")
    
    source_lang = file_elem.get('source-language', '')
    target_lang = file_elem.get('target-language', '')
    
    print(f"XLIFF source language: {source_lang}")
    print(f"XLIFF target language: {target_lang}")
    
    # Determine if we should extract from source or target elements
    use_source = (language_code == source_lang)
    use_target = (language_code == target_lang)
    
    if not (use_source or use_target):
        raise ValueError(f"Language code '{language_code}' not found in XLIFF languages: {source_lang}, {target_lang}")
    
    # Find all trans-unit elements
    trans_units = root.findall(f'.//{namespace}trans-unit')
    print(f"Total XLIFF segments to process: {len(trans_units)}")
    
    # Initialize tracking
    tokens = set()
    processed_count = 0
    skipped_count = 0
    skip_reasons = {"identical": 0, "square_brackets": 0, "all_caps": 0, "wip_markers": 0}
    
    for trans_unit in trans_units:
        source_elem = trans_unit.find(f'{namespace}source')
        target_elem = trans_unit.find(f'{namespace}target')
        
        source_text = source_elem.text if source_elem is not None and source_elem.text else ""
        target_text = target_elem.text if target_elem is not None and target_elem.text else ""
        
        # Determine which text to process
        text_to_process = source_text if use_source else target_text
        
        # Skip if text is empty
        if not text_to_process:
            skipped_count += 1
            continue
        
        # Apply filters based on configuration
        should_skip = False
        skip_reason = None
        
        # Filter 1: Identical translation (only relevant for target)
        if ignore_identical_translation and use_target and source_text == target_text:
            should_skip = True
            skip_reason = "identical"
        
        # Filter 2: Square brackets in source
        elif skip_square_brackets and re.search(r'\[.+\]', source_text):
            should_skip = True
            skip_reason = "square_brackets"
        
        # Filter 3: All caps target (only relevant for target)
        elif skip_all_caps and use_target and target_text.isupper() and len(target_text) > 2:
            should_skip = True
            skip_reason = "all_caps"
        
        # Filter 4: WIP markers (consolidated - removed duplicate check)
        elif skip_wip_markers and has_wip_markers(target_text):
            should_skip = True
            skip_reason = "wip_markers"
        
        if should_skip:
            skipped_count += 1
            skip_reasons[skip_reason] += 1
            continue
        
        # Process the text
        processed_count += 1
        text_tokens = tokenize_text(text_to_process, tokenize_language)
        tokens.update(text_tokens)
    
    # Print skip statistics
    print(f"Skip reasons breakdown:")
    for reason, count in skip_reasons.items():
        if count > 0:
            print(f"  - {reason}: {count}")
    
    return tokens, processed_count, skipped_count

def export_tokens_to_txt(tokens: Set[str], output_path: str) -> None:
    """
    Export tokens to a text file, one per line, sorted alphabetically.
    
    Args:
        tokens: Set of tokens to export
        output_path: Path where to save the file
    """
    with open(output_path, 'w', encoding='utf-8') as f:
        for token in sorted(tokens):
            f.write(token + '\n')
    print(f"Exported {len(tokens)} unique tokens to: {output_path}")

def create_sample_xliff() -> None:
    """Create a sample XLIFF file for testing"""
    sample_xliff_content = """<?xml version="1.0" encoding="UTF-8"?>
<xliff version="1.2" xmlns="urn:oasis:names:tc:xliff:document:1.2">
    <file datatype="plaintext" original="sample" source-language="fr-fr" target-language="es-es">
        <body>
            <trans-unit id="sample.1">
                <source>Votre alignement est probablement au sommet, vos ennemis n'existent plus à l'Apogée.</source>
                <target>Tu alineamiento está probablemente en la cumbre, tus enemigos no existen en el Apogeo.</target>
            </trans-unit>
            <trans-unit id="sample.2">
                <source>Test avec des crochets [DEBUG] dans le source</source>
                <target>Prueba con corchetes en el origen</target>
            </trans-unit>
        </body>
    </file>
</xliff>"""
    
    with open("sample.xliff", "w", encoding="utf-8") as f:
        f.write(sample_xliff_content)
    print("Sample XLIFF file created!")
    print("Sample XLIFF file created!")


# LOAD AND TOKENIZE TERMINOLOGY BASE
# ==============================================================================

# Regex that matches typical language-code column headers:
#   "es", "es-ES", "es_ES", "pt-BR", "fr-FR", "de-DE (info)", etc.
# Two lowercase letters, optionally followed by [-_] + two letters, optionally " (info)".
_LANG_COL_RE = re.compile(
    r'^[a-z]{2}(?:[-_][a-zA-Z]{2})?(?:\s+\(info\))?$',
    re.IGNORECASE
)

def _is_lang_col(col: str) -> bool:
    """Return True if the column name looks like a language-code column."""
    return bool(_LANG_COL_RE.match(col.strip().lstrip('\ufeff')))
    

# Cache: (game, lang, filtered_path, mtime, size) -> token evidence map
_ES_I18N_CASE_EVIDENCE_CACHE: Dict[Tuple[str, str, str, float, int], Dict[str, Dict[str, int]]] = {}
_ES_WORD_RE = re.compile(r"[A-Za-zÀ-ÖØ-öø-ÿÑñÜü][A-Za-zÀ-ÖØ-öø-ÿÑñÜü'’-]*")

def _is_sentence_start_boundary(text: str, pos: int) -> bool:
    """True when token at pos starts a sentence-like segment."""
    i = pos - 1
    while i >= 0 and text[i].isspace():
        i -= 1
    if i < 0:
        return True
    return text[i] in '.!?:;'

def _resolve_filtered_i18n_path(game: str, lang: str) -> str:
    """Return filtered i18n path for game/lang or empty string."""
    path = (i18n_PATHS.get(game) or {}).get(lang, '')
    if not path or not os.path.exists(path):
        return ''
    base = os.path.basename(path)
    if base in (f"{game}_{lang}_i18n_filtered.json", f"{game}_{lang}_i18n_filtered.properties"):
        return path
    return ''

def _build_es_i18n_case_evidence(game: str, lang: str = 'es') -> Dict[str, Dict[str, int]]:
    """Build lowercase/uppercase-mid-sentence evidence from filtered i18n only."""
    if lang != 'es':
        return {}
    filtered_path = _resolve_filtered_i18n_path(game, lang)
    if not filtered_path:
        return {}
    try:
        _st = os.stat(filtered_path)
    except OSError:
        return {}
    _cache_key = (game, lang, filtered_path, float(_st.st_mtime), int(_st.st_size))
    if _cache_key in _ES_I18N_CASE_EVIDENCE_CACHE:
        return _ES_I18N_CASE_EVIDENCE_CACHE[_cache_key]

    evidence: Dict[str, Dict[str, int]] = {}

    def _scan_text(raw_text: str) -> None:
        clean = demorph_string(remove_html_tags(str(raw_text)))
        for m in _ES_WORD_RE.finditer(clean):
            tok_raw = m.group(0).strip("'’-")
            if not tok_raw:
                continue
            tok_lower = tok_raw.lower()
            rec = evidence.setdefault(tok_lower, {
                'lowercase_count': 0,
                'uppercase_mid_sentence_count': 0,
            })
            if tok_raw == tok_raw.lower():
                rec['lowercase_count'] += 1
            if tok_raw[:1].isupper() and not _is_sentence_start_boundary(clean, m.start()):
                rec['uppercase_mid_sentence_count'] += 1

    try:
        if filtered_path.endswith('.json'):
            with open(filtered_path, 'r', encoding='utf-8') as fh:
                data = json.load(fh)
            entries = data.get('entries', data)
            for raw_val in entries.values() if isinstance(entries, dict) else []:
                if isinstance(raw_val, str) and raw_val.strip():
                    _scan_text(raw_val)
        elif filtered_path.endswith('.properties'):
            with open(filtered_path, 'r', encoding='utf-8') as fh:
                for line_raw in fh:
                    line = line_raw.rstrip('\n')
                    if not line.strip() or line.lstrip().startswith('#') or '=' not in line:
                        continue
                    _, _, raw_val = line.partition('=')
                    if raw_val.strip():
                        _scan_text(raw_val)
    except Exception:
        return {}

    _ES_I18N_CASE_EVIDENCE_CACHE[_cache_key] = evidence
    return evidence
    

_ANK_PT_EXCEPTION = "[S'il s'agît d'un typo - Armand - son nom en portugais est ARMANDO]"
_ANK_WORD_RE = re.compile(r"[A-Za-zÀ-ÖØ-öø-ÿÑñÜü]+(?:[-'’][A-Za-zÀ-ÖØ-öø-ÿÑñÜü]+)*")

def _ank_title_case_words(text: str) -> str:
    """Title-case lexical words while preserving punctuation separators."""
    def _title_word(match: re.Match) -> str:
        word = match.group(0)
        return word[:1].upper() + word[1:].lower() if word else word
    return _ANK_WORD_RE.sub(_title_word, text)

def _preprocess_ankanimation_term(raw_value: object, lang_prefix: str) -> str:
    """Normalize ANKANIMATION TB term comments/variants/casing."""
    if pd.isna(raw_value):
        return ''

    text = str(raw_value).strip()
    if not text:
        return ''

    # PT exception must remain untouched.
    if lang_prefix == 'pt' and text == _ANK_PT_EXCEPTION:
        # Keep only the intended Portuguese term, avoid noisy French sentence tokens.
        return 'Armando'

    # If term starts with a bracketed note, keep first lexical token from inside.
    stripped = text.lstrip()
    if stripped.startswith('['):
        m_first = re.match(r"^\s*\[([^\]]*)\]", text)
        if m_first:
            inner = m_first.group(1).strip()
            m_tok = _ANK_WORD_RE.search(inner)
            text = m_tok.group(0) if m_tok else ''

    # Remove all remaining [comments].
    text = re.sub(r"\[[^\]]*\]", " ", text)

    # Remove parenthetical variants/comments.
    text = re.sub(r"\([^)]*\)", " ", text)

    # Cleanup spacing and stray separator padding.
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"\s*([,;:])\s*", r"\1 ", text).strip()
    text = text.rstrip(' ,;:')

    if not text:
        return ''

    # Normalize all-caps content to title case per lexical word.
    letters_only = re.sub(r"[^A-Za-zÀ-ÖØ-öø-ÿÑñÜü]", "", text)
    if letters_only and letters_only.isupper():
        text = _ank_title_case_words(text)

    return text

def _preprocess_ankanimation_dataframe(df: pd.DataFrame, term_column: str, lang_prefix: str) -> Tuple[pd.DataFrame, Dict[str, int]]:
    """Apply ANKANIMATION text cleanup on a term column and report stats."""
    out_df = df.copy()
    before_series = out_df[term_column].astype(str)
    out_df[term_column] = out_df[term_column].apply(lambda v: _preprocess_ankanimation_term(v, lang_prefix))
    after_series = out_df[term_column].astype(str)

    stats = {
        'total_rows': int(len(out_df)),
        'changed_rows': int((before_series != after_series).sum()),
        'empty_after': int((after_series.str.strip() == '').sum()),
    }
    return out_df, stats

def _apply_ankanimation_token_overrides(tokens: Set[str], lang_prefix: str) -> Set[str]:
    """Apply ANKANIMATION token overrides before dictionary filtering/export."""
    out = {t for t in tokens if isinstance(t, str) and t}

    if lang_prefix == 'es':
        # 2) Force lowercase-only for this list (remove capitalized/other variants).
        force_lower_only = [
            "Brakmarianos", "Dragopavo", "Fab'huritus", "Núcrumos", "Selocosa", "Sadidas"
        ]
        for base in force_lower_only:
            base_cf = base.casefold()
            out = {tok for tok in out if tok.casefold() != base_cf}
            out.add(base.lower())

        # 2B) Keep capitalized and add lowercase.
        keep_both = ["Selacubo", "Selaculus", "Selasfera"]
        for base in keep_both:
            out.add(base)
            out.add(base.lower())

        # 3) Replace Mechasme -> Mecasma, and add mecasma/mecasmas if source had Mechasme.
        if any(tok.casefold() == 'mechasme' for tok in out):
            out = {tok for tok in out if tok.casefold() != 'mechasme'}
            out.update({'Mecasma', 'mecasma', 'mecasmas'})

    return out


def load_and_tokenize_terminology_base(excel_file_path: str, language_code: str,
                                      term_column: str = None, tokenize_language: str = "default",
                                      output_file_path: str = None,
                                      save_propernoun_sidecar: bool = True,
                                      game_tag: str = "",
                                      allow_language_fallback: bool = True) -> Tuple[List[str], str]:
    """
    Load terminology base from Excel/CSV/TSV, tokenize it, and optionally export to file.

    This unified function:
    1. Loads a terminology file (Excel, CSV, or TSV)
    2. Pre-filters rows: drops emote shortcut entries and entries flagged as morse
    3. Extracts terms from the specified language column
    4. For WAVEN-style files, also extracts plural forms from the matching "(info)" column
    5. Tokenizes each term using the custom tokenize_text() function
    6. Deduplicates and returns as a sorted list
    7. (Optional) Exports to a text file
    8. (Optional) Writes a proper-noun sidecar JSON for gender ghost generation

    Args:
        excel_file_path: Path to terminology file (.xlsx, .xls, .csv, .tsv)
        language_code: Language code (e.g., "es-es", "pt-br"). Auto-normalized to 2 letters
        term_column: Column name containing terms. If None, auto-detects based on language_code
        tokenize_language: Language for tokenization ("english", "portuguese", or "default")
        output_file_path: Path to save the tokenized list. If None, no export occurs
        save_propernoun_sidecar: If True, writes a JSON mapping key-types in
            PROPER_NOUN_KEY_PATTERNS to their token sets, for use by find_corpus_wordforms().
        game_tag: Game name tag (e.g. "DOFUS") used in the sidecar filename.
        allow_language_fallback: If True, fallback to last language column when
            exact language column is missing. If False, raises ValueError.

    Returns:
        Tuple[List[str], str]: (tokenized_list_sorted, output_file_path_or_empty)

    Raises:
        FileNotFoundError: If file does not exist
        ValueError: If language code or term column cannot be detected

    Example:
        >>> tokens, output_path = load_and_tokenize_terminology_base(
        ...     "terminology.csv",
        ...     "es-es",
        ...     tokenize_language="default",
        ...     output_file_path="terminology_tokens_es.txt"
        ... )
        >>> len(tokens)
        245
    """

    if not os.path.exists(excel_file_path):
        raise FileNotFoundError(f"File not found: {excel_file_path}")
    # Load file (auto-detects Excel vs CSV/TSV)
    print(f"📂 Loading terminology from: {excel_file_path}")

    # Load file (auto-detects Excel vs CSV/TSV)
    try:
        df = load_dataframe(excel_file_path)
    except Exception as e:
        raise ValueError(f"Error reading file: {e}")

    print(f"📊 Shape: {df.shape} (rows, columns)")
    print(f"📋 Columns: {list(df.columns)}")

    # Normalized language code reused across auto-detection and ES-specific
    # item/monster first-token lowercasing logic.
    lang_prefix = normalize_language_code(language_code)
    _is_spanish = (lang_prefix == 'es')

    # If corpus loader is already available, prewarm filtered i18n so
    # ES default-lowercasing can use corpus evidence in the same run.
    if _is_spanish and game_tag and 'load_i18n_corpus' in globals():
        try:
            load_i18n_corpus(lang_prefix, [game_tag], source_type='i18n', lang_detect=True)
        except Exception as _e:
            print(f"⚠️  [es-first-lower] i18n prewarm skipped for {game_tag}/{lang_prefix}: {_e}")

    def _extract_first_token_lower(text: str) -> str:
        """Return first lexical token lowercased, preserving accents/apostrophes."""
        m = re.search(r"[A-Za-zÀ-ÖØ-öø-ÿÑñÜü][A-Za-zÀ-ÖØ-öø-ÿÑñÜü'’-]*", text)
        if not m:
            return ''
        tok = m.group(0).strip("'’-")
        return tok.lower() if tok else ''

    # ── Identify key/note column ───────────────────────────────────────────────
    # Priority 1: an explicit key/definition/id/note column name.
    # Priority 2: the first column that does NOT look like a language column.
    # Language columns are NEVER used as the key column so we never corrupt them.
    _KEY_COL_NAMES = {'key', 'definition', 'id', 'note'}
    _named_key = next(
        (col for col in df.columns if col.strip().lower().lstrip('\ufeff') in _KEY_COL_NAMES),
        None
    )
    if _named_key:
        key_col = _named_key
    else:
        # Fall back to the first column that is not a language-code column
        _non_lang_cols = [col for col in df.columns if not _is_lang_col(col)]
        key_col = _non_lang_cols[0] if _non_lang_cols else None

    if key_col:
        print(f"🔑 Key column detected: '{key_col}'")
    else:
        print("⚠️  No key column found — emote/morse pre-filters will be skipped")

    # ── Pre-filter: drop emote shortcut rows ──────────────────────────────────
    if key_col:
        _before = len(df)
        df = df[~df[key_col].astype(str).str.match(r'emoticon\.\d+\.shortcut', na=False)].reset_index(drop=True)
        _dropped = _before - len(df)
        if _dropped:
            print(f"🚫 Dropped {_dropped:,} emote shortcut rows")

    # ── Pre-filter: drop morse rows (key/note contains \bmorse\b) ─────────────
    if key_col:
        _before = len(df)
        df = df[~df[key_col].astype(str).str.contains(r'\bmorse\b', flags=re.IGNORECASE, na=False)].reset_index(drop=True)
        _dropped = _before - len(df)
        if _dropped:
            print(f"🚫 Dropped {_dropped:,} morse rows")

    # ── Auto-detect term column if not provided ───────────────────────────────
    if term_column is None:
        # Normalize to 2-letter prefix so "es" matches "es-es", "es-ES", "es_ES", etc.
        common_names = [language_code, language_code.upper(), language_code.replace('-', '_'),
                       'Term', 'Terminology', 'Translation', 'Target']

        def _col_matches(col: str) -> bool:
            col_l = col.lower().lstrip('\ufeff')  # also strip any residual BOM
            # Exact match against common names
            if col_l in [n.lower() for n in common_names]:
                return True
            # "es" matches "es-es", "es_es", "es-ES" ...
            if col_l == lang_prefix or col_l.startswith(lang_prefix + '-') or col_l.startswith(lang_prefix + '_'):
                return True
            return False
        matched_cols = [col for col in df.columns if _col_matches(col)]

        if matched_cols:
            term_column = matched_cols[0]
            print(f"🔍 Auto-detected term column: '{term_column}'")
        else:
            # Fall back to the last language-looking column (excluding (info) columns)
            lang_cols = [col for col in df.columns if _is_lang_col(col) and '(info)' not in col.lower()]
            if lang_cols and allow_language_fallback:
                term_column = lang_cols[-1]
                print(f"⚠️  No exact match found. Using last language column: '{term_column}'")
            else:
                raise ValueError(
                    f"Could not detect exact term column for language '{language_code}' (normalized='{lang_prefix}'). Available columns: {list(df.columns)}"
                )

    if term_column not in df.columns:
        raise ValueError(f"Column '{term_column}' not found. Available columns: {list(df.columns)}")

    # ── ANKANIMATION-specific term cleanup ────────────────────────────────
    if str(game_tag).strip().upper() == 'ANKANIMATION':
        df, _ank_stats = _preprocess_ankanimation_dataframe(df, term_column, lang_prefix)
        print(
            f"🧼 ANKANIMATION cleanup: changed={_ank_stats['changed_rows']:,}/"
            f"{_ank_stats['total_rows']:,} rows, empty_after={_ank_stats['empty_after']:,}"
        )

    # ── Detect optional plural-info column (WAVEN style) ─────────────────────
    # WAVEN files carry a "<lang> (info)" column with plural forms:
    #   One: "Crujidor Erosionado", Other: "Crujidores Erosionados"
    info_col = f"{term_column} (info)" if f"{term_column} (info)" in df.columns else None
    if info_col:
        print(f"ℹ️  Plural-info column detected: '{info_col}'")

    def _parse_plural_info(info_text) -> List[str]:
        """Parse WAVEN plural format: 'One: "X", Other: "Y"' → list of plain strings."""
        if pd.isna(info_text):
            return []
        return re.findall(r'(?:One|Other|Few|Many|Zero|Two):\s*"([^"]*)"', str(info_text))

    # ── Extract and tokenize terms ────────────────────────────────────────────
    print(f"\n🧹 Tokenizing {len(df):,} entries from column '{term_column}'...")

    all_tokens = set()
    skipped_empty = 0
    processed = 0
    info_series = df[info_col] if info_col is not None else None

    # ── Proper-noun tracking ──────────────────────────────────────────────────
    # propernoun_map: key_type -> set of lowercased tokens from that category
    propernoun_map: Dict[str, Set[str]] = {}
    _key_series = df[key_col].astype(str) if key_col else None

    # Spanish-only fast index for default lowercasing of item/monster first tokens.
    # We collect category token sets in one pass and resolve additions after the loop.
    es_npc_area_tokens_lower: Set[str] = set()
    es_item_monster_first_candidates: Set[str] = set()

    for idx, term in enumerate(df[term_column]):
        # Skip empty or null values
        if pd.isna(term) or not str(term).strip():
            skipped_empty += 1
            continue
        # Tokenize the term (applies all custom filters and morphological expansion)
        term_str = str(term).strip()

        # Tokenize the term (applies all custom filters and morphological expansion)
        tokens = tokenize_text(term_str, language=tokenize_language)

        if tokens:
            all_tokens.update(tokens)
            processed += 1

            # ── Track proper-noun tokens by key-type ──────────────────────
            if _key_series is not None:
                raw_key = _key_series.iloc[idx]
                # TB keys are comma-separated groups of dot-notation keys
                # e.g. "monster.123.name, NPC.456.name"
                # Each part is matched against PROPER_NOUN_KEY_PATTERNS using
                # re.search() so we can distinguish .name from .reply etc.
                matched_categories: Set[str] = set()
                for key_part in raw_key.split(','):
                    key_part_s = key_part.strip()
                    for category, compiled_pats in _COMPILED_PROPER_NOUN_PATTERNS.items():
                        if any(p.search(key_part_s) for p in compiled_pats):
                            matched_categories.add(category)
                            if save_propernoun_sidecar:
                                propernoun_map.setdefault(category, set()).update(
                                    tok.lower() for tok in tokens
                                )
                            break  # stop at first matching category for this key_part
                    # continue outer loop — each comma-part may be a diff category

                # Spanish convention: item/monster names default to lowercase
                # first token, except when that token collides with NPC/area
                # vocabulary (token-level match, including compound names).
                if _is_spanish and matched_categories:
                    first_tok = _extract_first_token_lower(term_str)
                    if first_tok:
                        if ('npc' in matched_categories) or ('area' in matched_categories):
                            es_npc_area_tokens_lower.update(tok.lower() for tok in tokens)
                            es_npc_area_tokens_lower.add(first_tok)
                        if ('item' in matched_categories) or ('monster' in matched_categories):
                            es_item_monster_first_candidates.add(first_tok)

        # Also tokenize plural-info column if present (WAVEN)
        if info_series is not None:
            for plural_form in _parse_plural_info(info_series.iloc[idx]):
                all_tokens.update(tokenize_text(plural_form, language=tokenize_language))

        # Progress indicator every 1000 entries (single-line refresh)
        if (idx + 1) % 1000 == 0:
            print(
                f"   ✓ Processed {idx+1:,} entries... ({len(all_tokens):,} unique tokens so far)",
                end='\r',
                flush=True,
            )

    # Ensure next log starts on a new line after single-line progress refresh
    print()

    # ── Spanish default lowercasing for item/monster first token ───────────
    if _is_spanish and es_item_monster_first_candidates:
        _evidence = _build_es_i18n_case_evidence(game_tag, lang_prefix) if game_tag else {}
        _cand = len(es_item_monster_first_candidates)
        _blocked_npc_area_set = {
            tok for tok in es_item_monster_first_candidates
            if tok in es_npc_area_tokens_lower
        }
        if _evidence:
            _blocked_corpus_set = {
                tok for tok in es_item_monster_first_candidates
                if _evidence.get(tok, {}).get('uppercase_mid_sentence_count', 0) > 0
                and _evidence.get(tok, {}).get('lowercase_count', 0) == 0
            }
            _evidence_note = 'ready'
        else:
            # Conservative fallback: without filtered i18n evidence, do not
            # auto-add ES default lowercase candidates.
            _blocked_corpus_set = set(es_item_monster_first_candidates)
            _evidence_note = 'missing'

        _to_add = es_item_monster_first_candidates - _blocked_npc_area_set - _blocked_corpus_set
        _before = len(all_tokens)
        all_tokens.update(_to_add)
        _added = len(all_tokens) - _before
        print(
            f"   [es-first-lower] candidates={_cand:,}  "
            f"blocked(npc/area)={len(_blocked_npc_area_set):,}  "
            f"blocked(corpus-rule)={len(_blocked_corpus_set):,}  "
            f"added={_added:,}  evidence={_evidence_note}"
        )

    if str(game_tag).strip().upper() == 'ANKANIMATION':
        _before_ank = len(all_tokens)
        all_tokens = _apply_ankanimation_token_overrides(all_tokens, lang_prefix)
        _after_ank = len(all_tokens)
        print(f"   [ank-token-overrides] before={_before_ank:,} after={_after_ank:,}")

    # Convert to sorted list
    result_tokens = sorted(list(all_tokens))

    print(f"\n✅ TOKENIZATION COMPLETE:")
    print(f"   📥 Total entries: {len(df):,}")
    print(f"   ✓ Processed: {processed:,}")
    print(f"   ⚠️  Skipped (empty): {skipped_empty:,}")
    print(f"   🎯 Unique tokens: {len(result_tokens):,}")

    if result_tokens:
        print(f"\n   📝 Sample tokens (first 10):")
        for token in result_tokens[:10]:
            print(f"      - {token}")

    # Export if output path provided
    result_output_path = ""
    if output_file_path:
        print(f"\n💾 Exporting {len(result_tokens):,} tokenized terms to: {output_file_path}")
        export_tokens_to_txt(set(result_tokens), output_file_path)
        result_output_path = output_file_path
    else:
        # Generate default output path if not provided but would be useful to know
        base_name = Path(excel_file_path).stem
        default_output = f"{base_name}_{language_code}_tokenized.txt"

    # ── Save proper-noun sidecar JSON ─────────────────────────────────────────
    if save_propernoun_sidecar and propernoun_map:
        import json as _json
        lang_prefix = normalize_language_code(language_code)
        _game = game_tag or Path(excel_file_path).stem.split('_')[0]
        os.makedirs(INTERMEDIARY_DIR, exist_ok=True)
        _sidecar_path = os.path.join(
            INTERMEDIARY_DIR, f"{_game}_{lang_prefix}_propernoun_tokens.json"
        )
        # Convert sets to sorted lists for JSON serialisation
        _sidecar_data = {
            kt: sorted(tokens_set) for kt, tokens_set in propernoun_map.items()
        }
        with open(_sidecar_path, 'w', encoding='utf-8') as _fh:
            _json.dump(_sidecar_data, _fh, ensure_ascii=False, indent=2)
        _total_pn = sum(len(v) for v in _sidecar_data.values())
        print(f"\n🔖 Proper-noun sidecar saved → {_sidecar_path}")
        print(f"   Key-types: {list(_sidecar_data.keys())}  |  Tokens: {_total_pn:,}")

    elif save_propernoun_sidecar:
        print(f"\n🔖 No proper-noun tokens found for key patterns ({list(PROPER_NOUN_KEY_PATTERNS.keys())})")

    return result_tokens, result_output_path

## 2. Filtering

In [3]:
import concurrent.futures

# ==============================================================================
# 2.1  AFF PARSING
# ==============================================================================

def parse_aff_file(aff_file_path: str) -> Dict:
    """
    Parse Hunspell .aff file and extract affix rules.

    Fully handles FLAG directives (single/long/utf8/num) so that multi-char
    flags (e.g. Spanish es_ES.aff uses FLAG long with 2-char flags like "L1",
    "S1") are parsed correctly.

    Also detects NEEDAFFIX, CIRCUMFIX, ONLYINCOMPOUND and FORBIDDENWORD
    pseudo-flags so generate_word_forms() can skip them.

    Now also parses compound-word directives:
      COMPOUNDBEGIN, COMPOUNDMIDDLE, COMPOUNDEND, COMPOUNDFLAG, COMPOUNDMIN,
      COMPOUNDPERMITFLAG, COMPOUNDRULE.

    Also parses:
      CHECKSHARPS  — enables ß↔SS equivalence for uppercase matching.
                     When set, filter_tokens_by_dictionary_with_affixes() adds
                     ss-variants of every form containing ß to the valid-forms
                     set, so tokens with 'ss' (common in game texts that avoid
                     the ß glyph) are still recognised.
      KEEPCASE     — flag whose carriers must be matched case-sensitively.
                     Parsed and stored; the filter function currently uses
                     case-insensitive (lowercased) matching, which is the safe
                     conservative direction for filtering known words.

    Performance note: each rule dict carries a pre-compiled 're.Pattern' object
    under the key 'compiled_condition'. condition_matches() uses it directly so
    that re.compile() is called once per rule rather than once per word-form
    check. For a .aff with ~1 K unique conditions called ~10 M times this makes
    condition checking ~3-5x faster.

    ── BUG FIX (header-detection false positive) ──────────────────────────────
    Previous versions used `parts[2].upper() in ('Y', 'N')` to decide whether
    a PFX/SFX line is a *header* or a *rule*.  This incorrectly classified rule
    lines whose STRIP field happened to be the letter 'n' or 'N' (e.g.
    `SFX Y  n  te  ern`) as duplicate headers, silently dropping them.
    In de_DE_frami.aff this affected all 18 out of 36 SFX Y rules that strip
    the infinitive '-n', causing German verb past-tense forms like
    'überforderte', 'berechnete', etc. to be missed.  The fix adds a
    `parts[3].isdigit()` guard — header lines always have a numeric count as
    the fourth field, while rule ADD strings are never pure digits except '0'
    (which can only mean "add nothing" and is not a valid 0-rule-count header).
    ────────────────────────────────────────────────────────────────────────────

    Returns:
        Dict with keys:
          'PFX'               -> {flag: {cross_product, rules}}
          'SFX'               -> {flag: {cross_product, rules}}
          'flag_mode'         -> 'single' | 'long' | 'utf8' | 'num'
          'NEEDAFFIX'         -> str | None
          'CIRCUMFIX'         -> str | None
          'ONLYINCOMPOUND'    -> str | None
          'FORBIDDENWORD'     -> str | None
          'CHECKSHARPS'       -> bool  (True if directive is present)
          'KEEPCASE'          -> str | None  (flag letter, if present)
          'COMPOUNDBEGIN'     -> str | None   (flag marking compound-start stems)
          'COMPOUNDMIDDLE'    -> str | None   (flag marking compound-middle stems)
          'COMPOUNDEND'       -> str | None   (flag marking compound-end stems)
          'COMPOUNDFLAG'      -> str | None   (flag for any-position compounding)
          'COMPOUNDMIN'       -> int          (minimum part length, default 3)
          'COMPOUNDPERMITFLAG'-> str | None   (affix allowed inside compounds)
          'COMPOUNDRULE'      -> List[str]    (rule pattern strings)
        Each rule dict has keys:
          strip, add, add_flags (List[str]), condition (str),
          compiled_condition (re.Pattern | None  — None means "match all").
    """
    affixes: Dict = {
        'PFX': {},
        'SFX': {},
        'flag_mode': 'single',
        'encoding': 'utf-8',
        'NEEDAFFIX': None,
        'CIRCUMFIX': None,
        'ONLYINCOMPOUND': None,
        'FORBIDDENWORD': None,
        'CHECKSHARPS':    False,  # True when CHECKSHARPS directive is present
        'KEEPCASE':       None,   # flag letter for KEEPCASE entries
        # ── Compound directives ───────────────────────────────────────────
        'COMPOUNDBEGIN':      None,  # flag: word may START a compound (German-style)
        'COMPOUNDMIDDLE':     None,  # flag: word may appear MID-compound
        'COMPOUNDEND':        None,  # flag: word may END a compound
        'COMPOUNDFLAG':       None,  # flag: word may appear ANYWHERE in a compound
        'COMPOUNDMIN':        3,     # minimum chars per compound part (default 3)
        'COMPOUNDPERMITFLAG': None,  # affix flag: this affix is allowed inside compounds
        'COMPOUNDRULE':       [],    # list of rule-pattern strings (flag-sequence regexes)
    }

    def _parse_flags(flag_str: str, flag_mode: str) -> List[str]:
        """Split a raw flag string into individual flag tokens."""
        if not flag_str:
            return []
        if flag_mode == 'long':
            return [flag_str[i:i+2] for i in range(0, len(flag_str), 2)
                    if len(flag_str[i:i+2]) == 2]
        elif flag_mode == 'num':
            return [f.strip() for f in flag_str.split(',') if f.strip()]
        else:  # 'single' or 'utf8' - one codepoint per flag
            return list(flag_str)

    def _compile_condition(condition: str, is_prefix: bool):
        """Pre-compile a Hunspell condition pattern.  Returns None for '.' (match all)."""
        if condition == '.':
            return None
        try:
            if is_prefix:
                return re.compile(f'^(?:{condition})')
            else:
                return re.compile(f'(?:{condition})$')
        except re.error:
            return None  # Conservative: treated as match-all in condition_matches()

    # Detect encoding from the SET directive (safe: read as bytes first)
    _enc = 'utf-8'
    _enc_map = {
        'UTF-8': 'utf-8', 'UTF8': 'utf-8',
        'ISO8859-1': 'latin-1', 'ISO-8859-1': 'latin-1',
        'ISO8859-2': 'iso-8859-2', 'ISO-8859-2': 'iso-8859-2',
        'ISO8859-15': 'iso-8859-15', 'ISO-8859-15': 'iso-8859-15',
        'KOI8-R': 'koi8-r', 'KOI8-U': 'koi8-u',
        'CP1251': 'cp1251', 'WINDOWS-1251': 'cp1251',
        'CP1252': 'cp1252', 'WINDOWS-1252': 'cp1252',
    }
    with open(aff_file_path, 'rb') as _f:
        for _raw_line in _f:
            _l = _raw_line.decode('latin-1').strip()
            if _l.upper().startswith('SET ') and len(_l.split()) >= 2:
                _set_val = _l.split(None, 1)[1].strip().upper()
                _enc = _enc_map.get(_set_val, _set_val.lower())
                break

    affixes['encoding'] = _enc

    with open(aff_file_path, 'r', encoding=_enc) as f:
        lines = f.readlines()

    current_affix = None
    current_type = None

    for line in lines:
        line = line.strip()
        if not line or line.startswith('#'):
            continue

        parts = line.split()
        if not parts:
            continue

        directive = parts[0].upper()

        # Global directives
        if directive == 'FLAG' and len(parts) >= 2:
            mode = parts[1].lower()
            if mode == 'long':
                affixes['flag_mode'] = 'long'
            elif mode in ('utf-8', 'utf8'):
                affixes['flag_mode'] = 'utf8'
            elif mode == 'num':
                affixes['flag_mode'] = 'num'
            continue

        if directive in ('NEEDAFFIX', 'CIRCUMFIX', 'ONLYINCOMPOUND', 'FORBIDDENWORD',
                         'KEEPCASE') and len(parts) >= 2:
            affixes[directive] = parts[1]
            continue

        # CHECKSHARPS has no argument — its mere presence activates the feature
        if directive == 'CHECKSHARPS':
            affixes['CHECKSHARPS'] = True
            continue

        # Compound directives
        if directive in ('COMPOUNDBEGIN', 'COMPOUNDMIDDLE', 'COMPOUNDEND',
                         'COMPOUNDFLAG', 'COMPOUNDPERMITFLAG') and len(parts) >= 2:
            affixes[directive] = parts[1]
            continue

        if directive == 'COMPOUNDMIN' and len(parts) >= 2:
            try:
                affixes['COMPOUNDMIN'] = int(parts[1])
            except ValueError:
                pass
            continue

        if directive == 'COMPOUNDRULE' and len(parts) >= 2:
            # Skip pure-integer count header; collect actual pattern strings
            if not parts[1].isdigit():
                affixes['COMPOUNDRULE'].append(parts[1])
            continue

        # PFX / SFX
        if directive not in ('PFX', 'SFX') or len(parts) < 3:
            continue

        affix_type = directive
        flag = parts[1]
        cross_product = parts[2].upper() == 'Y'

        # ── Header-line detection ────────────────────────────────────────────
        # A genuine header has the form:  SFX flag Y|N count
        # i.e. exactly 4 space-separated tokens where:
        #   parts[2] is the cross-product indicator ('Y' or 'N', case-insensitive)
        #   parts[3] is a non-negative integer count of the rule lines that follow
        #
        # PREVIOUS BUG: `parts[2].upper() in ('Y','N')` also matches rule lines
        # whose STRIP field is the letter 'n' (e.g. `SFX Y  n  te  ern`),
        # silently dropping them and causing entire paradigm classes to vanish
        # (German -n infinitive past-tense → 18/36 SFX Y rules were dropped,
        #  also affected SFX X, I, J, O and others).
        #
        # FIX: require exactly 4 tokens AND a pure-digit count field.
        # Rule lines always have ≥5 tokens (SFX flag strip add condition).
        # ────────────────────────────────────────────────────────────────────
        if (len(parts) == 4
                and parts[2].upper() in ('Y', 'N')
                and parts[3].isdigit()):
            # Header line: SFX A Y 14
            if flag not in affixes[affix_type]:
                affixes[affix_type][flag] = {'cross_product': cross_product, 'rules': []}
            current_affix = flag
            current_type = affix_type
            continue

        # Rule line: PFX/SFX flag strip add[/secondary] [condition]
        if len(parts) >= 4 and current_affix == flag and current_type == affix_type:
            strip = parts[2] if parts[2] != '0' else ''

            raw_add = parts[3] if parts[3] != '0' else ''
            if '/' in raw_add:
                add, add_flags_str = raw_add.split('/', 1)
            else:
                add, add_flags_str = raw_add, ''

            add_flags = _parse_flags(add_flags_str, affixes['flag_mode'])
            condition = parts[4] if len(parts) > 4 else '.'

            # Pre-compile condition pattern (is_prefix = True for PFX, False for SFX)
            compiled_condition = _compile_condition(condition, is_prefix=(affix_type == 'PFX'))

            if current_affix in affixes[current_type]:
                affixes[current_type][current_affix]['rules'].append({
                    'strip': strip,
                    'add': add,
                    'add_flags': add_flags,          # List[str], pre-split
                    'condition': condition,           # raw string (kept for debugging)
                    'compiled_condition': compiled_condition  # re.Pattern | None
                })

    return affixes


# ==============================================================================
# 2.2  CONDITION MATCHING
# ==============================================================================

def condition_matches(word: str, condition: str, is_prefix: bool = True,
                      compiled=None) -> bool:
    """
    Check if word matches the Hunspell affix condition pattern.

    Conditions are anchored: PFX conditions match from the left,
    SFX conditions match at the right end of the word.

    Args:
        word:       Word to test
        condition:  Hunspell condition pattern (supports '.', [abc], [^abc])
        is_prefix:  True -> match at start (PFX); False -> match at end (SFX)
        compiled:   Optional pre-compiled re.Pattern from parse_aff_file().
                    When provided the raw condition string is ignored and the
                    compiled pattern is used directly for ~3-5x speedup.

    Returns:
        bool: True if condition matches
    """
    if condition == '.' and compiled is None:
        return True
    if compiled is None:
        # Fallback: compile on the fly (slow path, kept for direct callers)
        try:
            if is_prefix:
                return bool(re.match(f'^(?:{condition})', word))
            else:
                return bool(re.search(f'(?:{condition})$', word))
        except re.error:
            return True  # Conservative: assume match

    # Fast path: use pre-compiled pattern
    # None means '.' (match all)
    if compiled is None:
        return True
    return bool(compiled.search(word))


# ==============================================================================
# 2.3  WORD-FORM GENERATION  (forward / exhaustive expansion)
# ==============================================================================

def generate_word_forms(base_word: str, flags: List[str], affixes: Dict,
                        _visited: Set[str] = None, _depth: int = 0) -> Set[str]:
    """
    Generate all possible word forms using Hunspell affix rules.

    Uses pre-compiled condition patterns stored in each rule dict by
    parse_aff_file() for significantly faster condition matching.

    Implements cross-product (XPRODUCT): suffix-derived forms whose rule set
    has cross_product=True are further expanded with applicable prefixes,
    matching unmunch.cxx behaviour.

    Args:
        base_word: Base (lowercased) word to inflect
        flags:     List of flag identifiers (pre-parsed from .dic entry or add_flags)
        affixes:   Full affixes dict from parse_aff_file()
        _visited:  Internal cycle guard (do not set manually)
        _depth:    Internal recursion depth (do not set manually)

    Returns:
        Set[str]: All inflected forms including the base word
    """
    if _visited is None:
        _visited = set()

    word_forms: Set[str] = {base_word}

    if not flags or _depth > 3:
        return word_forms

    skip_flags: Set = {
        affixes.get('NEEDAFFIX'),
        affixes.get('CIRCUMFIX'),
        affixes.get('ONLYINCOMPOUND'),
        affixes.get('FORBIDDENWORD'),
        None, '',
    }

    cross_product_forms: List[Tuple[str, List[str]]] = []

    # Suffixes
    for flag in flags:
        if flag in skip_flags or flag not in affixes['SFX']:
            continue

        rule_set = affixes['SFX'][flag]
        for rule in rule_set['rules']:
            compiled = rule.get('compiled_condition')
            if not condition_matches(base_word, rule['condition'],
                                     is_prefix=False, compiled=compiled):
                continue
            if rule['strip']:
                if not base_word.endswith(rule['strip']):
                    continue
                derived = base_word[:-len(rule['strip'])] + rule['add']
            else:
                derived = base_word + rule['add']

            if not derived or derived in _visited:
                continue

            word_forms.add(derived)

            if rule_set['cross_product']:
                cross_product_forms.append((derived, rule['add_flags']))

            if rule['add_flags']:
                _visited.add(derived)
                word_forms.update(
                    generate_word_forms(derived, rule['add_flags'], affixes,
                                        _visited, _depth + 1)
                )

    # Prefixes
    for flag in flags:
        if flag in skip_flags or flag not in affixes['PFX']:
            continue

        rule_set = affixes['PFX'][flag]
        for rule in rule_set['rules']:
            compiled = rule.get('compiled_condition')
            if not condition_matches(base_word, rule['condition'],
                                     is_prefix=True, compiled=compiled):
                continue
            if rule['strip']:
                if not base_word.startswith(rule['strip']):
                    continue
                derived = rule['add'] + base_word[len(rule['strip']):]
            else:
                derived = rule['add'] + base_word

            if not derived or derived in _visited:
                continue

            word_forms.add(derived)

            if rule['add_flags']:
                _visited.add(derived)
                word_forms.update(
                    generate_word_forms(derived, rule['add_flags'], affixes,
                                        _visited, _depth + 1)
                )

    # Cross-product: apply prefixes to suffix-derived forms
    for derived_form, extra_flags in cross_product_forms:
        for flag in flags:
            if flag in skip_flags or flag not in affixes['PFX']:
                continue
            if not affixes['PFX'][flag]['cross_product']:
                continue
            for rule in affixes['PFX'][flag]['rules']:
                compiled = rule.get('compiled_condition')
                if not condition_matches(derived_form, rule['condition'],
                                         is_prefix=True, compiled=compiled):
                    continue
                if rule['strip']:
                    if not derived_form.startswith(rule['strip']):
                        continue
                    cross_derived = rule['add'] + derived_form[len(rule['strip']):]
                else:
                    cross_derived = rule['add'] + derived_form

                if cross_derived and cross_derived not in _visited:
                    word_forms.add(cross_derived)

    return word_forms


# ==============================================================================
# 2.4  REVERSE-LOOKUP  (alternative - fast but less exhaustive)
# ==============================================================================

def token_is_known(token_lower: str, base_words: Set[str], affixes: Dict,
                   sfx_rules: List[Tuple], pfx_rules: List[Tuple],
                   cross_sfx_rules: List[Tuple]) -> bool:
    """
    ALTERNATIVE reverse-lookup: check whether a token is derivable from the
    Hunspell dictionary without expanding every dictionary entry.

    STATUS: Correct and FAST (~seconds vs ~1 min for forward expansion).
    Kept as a documented alternative.  The primary path used by
    filter_tokens_by_dictionary_with_affixes() is generate_word_forms()
    (forward expansion), which is exhaustive and easier to reason about.

    Use token_is_known() directly if speed is critical and you accept
    that very deep chained derivations (depth > 1 affix) might be missed.

    Strategy: for each token, reconstruct a potential dictionary base word by
    reversing known suffix/prefix rules, then do a fast O(1) set lookup.

    Complexity: O(tokens x rules)  vs  O(entries x avg_forms) for forward.
    For pt_BR: ~few K tokens x ~few hundred rules  <<  312 K entries x 33 forms.

    Args:
        token_lower:     Lowercased token to test
        base_words:      Set of lowercased base words from the .dic file
        affixes:         Full affixes dict from parse_aff_file()
        sfx_rules:       Flat list of (add, strip, condition, cross_product)
        pfx_rules:       Flat list of (add, strip, condition, cross_product)
        cross_sfx_rules: SFX rules with cross_product=True

    Returns:
        bool: True if the token matches any dictionary form
    """
    if token_lower in base_words:
        return True

    for add, strip, condition, _ in sfx_rules:
        if add:
            if not token_lower.endswith(add):
                continue
            root = token_lower[:-len(add)]
        else:
            root = token_lower
        candidate = root + strip
        if not candidate:
            continue
        if not condition_matches(candidate, condition, is_prefix=False):
            continue
        if candidate in base_words:
            return True

    for add, strip, condition, _ in pfx_rules:
        if add:
            if not token_lower.startswith(add):
                continue
            rest = token_lower[len(add):]
        else:
            rest = token_lower
        candidate = strip + rest
        if not candidate:
            continue
        if not condition_matches(candidate, condition, is_prefix=True):
            continue
        if candidate in base_words:
            return True

    for pfx_add, pfx_strip, pfx_cond, pfx_xprod in pfx_rules:
        if not pfx_xprod:
            continue
        if pfx_add:
            if not token_lower.startswith(pfx_add):
                continue
            de_prefixed = token_lower[len(pfx_add):]
        else:
            de_prefixed = token_lower
        if not de_prefixed:
            continue
        for add, strip, condition, xprod in cross_sfx_rules:
            if add:
                if not de_prefixed.endswith(add):
                    continue
                root = de_prefixed[:-len(add)]
            else:
                root = de_prefixed
            candidate = root + strip
            if not candidate:
                continue
            if not condition_matches(candidate, condition, is_prefix=False):
                continue
            if not condition_matches(candidate, pfx_cond, is_prefix=True):
                continue
            if candidate in base_words:
                return True

    return False


# ==============================================================================
# 2.5  THREADING HELPERS  (module-level so threads can call them)
# ==============================================================================

def _expand_chunk(chunk: List[Tuple[str, List[str]]], affixes: Dict) -> Set[str]:
    """
    Thread worker: expand a list of (base_word, flags) pairs into all word
    forms using generate_word_forms().

    Defined at module level so it is safely callable from ThreadPoolExecutor
    workers without pickling issues.

    Args:
        chunk:   List of (base_word_lower, flags) tuples
        affixes: Full affixes dict (shared read-only across threads - safe
                 because it is never mutated after parse_aff_file() returns)

    Returns:
        Set[str]: All inflected surface forms for the entries in this chunk
    """
    forms: Set[str] = set()
    for base_word, flags in chunk:
        forms.update(generate_word_forms(base_word, flags, affixes))
    return forms


def _expand_chunk_compound(chunk: List[Tuple[str, List[str]]],
                           affixes: Dict) -> Dict[str, Set[str]]:
    """
    Thread worker: expand compound-capable entries, returning a per-entry mapping.

    Differs from _expand_chunk in two ways:
      1. Returns {base_word: Set[forms]} instead of a flat set so the caller
         can assign forms to role buckets (BEGIN / MIDDLE / END).
      2. The ONLYINCOMPOUND special flag is *not* skipped - stems that only
         exist inside a compound (e.g. German fuge-elements) must still be
         expanded so they can be recognised as valid compound parts.

    ── BUG FIX (dict-key collision on shared lowercased base_word) ─────────────
    German (and other) dictionaries list the same base word multiple times with
    different capitalisation and different flag sets, e.g.:
        Fluss/TpMmij   (capitalised entry — compound-BEGIN via flag j)
        fluss/TpMozm   (lowercase entry  — compound-END   via flag z)
        fluss/hke      (compound-MIDDLE form with fuge flags)
    All three lowercase to the key 'fluss'.  Overwriting with plain dict
    assignment (`result[key] = ...`) kept only the last entry's forms, dropping
    e.g. 'flusses' (genitive via SFX T from `fluss/TpMozm`) when `fluss/hke`
    happened to be written last.
    Fix: use set-union merge so EVERY entry's forms accumulate under the key.
    ────────────────────────────────────────────────────────────────────────────

    Args:
        chunk:   List of (base_word_lower, flags) tuples.
        affixes: Full affixes dict from parse_aff_file().

    Returns:
        Dict mapping each base_word to its UNION of all inflected surface forms
        across every .dic entry that shares that lowercased base string.
    """
    # Shallow-clone affixes with ONLYINCOMPOUND cleared so those stems expand
    compound_affixes = {**affixes, 'ONLYINCOMPOUND': None}
    result: Dict[str, Set[str]] = {}
    for base_word, flags in chunk:
        forms = generate_word_forms(base_word, flags, compound_affixes)
        if base_word in result:
            result[base_word].update(forms)   # ← merge, never overwrite
        else:
            result[base_word] = forms
    return result


# ==============================================================================
# 2.6  COMPOUND-WORD CHECKER
# ==============================================================================

def is_valid_compound(token: str,
                      begin_forms: Set[str],
                      middle_forms: Set[str],
                      end_forms: Set[str],
                      flag_forms: Set[str],
                      all_forms: Set[str],
                      compound_min: int = 3,
                      _depth: int = 0) -> bool:
    """
    Check whether a token is a valid Hunspell compound word.

    Handles three compound systems found in real-world dictionaries:

    1. COMPOUNDBEGIN / COMPOUNDMIDDLE / COMPOUNDEND  (German de_DE_frami style):
       Tries every split-point where each part is >= compound_min chars:
         - 2-part: left in begin_forms  AND  right in end_forms
         - 3-part: left in begin_forms  AND  mid in middle_forms  AND  right in end_forms
       In both cases the left (BEGIN) part may itself be a valid sub-compound,
       enabling recognition of long German chains like:
         'überschallschraubenzieher'
           → 'überschall' (sub-compound: 'über'+'schall') + 'schrauben' + 'zieher'
       Recursion depth is capped independently per call to avoid explosion.

    2. COMPOUNDFLAG  (any-position flag, simpler dictionaries):
       Both parts must be in flag_forms.

    3. Hyphenated tokens  (e.g. 'uebermagier-rune'):
       Splits on '-'; each sub-part must be in all_forms OR be itself a valid
       compound.  This mirrors Hunspell's European behaviour of treating the
       hyphen as a compound word separator.

    Recursion is capped at depth 4 to prevent runaway on adversarial inputs.

    Args:
        token:        Lowercased token to test.
        begin_forms:  Surface forms of COMPOUNDBEGIN-flagged dictionary entries.
        middle_forms: Surface forms of COMPOUNDMIDDLE-flagged dictionary entries.
        end_forms:    Surface forms of COMPOUNDEND-flagged dictionary entries.
        flag_forms:   Surface forms of COMPOUNDFLAG-flagged dictionary entries.
        all_forms:    Full set of known surface forms (for hyphen-split validation).
        compound_min: Minimum characters per compound segment (from COMPOUNDMIN).
        _depth:       Internal recursion guard - do not set manually.

    Returns:
        bool: True if the token is recognisable as a valid Hunspell compound.
    """
    if _depth > 4:
        return False
    n = len(token)
    if n < compound_min * 2:
        return False

    # ── Helper: is `s` a valid compound-BEGIN segment? ──────────────────────
    # A segment qualifies as a BEGIN part if it is directly in begin_forms OR
    # if it is itself a valid compound (recursive, depth-limited).  This lets
    # us handle long concatenative chains like 'überschall' = 'über'+'schall'.
    def _is_valid_begin(s: str) -> bool:
        if s in begin_forms:
            return True
        # Allow one level of recursion for sub-compounds (depth guard prevents
        # runaway; minimum length ensures we never recurse on trivial strings).
        if _depth < 3 and len(s) >= compound_min * 2:
            return is_valid_compound(s, begin_forms, middle_forms, end_forms,
                                     flag_forms, all_forms, compound_min, _depth + 1)
        return False

    # 1. Hyphenated compounds
    # European Hunspell dictionaries validate each hyphen-delimited segment
    # independently.  A hyphenated form is valid when every non-empty segment
    # is either a known surface form or itself a valid compound.
    if '-' in token:
        parts = [p for p in token.split('-') if p]
        if len(parts) >= 2 and all(
            p in all_forms or
            is_valid_compound(p, begin_forms, middle_forms, end_forms,
                              flag_forms, all_forms, compound_min, _depth + 1)
            for p in parts
        ):
            return True
        # Fall through: also try concatenation-based checks in case the hyphen
        # is part of a fuge-element stem (German 'Arbeits-' etc.).

    # 2. COMPOUNDBEGIN / COMPOUNDEND  (2-part)
    if begin_forms and end_forms:
        for i in range(compound_min, n - compound_min + 1):
            left  = token[:i]
            right = token[i:]
            if right in end_forms and _is_valid_begin(left):
                return True

    # 3. COMPOUNDBEGIN / COMPOUNDMIDDLE / COMPOUNDEND  (3-part)
    if begin_forms and middle_forms and end_forms:
        for i in range(compound_min, n - 2 * compound_min + 1):
            left = token[:i]
            if not _is_valid_begin(left):
                continue
            for j in range(i + compound_min, n - compound_min + 1):
                mid   = token[i:j]
                right = token[j:]
                if mid in middle_forms and right in end_forms:
                    return True

    # 4. COMPOUNDFLAG  (simpler: any-position flag)
    if flag_forms:
        for i in range(compound_min, n - compound_min + 1):
            left  = token[:i]
            right = token[i:]
            if left in flag_forms and right in flag_forms:
                return True

    return False


# ==============================================================================
# 2.7  MAIN FILTER FUNCTION  (forward expansion, threaded)
# ==============================================================================

def filter_tokens_by_dictionary_with_affixes(txt_file_path: str, dic_file_path: str,
                                             aff_file_path: str, output_dic_path: str,
                                             num_threads: int = None) -> Dict[str, int]:
    """
    Filter tokens by removing those found in the Hunspell dictionary.

    Uses **forward expansion** (generate_word_forms) to build the full set of
    valid surface forms, then removes any input token that appears in that set.
    Compound words are handled via a second pass using is_valid_compound().

    Performance improvements:

    1. Pre-compiled regex conditions - parse_aff_file() stores a compiled
       re.Pattern in every rule so condition_matches() never calls re.compile()
       at match time.  With ~10 M condition checks this saves significant time.

    2. ThreadPoolExecutor parallelism - dictionary entries are split into
       chunks proportional to the number of logical CPU cores.  Each thread
       expands its chunk independently; results are unioned at the end.
       Because generate_word_forms() is mostly string/set operations and
       the regex search calls release the GIL, threading gives a real speedup
       (typically 2-4x on a quad-core machine).

    3. Compound-word detection - after the main expansion, a second parallel
       pass builds compound-role form sets (BEGIN / MIDDLE / END / FLAG) so
       that German-style concatenative compounds (e.g. 'Uebergang',
       'Zaubertraenke') and hyphenated compounds (e.g. 'Uebermagier-Rune')
       are correctly recognised as dictionary words and removed.

    4. Trailing-dot normalisation - Hunspell (pt_BR) stores abbreviations
       as 'bomb.' so that suffix rules append directly to the stem.

    5. BOM-aware file reading - the .dic file is opened with 'utf-8-sig'
       so the leading BOM that some distributions include is silently consumed.

    6. CHECKSHARPS support - when the .aff declares CHECKSHARPS (German),
       every generated form containing 'ß' is also added in its ss-variant
       so that game tokens written with 'ss' (e.g. from older encodings or
       all-caps text) are still matched.

    7. FORBIDDENWORD exclusion - .dic entries that carry the FORBIDDENWORD
       flag are completely excluded from the valid-forms set.  Hunspell itself
       rejects such words as spelling errors; we must not remove them from the
       token list either.

    Args:
        txt_file_path:  Path to the txt file with tokens (one per line)
        dic_file_path:  Path to the Hunspell .dic file
        aff_file_path:  Path to the Hunspell .aff file with affix rules
        output_dic_path: Path where the filtered .dic file will be saved
        num_threads:    Number of worker threads.  Defaults to os.cpu_count().

    Returns:
        Dict[str, int]: Statistics - original_txt_tokens, dictionary_base_words,
            generated_word_forms, removed_tokens, compound_removed_tokens,
            remaining_tokens
    """
    if not os.path.exists(txt_file_path):
        raise FileNotFoundError(f"Token file not found: {txt_file_path}")
    if not os.path.exists(dic_file_path):
        raise FileNotFoundError(f"Dictionary file not found: {dic_file_path}")
    if not os.path.exists(aff_file_path):
        raise FileNotFoundError(f"Affix file not found: {aff_file_path}")

    if num_threads is None:
        num_threads = os.cpu_count() or 4
    print(f"Using {num_threads} threads for parallel dictionary expansion")

    # Parse affix rules (with pre-compiled conditions)
    print(f"Parsing affix rules from: {aff_file_path}")
    affixes = parse_aff_file(aff_file_path)

    flag_mode = affixes.get('flag_mode', 'single')
    print(f"FLAG mode: {flag_mode}")
    for special in ('NEEDAFFIX', 'CIRCUMFIX', 'ONLYINCOMPOUND', 'FORBIDDENWORD', 'KEEPCASE'):
        if affixes.get(special):
            print(f"  {special} pseudo-flag: {affixes[special]!r}")
    if affixes.get('CHECKSHARPS'):
        print(f"  CHECKSHARPS: active (ß↔ss variants will be added)")
    for cdir in ('COMPOUNDBEGIN', 'COMPOUNDMIDDLE', 'COMPOUNDEND',
                 'COMPOUNDFLAG', 'COMPOUNDPERMITFLAG'):
        if affixes.get(cdir):
            print(f"  {cdir}: {affixes[cdir]!r}")
    if affixes.get('COMPOUNDBEGIN') or affixes.get('COMPOUNDFLAG'):
        print(f"  COMPOUNDMIN: {affixes['COMPOUNDMIN']}")

    prefix_count = sum(len(rs['rules']) for rs in affixes['PFX'].values())
    suffix_count = sum(len(rs['rules']) for rs in affixes['SFX'].values())
    print(f"Loaded {len(affixes['PFX'])} prefix flags ({prefix_count} rules) "
          f"and {len(affixes['SFX'])} suffix flags ({suffix_count} rules)")

    # Helper: split raw flag string from .dic entry
    def _parse_entry_flags(flag_str: str) -> List[str]:
        if not flag_str:
            return []
        if flag_mode == 'long':
            return [flag_str[i:i+2] for i in range(0, len(flag_str), 2)
                    if len(flag_str[i:i+2]) == 2]
        elif flag_mode == 'num':
            return [f.strip() for f in flag_str.split(',') if f.strip()]
        else:
            return list(flag_str)

    # Use the same encoding as the .aff file (stored in affixes['encoding'])
    # utf-8-sig also strips BOM (\ufeff) for UTF-8 dictionaries
    _dic_enc = affixes.get('encoding', 'utf-8')
    if _dic_enc == 'utf-8':
        _dic_enc = 'utf-8-sig'  # strip BOM if present
    with open(dic_file_path, 'r', encoding=_dic_enc) as f:
        dic_lines = f.readlines()

    if not dic_lines:
        raise ValueError("Dictionary file is empty")

    print(f"Dictionary declared entry count: {dic_lines[0].strip()}")

    # ── FORBIDDENWORD exclusion ──────────────────────────────────────────────
    # Entries carrying the FORBIDDENWORD flag (e.g. flag 'd' in de_DE_frami)
    # are treated as misspellings by Hunspell — they must NOT be added to the
    # valid-forms set, and we must NOT remove matching tokens from our output.
    _forbid = affixes.get('FORBIDDENWORD')
    forbidden_skipped = 0

    entries: List[Tuple[str, List[str]]] = []
    for line in dic_lines[1:]:
        line = line.strip()
        if not line:
            continue
        if '/' in line:
            raw_word, flags_str = line.split('/', 1)
            flags = _parse_entry_flags(flags_str)
        else:
            raw_word, flags = line, []

        # Skip FORBIDDENWORD entries entirely — Hunspell rejects these words
        # even though they appear in the .dic (old/invalid spelling variants).
        if _forbid and _forbid in flags:
            forbidden_skipped += 1
            continue

        # Trailing-dot normalisation:
        # pt_BR.dic stores abbreviations as "bomb." so that suffix rules append
        # directly to the stem.  Stripping the dot lets "+a" -> "bomba" work
        # instead of producing the spurious form "bomb.a".
        base_word = raw_word.rstrip('.').lower()
        if base_word:
            entries.append((base_word, flags))

    if forbidden_skipped:
        print(f"Skipped {forbidden_skipped:,} FORBIDDENWORD entries (flag {_forbid!r})")
    print(f"Parsed {len(entries):,} dictionary entries")

    # Parallel forward expansion
    chunk_size = max(1, len(entries) // (num_threads * 8))
    chunks = [entries[i:i + chunk_size] for i in range(0, len(entries), chunk_size)]
    print(f"Expanding word forms: {len(chunks)} chunks x ~{chunk_size} entries "
          f"across {num_threads} threads...")

    all_dictionary_forms: Set[str] = set()
    t0 = time.time()

    with concurrent.futures.ThreadPoolExecutor(max_workers=num_threads) as executor:
        futures = {executor.submit(_expand_chunk, chunk, affixes): i
                   for i, chunk in enumerate(chunks)}
        completed = 0
        for future in concurrent.futures.as_completed(futures):
            all_dictionary_forms.update(future.result())
            completed += 1
            if completed % max(1, len(chunks) // 10) == 0:
                elapsed = time.time() - t0
                print(
                    f"  {completed}/{len(chunks)} chunks done ({len(all_dictionary_forms):,} forms, {elapsed:.1f}s)",
                    end='\r',
                    flush=True,
                )

    print()
    print(f"Generated {len(all_dictionary_forms):,} unique word forms "
          f"from {len(entries):,} entries in {time.time()-t0:.1f}s")

    # ── CHECKSHARPS: add ß→ss variants ──────────────────────────────────────
    # When CHECKSHARPS is active (German), Hunspell equates ß with SS in
    # uppercase contexts.  Game texts often store these words with 'ss' instead
    # of 'ß' (e.g. from older Latin-1 sources or all-caps display rendering).
    # Adding ss-variants ensures tokens like 'strasse' match 'straße'.
    if affixes.get('CHECKSHARPS'):
        ss_variants: Set[str] = set()
        for form in all_dictionary_forms:
            if 'ß' in form:
                ss_variants.add(form.replace('ß', 'ss'))
        if ss_variants:
            all_dictionary_forms.update(ss_variants)
            print(f"CHECKSHARPS: added {len(ss_variants):,} ss-variants of ß forms")

    # Build compound form sets (if compound flags are present in this dictionary)
    compound_begin_forms:  Set[str] = set()
    compound_middle_forms: Set[str] = set()
    compound_end_forms:    Set[str] = set()
    compound_flag_forms:   Set[str] = set()

    _c_begin  = affixes.get('COMPOUNDBEGIN')
    _c_middle = affixes.get('COMPOUNDMIDDLE')
    _c_end    = affixes.get('COMPOUNDEND')
    _c_flag   = affixes.get('COMPOUNDFLAG')
    _c_oic    = affixes.get('ONLYINCOMPOUND')
    _compound_min = affixes.get('COMPOUNDMIN', 3)

    _any_compound = _c_begin or _c_middle or _c_end or _c_flag
    if _any_compound:
        print(f"Compound flags detected - building compound form sets "
              f"(COMPOUNDMIN={_compound_min})...")

        # German-style dictionaries (and others) assign compound-role flags
        # (COMPOUNDBEGIN x, COMPOUNDMIDDLE y, COMPOUNDEND z) only as add_flags
        # inside affix rules, not directly on .dic entries.  For example:
        #   SFX j 0 0/xoc .    <- SFX flag 'j' produces forms with flag 'x'
        # So we must discover which ENTRY-LEVEL affix flags transitively lead to
        # compound-role flags via their add_flags chains.
        def _entry_flags_for_role(role_flag: str) -> Set[str]:
            """
            Return all SFX/PFX flags whose rules (transitively) include
            role_flag in add_flags.  Uses BFS to follow the chain
            (flag A -> add_flag B -> add_flag C -> role_flag).
            """
            direct: Set[str] = set()
            for ft in ('SFX', 'PFX'):
                for aflag, rs in affixes[ft].items():
                    if any(role_flag in r.get('add_flags', [])
                           for r in rs['rules']):
                        direct.add(aflag)
            result = set(direct)
            changed = True
            while changed:
                changed = False
                for ft in ('SFX', 'PFX'):
                    for aflag, rs in affixes[ft].items():
                        if aflag in result:
                            continue
                        if any(af in result
                               for r in rs['rules']
                               for af in r.get('add_flags', [])):
                            result.add(aflag)
                            changed = True
            return result

        begin_eflag  = _entry_flags_for_role(_c_begin)  if _c_begin  else set()
        middle_eflag = _entry_flags_for_role(_c_middle) if _c_middle else set()
        end_eflag    = _entry_flags_for_role(_c_end)    if _c_end    else set()

        def _has_compound_role(flags: List[str]) -> bool:
            fset = set(flags)
            return ((_c_begin  and (fset & begin_eflag  or _c_begin  in fset)) or
                    (_c_middle and (fset & middle_eflag or _c_middle in fset)) or
                    (_c_end    and (fset & end_eflag    or _c_end    in fset)) or
                    (_c_flag   and _c_flag  in fset) or
                    (_c_oic    and _c_oic   in fset))

        compound_entries = [(w, f) for w, f in entries if _has_compound_role(f)]
        print(f"  {len(compound_entries):,} entries carry compound/OIC flags")
        print(f"  BEGIN via entry-flags: {begin_eflag or {'(direct only)'}}")
        print(f"  END via entry-flags:   {end_eflag   or {'(direct only)'}}")

        cchunks = [compound_entries[i:i + chunk_size]
                   for i in range(0, len(compound_entries), chunk_size)]

        # Map each base_word to its compound-expanded forms (includes OIC stems).
        # ── BUG FIX (thread-merge dict collision) ───────────────────────────
        # entry_forms_map.update(chunk_result) would silently overwrite forms
        # already accumulated for a base_word that appears in multiple chunks
        # (e.g. 'fluss' from Fluss/TpMmij AND fluss/TpMozm AND fluss/hke).
        # Fix: explicitly merge using set-union so no chunk's forms are lost.
        # ─────────────────────────────────────────────────────────────────────
        entry_forms_map: Dict[str, Set[str]] = {}
        with concurrent.futures.ThreadPoolExecutor(max_workers=num_threads) as executor:
            cfutures = [executor.submit(_expand_chunk_compound, ch, affixes)
                        for ch in cchunks]
            for fut in concurrent.futures.as_completed(cfutures):
                for k, v in fut.result().items():
                    if k in entry_forms_map:
                        entry_forms_map[k].update(v)   # ← merge, never overwrite
                    else:
                        entry_forms_map[k] = v

        # Assign forms to role sets.  An entry contributes to a role if it
        # either (a) carries the compound-role flag directly, OR (b) carries
        # an entry-level flag that transitively produces the compound-role flag.
        for base_word, flags in compound_entries:
            forms = entry_forms_map.get(base_word, {base_word})
            fset  = set(flags)
            if _c_begin  and (fset & begin_eflag  or _c_begin  in fset):
                compound_begin_forms.update(forms)
            if _c_middle and (fset & middle_eflag or _c_middle in fset):
                compound_middle_forms.update(forms)
            if _c_end    and (fset & end_eflag    or _c_end    in fset):
                compound_end_forms.update(forms)
            if _c_flag   and _c_flag  in fset:
                compound_flag_forms.update(forms)

        print(f"  BEGIN:{len(compound_begin_forms):,}  "
              f"MIDDLE:{len(compound_middle_forms):,}  "
              f"END:{len(compound_end_forms):,}  "
              f"FLAG:{len(compound_flag_forms):,}")

    # Load tokens
    print(f"Reading tokens from: {txt_file_path}")
    original_txt_tokens: List[str] = []
    with open(txt_file_path, 'r', encoding='utf-8') as f:
        for line in f:
            token = line.strip()
            if token:
                original_txt_tokens.append(token)
    print(f"Loaded {len(original_txt_tokens):,} tokens to check")

    # Filter
    _use_compound = bool(_any_compound)
    filtered_tokens: List[str] = []
    removed_count = 0
    compound_removed_count = 0
    sample_removals: List[str] = []

    for original_token in original_txt_tokens:
        tok_lower = original_token.lower()
        known = tok_lower in all_dictionary_forms

        if not known and _use_compound:
            # Check whether the token is a valid compound word.  This handles:
            #   - Concatenative compounds (German): Uebergang, Zaubertrank, ...
            #   - Hyphenated compounds:             Uebermagier-Rune, Elfen-Klaeger, ...
            if is_valid_compound(tok_lower,
                                 compound_begin_forms, compound_middle_forms,
                                 compound_end_forms,   compound_flag_forms,
                                 all_dictionary_forms, _compound_min):
                known = True
                compound_removed_count += 1

        if known:
            removed_count += 1
            if len(sample_removals) < 10:
                sample_removals.append(original_token)
        else:
            filtered_tokens.append(original_token)

    if _use_compound and compound_removed_count:
        print(f"  (of which {compound_removed_count:,} removed via compound-word detection)")

    if sample_removals:
        print(f"Sample removed tokens: {', '.join(sample_removals[:5])}"
              f"{'...' if len(sample_removals) > 5 else ''}")

    print(f"Removed {removed_count:,} tokens that match dictionary word forms")
    print(f"Remaining tokens: {len(filtered_tokens):,}")

    os.makedirs(os.path.dirname(output_dic_path), exist_ok=True) if os.path.dirname(output_dic_path) else None
    with open(output_dic_path, 'w', encoding='utf-8') as f:
        f.write(str(len(filtered_tokens)) + '\n')
        for token in filtered_tokens:
            f.write(token + '\n')

    print(f"Filtered tokens saved as dictionary to: {output_dic_path}")

    return {
        'original_txt_tokens':    len(original_txt_tokens),
        'dictionary_base_words':  len(entries),
        'generated_word_forms':   len(all_dictionary_forms),
        'removed_tokens':         removed_count,
        'compound_removed_tokens':compound_removed_count,
        'remaining_tokens':       len(filtered_tokens)
    }


## 3. Batch processing

In [4]:
import os
import glob
from pathlib import Path
from typing import Any, Dict, List

def _build_hunspell_dic_candidates(lang_prefix: str, dic_folder: str = "dics") -> List[str]:
    """Return ordered candidate .dic paths for a normalized language code."""
    candidates: List[str] = []

    configured = HUNSPELL_PATHS.get(lang_prefix, "")
    if configured:
        candidates.append(configured)

        # Retarget paths rooted at DIC_FOLDER when dic_folder is overridden.
        norm_cfg = os.path.normpath(configured)
        cfg_parts = norm_cfg.split(os.sep)
        dic_root = os.path.normpath(DIC_FOLDER)
        if cfg_parts and cfg_parts[0].lower() == dic_root.lower():
            candidates.append(os.path.join(dic_folder, *cfg_parts[1:]))

    # Fallback candidates for common naming/layout variants.
    fallback_map = {
        "fr": [
            os.path.join(dic_folder, "fr_dic", "fr_FR.dic"),
            os.path.join(dic_folder, "fr_dic", "fr.dic"),
        ],
        "en": [
            os.path.join(dic_folder, "en_dic", "en_GB.dic"),
            os.path.join(dic_folder, "en_dic", "en_US.dic"),
            os.path.join(dic_folder, "en_dic", "en.dic"),
        ],
        "es": [
            os.path.join(dic_folder, "es_dic", "es", "es_ES.dic"),
            os.path.join(dic_folder, "es_dic", "es_ES.dic"),
            os.path.join(dic_folder, "es_dic", "es.dic"),
        ],
        "pt": [
            os.path.join(dic_folder, "pt_dic", "pt_BR", "pt_BR.dic"),
            os.path.join(dic_folder, "pt_dic", "pt_PT", "pt_PT.dic"),
            os.path.join(dic_folder, "pt_dic", "pt.dic"),
        ],
        "de": [
            os.path.join(dic_folder, "de_dic", "de_DE_frami.dic"),
            os.path.join(dic_folder, "de_dic", "de_DE.dic"),
            os.path.join(dic_folder, "de_dic", "de.dic"),
        ],
    }
    candidates.extend(fallback_map.get(lang_prefix, []))

    # De-duplicate while preserving order.
    deduped: List[str] = []
    seen = set()
    for p in candidates:
        key = os.path.normcase(os.path.normpath(p))
        if key not in seen:
            seen.add(key)
            deduped.append(p)
    return deduped


def resolve_hunspell_paths(lang_code: str, dic_folder: str = "dics") -> Dict[str, Any]:
    """Resolve the first existing (.dic, .aff) pair for a language."""
    lang_prefix = normalize_language_code(lang_code)
    checks: List[Dict[str, Any]] = []

    for dic_path in _build_hunspell_dic_candidates(lang_prefix, dic_folder=dic_folder):
        aff_path = os.path.splitext(dic_path)[0] + ".aff"
        dic_exists = os.path.exists(dic_path)
        aff_exists = os.path.exists(aff_path)
        checks.append({
            "dic": dic_path,
            "aff": aff_path,
            "dic_exists": dic_exists,
            "aff_exists": aff_exists,
        })
        if dic_exists and aff_exists:
            return {
                "language": lang_prefix,
                "ok": True,
                "dic": dic_path,
                "aff": aff_path,
                "checks": checks,
            }

    return {
        "language": lang_prefix,
        "ok": False,
        "dic": "",
        "aff": "",
        "checks": checks,
    }


def batch_filter_tokens_by_dictionary(input_folder: str, target_languages: List[str], 
                                     dic_folder: str = "dics", 
                                     output_folder: str = None) -> List[Dict]:
    """
    Batch process all token files in a folder using dictionary filtering with affix rules.
    
    Processes token files for each target language, filters them against Hunspell dictionaries
    (using affix rules for morphological matching), and saves results to dic format.
    
    Args:
        input_folder: Folder containing token files to filter
        target_languages: List of language codes to process (e.g., ['es', 'pt', 'en'])
        dic_folder: Folder containing dictionary files (default: "dics")
        output_folder: Folder to save filtered results. If None, uses OUTPUT_DIR from hyperparameters
        
    Returns:
        List[Dict]: Processing summary for each file, containing:
            - language: Language code
            - input_file: Input filename
            - output_file: Output filename
            - original_tokens: Count of input tokens
            - removed_tokens: Count of tokens matching dictionary
            - remaining_tokens: Count of neologisms (new tokens)
            - processing_time: Seconds to process
            - removal_rate: Percentage of tokens removed
    """
    
    # Use hyperparameter if output_folder not specified
    if output_folder is None:
        output_folder = OUTPUT_DIR
    
    # Create output folder if it doesn't exist
    if not os.path.exists(output_folder):
        os.makedirs(output_folder)
    
    # Track processing statistics
    total_processed = 0
    total_errors = 0
    total_skipped = 0
    processing_summary: List[Dict] = []
    
    print("="*80)
    print("BATCH DICTIONARY FILTERING WITH AFFIX RULES")
    print("="*80)
    print(f"Input folder: {input_folder}")
    print(f"Target languages: {target_languages}")
    print(f"Dictionary folder: {dic_folder}")
    print(f"Output folder: {output_folder}")
    print("="*80)
    
    # Process each target language
    for lang_code in target_languages:
        # Normalize language code to 2 letters
        try:
            lang_prefix = normalize_language_code(lang_code)
        except ValueError as e:
            print(f"❌ Invalid language code '{lang_code}': {e}")
            total_errors += 1
            continue
        
        print(f"\n🌐 Processing language: {lang_code} (normalized: {lang_prefix})")
        print("-" * 50)
        
        # Resolve dictionary/affix paths with fallback candidates.
        resolved_paths = resolve_hunspell_paths(lang_prefix, dic_folder=dic_folder)
        if not resolved_paths["ok"]:
            print(f"❌ No valid dictionary+affix pair found for language '{lang_code}'")
            for chk in resolved_paths["checks"]:
                print(
                    f"   - DIC {'OK' if chk['dic_exists'] else 'MISSING'} | "
                    f"AFF {'OK' if chk['aff_exists'] else 'MISSING'} :: {chk['dic']}"
                )
            total_errors += 1
            continue

        dic_file_path = resolved_paths["dic"]
        aff_file_path = resolved_paths["aff"]
        
        print(f"✅ Dictionary files found:")
        print(f"   DIC: {dic_file_path}")
        print(f"   AFF: {aff_file_path}")
        
        # Find all token files for this language
        # Pattern: *{lang_code}*tokens*.txt or *{lang_prefix}*tokens*.txt
        token_patterns = [
            os.path.join(input_folder, f"*{lang_code}*tokens*.txt"),
            os.path.join(input_folder, f"*{lang_prefix}*tokens*.txt")
        ]
        token_files = []
        for pattern in token_patterns:
            token_files.extend(glob.glob(pattern))
        
        # Remove duplicates while preserving order
        token_files = list(dict.fromkeys(token_files))
        
        if not token_files:
            print(f"⏭️  No token files found for pattern: {token_patterns}")
            total_skipped += 1
            continue
            
        print(f"📁 Found {len(token_files)} token file(s) for {lang_prefix}:")
        
        # Process each token file for this language
        for token_file in token_files:
            token_filename = os.path.basename(token_file)
            print(f"\n  📄 Processing: {token_filename}")
            
            try:
                # Generate output filename by replacing 'tokens' with 'filtered_tokens'
                if 'tokens' in token_filename:
                    filtered_filename = token_filename.replace('tokens', 'filtered_tokens')
                    filtered_filename = filtered_filename.replace('.txt', '.dic')
                else:
                    base_name = Path(token_filename).stem
                    filtered_filename = f"{base_name}_filtered_tokens.dic"
                
                output_path = os.path.join(output_folder, filtered_filename)
                
                # Check if output already exists
                if os.path.exists(output_path):
                    print(f"  ⏭️  Output already exists: {filtered_filename} - skipping")
                    total_skipped += 1
                    continue
                
                # Perform filtering
                start_time = time.time()
                result = filter_tokens_by_dictionary_with_affixes(
                    token_file,      # Input token file
                    dic_file_path,   # Dictionary file
                    aff_file_path,   # Affix file
                    output_path      # Output file
                )
                end_time = time.time()
                
                # Calculate statistics
                processing_time = end_time - start_time
                removal_rate = (result['removed_tokens'] / result['original_txt_tokens'] * 100) if result['original_txt_tokens'] > 0 else 0
                
                print(f"  ✅ Successfully processed in {processing_time:.2f}s:")
                print(f"     Original tokens: {result['original_txt_tokens']:,}")
                print(f"     Removed tokens: {result['removed_tokens']:,} ({removal_rate:.1f}%)")
                print(f"     Remaining tokens: {result['remaining_tokens']:,}")
                print(f"     Output: {filtered_filename}")
                
                # Store summary for final report
                processing_summary.append({
                    'language': lang_code,
                    'input_file': token_filename,
                    'output_file': filtered_filename,
                    'original_tokens': result['original_txt_tokens'],
                    'removed_tokens': result['removed_tokens'],
                    'remaining_tokens': result['remaining_tokens'],
                    'processing_time': processing_time,
                    'removal_rate': removal_rate
                })
                
                total_processed += 1
                
            except Exception as e:
                print(f"  ❌ Error processing {token_filename}: {e}")
                total_errors += 1
    
    # Print final summary
    print("\n" + "="*80)
    print("📊 BATCH PROCESSING SUMMARY")
    print("="*80)
    print(f"Total files processed: {total_processed}")
    print(f"Total errors: {total_errors}")
    print(f"Total skipped: {total_skipped}")
    
    if processing_summary:
        print(f"\n📈 DETAILED RESULTS:")
        print("-" * 80)
        
        # Group by language for better organization
        by_language: Dict[str, List[Dict]] = {}
        for item in processing_summary:
            lang = item['language']
            if lang not in by_language:
                by_language[lang] = []
            by_language[lang].append(item)
        
        total_original = sum(item['original_tokens'] for item in processing_summary)
        total_removed = sum(item['removed_tokens'] for item in processing_summary)
        total_remaining = sum(item['remaining_tokens'] for item in processing_summary)
        total_time = sum(item['processing_time'] for item in processing_summary)
        
        for lang, items in by_language.items():
            print(f"\n🌐 {lang.upper()}:")
            for item in items:
                print(f"  📄 {item['input_file']}")
                print(f"     → {item['remaining_tokens']:,} tokens ({item['removal_rate']:.1f}% removed)")
        
        print(f"\n📊 OVERALL STATISTICS:")
        print(f"   Total original tokens: {total_original:,}")
        print(f"   Total removed tokens: {total_removed:,}")
        print(f"   Total remaining tokens: {total_remaining:,}")
        if total_original > 0:
            print(f"   Overall removal rate: {(total_removed/total_original*100):.1f}%")
        print(f"   Total processing time: {total_time:.2f}s ({total_time/60:.2f} minutes)")
        
        if total_processed > 0:
            print(f"   Average processing time: {total_time/total_processed:.2f}s per file")
    
    print(f"\n🎯 Next steps:")
    print(f"   - Check filtered files in: {output_folder}/")
    print(f"   - Review remaining tokens for quality")
    print(f"   - Use filtered tokens for translation validation or dictionary assembly")
    
    return processing_summary


# ==============================================================================
# USAGE EXAMPLE - Configure with hyperparameters
# ==============================================================================

# Use INTERMEDIARY_DIR for testing before final assembly
# Use OUTPUT_DIR for production dictionary outputs

# Recommended workflow:
# 1. First run: batch_filter_tokens_by_dictionary(..., output_folder=INTERMEDIARY_DIR)
# 2. Review results and validate quality
# 3. Final run: batch_filter_tokens_by_dictionary(..., output_folder=OUTPUT_DIR)

# Example (uncomment to run):
# TARGET_LANGUAGES = ["es", "pt", "en", "fr", "de"]
# INPUT_FOLDER = os.path.join(INTERMEDIARY_DIR, "raw_tokens")
# 
# batch_results = batch_filter_tokens_by_dictionary(
#     input_folder=INPUT_FOLDER,
#     target_languages=TARGET_LANGUAGES,
#     dic_folder=DIC_FOLDER,
#     output_folder=INTERMEDIARY_DIR  # or OUTPUT_DIR for final output
# )


## 4. Find wordforms in corpus

In [5]:
import json
import csv
from concurrent.futures import ThreadPoolExecutor, as_completed
from time import time as _time_now
try:
    from lingua import Language, LanguageDetectorBuilder
    _LINGUA_AVAILABLE = True
except ImportError:
    _LINGUA_AVAILABLE = False
    print("  [corpus] WARNING: lingua-language-detector not installed.\n"
          "           Run:  pip install lingua-language-detector")

# ══════════════════════════════════════════════════════════════════════════════
# SECTION: Gender ghost generation for proper nouns
# ══════════════════════════════════════════════════════════════════════════════

# Language-specific suffix tables for generating "ghost" gender/plural forms
# from proper-noun base words.  These bypass standard AFF condition patterns
# so that words with non-standard endings (e.g. "bwork" → "bworka") are still
# checked against the i18n corpus.
#
# Each entry: {lang: {"suffixes": [...], "accent_map": {ending: replacement}}}
# accent_map handles accent-shifting: "jalatín" → strip accent → "jalatin" + "a"

GENDER_GHOST_SUFFIXES: Dict[str, dict] = {
    "es": {
        # Spanish: masculine -o → feminine -a, plurals -s/-es
        "suffixes": ["a", "as", "os", "es"],
        "accent_map": {
            "ín": "in",   # jalatín → jalatina / jalatinas
            "ón": "on",   # dragón  → dragona  / dragonas
            "án": "an",   # capitán → capitana / capitanas
            "és": "es",   # francés → francesa / francesas
        },
    },
    "pt": {
        # Portuguese: masculine → feminine -a, plurals -s/-es, -ão/-ões
        "suffixes": ["a", "as", "os", "es", "ões"],
        "accent_map": {
            "ão": "ã",    # dragão  → dragã    (rare for monsters)
            "ín": "in",
            "ón": "on",
            "ês": "es",   # português → portuguesa
        },
    },
    "de": {
        # German: feminine -in/-innen (not accent-dependent)
        "suffixes": ["in", "innen"],
        "accent_map": {},
    },
    "en": {
        # English: no grammatical gender inflection
        "suffixes": [],
        "accent_map": {},
    },
    "fr": {
        # French: not used (source language)
        "suffixes": [],
        "accent_map": {},
    },
}

# Spanish stems that repeatedly generate verb-like false positives in
# brute-force AFF expansion and should be excluded from ghost/noise paths.
SPANISH_GHOST_NOISE_BASES: Set[str] = {'tar', 'star', 'sir', 'over'}


def _generate_gender_ghosts(base_lower: str, lang: str) -> Set[str]:
    """
    Generate "ghost" gender/plural forms for a proper-noun base word.

    These are hypothetical inflected forms that might exist in the corpus
    even though the standard Hunspell AFF rules don't have conditions
    matching the stem ending (e.g. Spanish -o/-a doesn't fire for -k stems).

    Returns a set of lowercased candidate forms (excluding the base itself).
    """
    cfg = GENDER_GHOST_SUFFIXES.get(lang)
    if not cfg or not cfg["suffixes"]:
        return set()

    # Spanish guardrails for short/proper-name stems that tend to create
    # verb-like noise (e.g. tar -> tado, star -> prestadle).
    if lang == 'es':
        if base_lower in SPANISH_GHOST_NOISE_BASES:
            return set()
        if len(base_lower) <= 3:
            return set()
        if len(base_lower) <= 4 and base_lower.endswith('r'):
            return set()

    ghosts: Set[str] = set()
    suffixes  = cfg["suffixes"]
    acmap     = cfg["accent_map"]

    # ── Accent-neutralised stem ───────────────────────────────────────────────
    # If the word ends with an accented pattern, produce forms from the
    # de-accented stem + suffix.  E.g. "jalatín" → stem "jalatin", ghosts
    # "jalatina", "jalatinas".
    neutralised_stem = None
    for ending, replacement in acmap.items():
        if base_lower.endswith(ending):
            neutralised_stem = base_lower[:-len(ending)] + replacement
            break

    # ── Direct suffixing (for non-standard endings like "bwork") ─────────────
    for sfx in suffixes:
        ghosts.add(base_lower + sfx)

    # ── From neutralised stem ─────────────────────────────────────────────────
    if neutralised_stem:
        for sfx in suffixes:
            ghosts.add(neutralised_stem + sfx)
        # Also add the bare neutralised stem (e.g. "jalatin" without suffix)
        ghosts.add(neutralised_stem)

    ghosts.discard(base_lower)
    return ghosts


# ══════════════════════════════════════════════════════════════════════════════
# SECTION: Find missing word forms in i18n corpus
# ══════════════════════════════════════════════════════════════════════════════

# Maps the 5 two-letter game language codes to lingua Language enum values.
# Detector is built from all five so it can distinguish the target language
# from any of the four others that might appear as untranslated fallbacks.
_LINGUA_LANG_MAP: Dict[str, object] = {}
if _LINGUA_AVAILABLE:
    _LINGUA_LANG_MAP = {
        "fr": Language.FRENCH,       # type: ignore[union-attr]
        "en": Language.ENGLISH,      # type: ignore[union-attr]
        "es": Language.SPANISH,      # type: ignore[union-attr]
        "pt": Language.PORTUGUESE,   # type: ignore[union-attr]
        "de": Language.GERMAN,       # type: ignore[union-attr]
    }

# Backup of the original i18n_PATHS values, populated once on the first call
# to load_i18n_corpus.  Used to auto-recover when the in-place mutation points
# to a filtered file that has since been deleted.
_i18n_PATHS_original: Dict[str, Dict[str, str]] = {}


def load_i18n_corpus(
    lang            : str,
    games           : List[str],
    source_type     : str   = 'i18n',
    lang_detect     : bool  = True,
    min_words       : int   = 7,
    min_confidence  : float = 0.12,
    short_confidence: float = 0.9,
) -> Dict[str, Set[str]]:
    """
    Load and tokenize i18n strings for the given lang/games.

    Handles two file formats:
      * JSON        (DOFUS):  {"entries": {"<id>": "<text>", ...}}
      * .properties (WAKFU):  dotted.key=translated value  (Java properties)

    Each raw string pipeline:
      has_wip_markers() -> skip  ->  remove_html_tags()
      -> demorph_string()  ->  [lang_detect: skip if wrong language]
      -> tokenize_text()  ->  record true-case + lowercase key

    demorph_string() is intentionally called BEFORE language detection so that
    game-engine morphology markup such as {[1*]?a:o} or {~f} is expanded into
    plain text first.  Leaving the raw markup in the string confuses the lingua
    detector and causes false positives (e.g. "Repurgador{[1*]?a:} cristalin{[1*]?a:o}"
    being mis-classified as French/English and dropped even though the underlying
    content is valid Spanish).

    Language detection (lang_detect=True)
    --------------------------------------
    Entries absent in the target language are sometimes shown in French or
    English as a fallback.  The lingua detector filters those out by checking
    the confidence that each string belongs to *lang*.

    Two thresholds are applied:
      * Strings with >= min_words words  ->  min_confidence   (default 0.5)
        Most sentences only pass if the model is reasonably confident.
      * Strings with <  min_words words  ->  short_confidence (default 0.9)
        Very strict for short strings to avoid discarding game-specific
        neologisms, item names and monster names that the model might
        mis-classify (short text detection is inherently less reliable).

    Re-run detection
    ----------------
    If i18n_PATHS[game][lang] already points to a previously-written filtered
    file (_i18n_filtered.*), language detection is skipped for that entry to
    avoid double-removal.  A notice is printed to make re-runs visible.

    Side effects when lang_detect=True (first run only):
      * A filtered copy of each i18n file is written to INTERMEDIARY_DIR.
      * A removed-entries CSV is written to INTERMEDIARY_DIR with every
        excluded string, its confidence score, word count and threshold used.
      * i18n_PATHS[game][lang] is updated in-place to point to the filtered
        file so that subsequent calls in the same kernel session are
        idempotent.

    Args:
        lang            : 2-letter language code ('es', 'pt', 'en', 'de')
        games           : Game names matching keys in i18n_PATHS  (e.g. ['WAKFU'])
        source_type     : 'i18n' (only mode currently implemented)
        lang_detect     : Enable lingua language-detection filtering (default True)
        min_words       : Word-count boundary between the two thresholds
        min_confidence  : Minimum p(target lang) for strings with >= min_words words
        short_confidence: Minimum p(target lang) for strings with <  min_words words

    Returns:
        Dict[str, Set[str]]:
            Keys   = lowercased tokens
            Values = set of all original-case forms seen for that key in the corpus
            Example: {'jalatines': {'jalatines', 'Jalatines'}, 'abella': {'Abella'}}
    """
    if source_type != 'i18n':
        raise NotImplementedError(
            f"source_type={source_type!r} — only 'i18n' is currently supported"
        )

    # ── Lazily back up original paths before any in-place mutation ─────────────
    # _i18n_PATHS_original holds the pre-mutation values so we can auto-recover
    # if i18n_PATHS[game][lang] was previously mutated to point to a filtered
    # file that has since been deleted (e.g. after a kernel restart or cleanup).
    global _i18n_PATHS_original
    for _g, _paths in i18n_PATHS.items():
        if isinstance(_paths, dict) and _g not in _i18n_PATHS_original:
            _i18n_PATHS_original[_g] = dict(_paths)

    corpus_map: Dict[str, Set[str]] = {}
    total_strings  = 0
    skipped_wip    = 0
    skipped_lang   = 0
    removed_entries: List[dict] = []   # {game, key, raw_val, demorphed_val, confidence, word_count}

    def _add_tokens(tokens):
        for tok in tokens:
            key = tok.lower()
            if key not in corpus_map:
                corpus_map[key] = set()
            corpus_map[key].add(tok)

    # ── Build lingua detector (once per call) ────────────────────────────────
    _lingua_detector  = None
    _lingua_target    = None
    _effective_detect = lang_detect and _LINGUA_AVAILABLE

    if lang_detect and not _LINGUA_AVAILABLE:
        print("  [corpus] WARNING: lingua not installed — "
              "language detection skipped for this call")
    elif _effective_detect:
        if lang not in _LINGUA_LANG_MAP:
            print(f"  [corpus] WARNING: lang={lang!r} not in _LINGUA_LANG_MAP — "
                  "language detection disabled for this call")
            _effective_detect = False
        else:
            _lingua_target   = _LINGUA_LANG_MAP[lang]
            _lingua_detector = (
                LanguageDetectorBuilder              # type: ignore[union-attr]
                .from_languages(*_LINGUA_LANG_MAP.values())
                .build()
            )
            other_langs = [
                str(l).split(".")[-1]
                for l in _LINGUA_LANG_MAP.values()
                if l != _lingua_target
            ]
            print(f"  [corpus] Lingua detector  : target={lang.upper()}"
                  f"  fallbacks={other_langs}")
            print(f"  [corpus] Thresholds       : short(<{min_words}w)=compare vs EN/FR"
                  f"  long(>={min_words}w)={min_confidence}")

    def _strip_for_detection(text: str) -> str:
        """Remove leftover game-engine markup that would confuse the lingua detector.

        After demorph_string(), placeholder tags like [#1], [>1] and unexpanded
        curly-brace blocks {…} may still be present.  Strip them so the detector
        sees only plain words."""
        text = re.sub(r'\[[^\]]*\]', ' ', text)   # [#1], [>1], [1*], …
        text = re.sub(r'\{[^}]*\}',  ' ', text)   # any remaining {…} markup
        return re.sub(r'\s+', ' ', text).strip()

    def _score_language(text: str):
        """Return (is_wrong, confidence, word_count, threshold).

        Long strings (>= min_words):  require p(target) >= min_confidence.

        Short strings (< min_words):  instead of the strict 0.9 fixed threshold,
        compare p(target) against p(EN) and p(FR).  A short string is considered
        wrong only when *both* EN and FR score higher than the target language.
        Special cases:
          * target == FR  ->  skip filtering entirely (no self-comparison).
          * target == EN  ->  compare against FR only (no self-comparison).
        """
        if _lingua_detector is None or _lingua_target is None:
            return False, 1.0, 0, 0.0
        clean = _strip_for_detection(text)
        word_count = len(clean.split()) if clean else 0
        if word_count == 0:
            return False, 1.0, 0, 0.0
        if word_count >= min_words:
            confidence = _lingua_detector.compute_language_confidence(clean, _lingua_target)  # type: ignore[union-attr]
            return confidence < min_confidence, confidence, word_count, min_confidence
        # ── Short-string path: compare target vs EN / FR ─────────────────────
        _lang_fr = _LINGUA_LANG_MAP.get('fr')
        _lang_en = _LINGUA_LANG_MAP.get('en')
        # If target IS French, no point in detection (would compare FR vs FR)
        if _lingua_target == _lang_fr:
            return False, 1.0, word_count, 0.0
        confidence = _lingua_detector.compute_language_confidence(clean, _lingua_target)  # type: ignore[union-attr]
        comparators = []
        if _lang_en is not None and _lingua_target != _lang_en:
            comparators.append(_lingua_detector.compute_language_confidence(clean, _lang_en))  # type: ignore[union-attr]
        if _lang_fr is not None:
            comparators.append(_lingua_detector.compute_language_confidence(clean, _lang_fr))  # type: ignore[union-attr]
        # Wrong only if target score is strictly below ALL comparators
        is_wrong = bool(comparators) and all(confidence < c for c in comparators)
        return is_wrong, confidence, word_count, 0.0

    for game in games:
        paths_for_game = i18n_PATHS.get(game, {})
        if not paths_for_game:
            print(f"  [corpus] {game}: no i18n_PATHS entry — skipping")
            continue

        full_path = paths_for_game.get(lang, "")
        if not full_path or not os.path.exists(full_path):
            # Auto-recover: if the path is a stale filtered file, fall back to
            # the original path stored in _i18n_PATHS_original.
            _filtered_names = (
                f"{game}_{lang}_i18n_filtered.json",
                f"{game}_{lang}_i18n_filtered.properties",
            )
            if full_path and os.path.basename(full_path) in _filtered_names:
                _orig = (_i18n_PATHS_original.get(game) or {}).get(lang, "")
                if _orig and os.path.exists(_orig):
                    print(f"  [corpus] {game}/{lang}: filtered file missing, "
                          f"reverting to original -> {_orig!r}")
                    i18n_PATHS[game][lang] = _orig
                    full_path = _orig
                else:
                    print(f"  [corpus] {game}/{lang}: filtered file missing and original "
                          f"not found either ({_orig!r}) — skipping.  "
                          f"Re-run the config cell to reset i18n_PATHS.")
                    continue
            else:
                print(f"  [corpus] {game}/{lang}: file not found -> {full_path!r} — skipping")
                continue

        # Detect re-run: path already points to a previously-filtered file.
        # Skip detection to avoid double-removal and overwriting the same file.
        _already_filtered = os.path.basename(full_path) in (
            f"{game}_{lang}_i18n_filtered.json",
            f"{game}_{lang}_i18n_filtered.properties",
        )
        if _already_filtered:
            print(f"  [corpus] {game}/{lang}: reading already-filtered file "
                  f"(re-run detected — lang detection skipped)")
        else:
            print(f"  [corpus] Loading {game}/{lang}: {full_path}")

        _detect_this = _effective_detect and not _already_filtered
        game_count     = 0
        game_lang_skip = 0

        # ── JSON branch  (DOFUS: {"entries": {"id": "text"}}) ─────────────────
        if full_path.endswith('.json'):
            with open(full_path, 'r', encoding='utf-8') as fh:
                data = json.load(fh)
            entries = data.get('entries', data)
            filtered_entries: Dict[str, str] = {}
            for entry_id, raw_val in entries.items():
                if not isinstance(raw_val, str):
                    continue
                total_strings += 1
                if has_wip_markers(raw_val):
                    skipped_wip += 1
                    continue
                clean_val     = remove_html_tags(raw_val)
                demorphed_val = demorph_string(clean_val)   # expand markup BEFORE detection
                if _detect_this:
                    is_wrong, conf, wc, thr = _score_language(demorphed_val)
                    if is_wrong:
                        skipped_lang   += 1
                        game_lang_skip += 1
                        removed_entries.append({
                            'game'         : game,
                            'key'          : entry_id,
                            'confidence'   : round(conf, 4),
                            'word_count'   : wc,
                            'threshold'    : thr,
                            'raw_val'      : raw_val,
                            'demorphed_val': demorphed_val,
                        })
                        continue
                filtered_entries[entry_id] = raw_val
                _add_tokens(tokenize_text(demorphed_val, lang))
                game_count += 1

            # Write filtered JSON and update path mapping
            if _detect_this:
                os.makedirs(INTERMEDIARY_DIR, exist_ok=True)
                filtered_path = os.path.join(
                    INTERMEDIARY_DIR, f"{game}_{lang}_i18n_filtered.json"
                )
                with open(filtered_path, 'w', encoding='utf-8') as fh:
                    json.dump({"entries": filtered_entries}, fh,
                              ensure_ascii=False, indent=2)
                i18n_PATHS[game][lang] = filtered_path
                print(f"  [corpus]   -> lang-filtered : {game_lang_skip:,} removed")
                print(f"  [corpus]   -> written       : {filtered_path}")
                print(f"  [corpus]   -> i18n_PATHS[{game!r}][{lang!r}] updated in-place")

        # ── .properties branch  (WAKFU: dotted.key=translated value) ──────────
        elif full_path.endswith('.properties'):
            accepted_lines: List[str] = []
            with open(full_path, 'r', encoding='utf-8') as fh:
                for line_raw in fh:
                    line = line_raw.rstrip('\n')
                    # Preserve blank lines and comment lines in the filtered file
                    if not line.strip() or line.lstrip().startswith('#'):
                        if _detect_this:
                            accepted_lines.append(line)
                        continue
                    if '=' not in line:
                        if _detect_this:
                            accepted_lines.append(line)
                        continue
                    _, _, raw_val = line.partition('=')
                    if not raw_val.strip():
                        if _detect_this:
                            accepted_lines.append(line)
                        continue
                    total_strings += 1
                    if has_wip_markers(raw_val):
                        skipped_wip += 1
                        continue
                    clean_val     = remove_html_tags(raw_val)
                    demorphed_val = demorph_string(clean_val)   # expand markup BEFORE detection
                    if _detect_this:
                        is_wrong, conf, wc, thr = _score_language(demorphed_val)
                        if is_wrong:
                            skipped_lang   += 1
                            game_lang_skip += 1
                            prop_key = line.partition('=')[0].strip()
                            removed_entries.append({
                                'game'         : game,
                                'key'          : prop_key,
                                'confidence'   : round(conf, 4),
                                'word_count'   : wc,
                                'threshold'    : thr,
                                'raw_val'      : raw_val.strip(),
                                'demorphed_val': demorphed_val,
                            })
                            continue
                        accepted_lines.append(line)
                    _add_tokens(tokenize_text(demorphed_val, lang))
                    game_count += 1
            # Write filtered .properties and update path mapping
            if _detect_this:
                os.makedirs(INTERMEDIARY_DIR, exist_ok=True)
                filtered_path = os.path.join(
                    INTERMEDIARY_DIR, f"{game}_{lang}_i18n_filtered.properties"
                )
                with open(filtered_path, 'w', encoding='utf-8') as fh:
                    fh.write('\n'.join(accepted_lines))
                    if accepted_lines:
                        fh.write('\n')
                i18n_PATHS[game][lang] = filtered_path
                print(f"  [corpus]   -> lang-filtered : {game_lang_skip:,} removed")
                print(f"  [corpus]   -> written       : {filtered_path}")
                print(f"  [corpus]   -> i18n_PATHS[{game!r}][{lang!r}] updated in-place")

        else:
            print(f"  [corpus] {game}/{lang}: unsupported format for {full_path!r} — skipping")
            continue

        print(f"  [corpus]   -> {game_count:,} strings processed from {game}")

    print(f"  [corpus] ─────────────────────────────────────────────────────")
    print(f"  [corpus] Total strings  : {total_strings:,}")
    print(f"  [corpus] WIP skipped    : {skipped_wip:,}")
    if _effective_detect:
        filter_rate = (skipped_lang / total_strings * 100) if total_strings else 0.0
        print(f"  [corpus] Lang filtered  : {skipped_lang:,}"
              f"  ({filter_rate:.1f}%  p(target)<threshold)")
    print(f"  [corpus] Unique tokens  : {len(corpus_map):,}")

    # ── Save removed-entries log ──────────────────────────────────────────────
    if _effective_detect and removed_entries:
        os.makedirs(INTERMEDIARY_DIR, exist_ok=True)
        games_tag   = '_'.join(games)
        removed_csv = os.path.join(
            INTERMEDIARY_DIR, f"{games_tag}_{lang}_i18n_langdetect_removed.csv"
        )
        _removed_fields = ['game', 'key', 'confidence', 'word_count',
                           'threshold', 'raw_val', 'demorphed_val']
        with open(removed_csv, 'w', newline='', encoding='utf-8-sig') as _fh:
            _w = csv.DictWriter(_fh, fieldnames=_removed_fields)
            _w.writeheader()
            _w.writerows(removed_entries)
        print(f"  [corpus] Removed log    : {removed_csv}  ({len(removed_entries)} entries)")
        print(f"  [corpus] ─────────────────────────────────────────────────────")
        print(f"  [corpus] Removed entries (sorted by confidence asc):")
        for _r in sorted(removed_entries, key=lambda x: x['confidence'])[:10]:
            _raw_preview = (_r['raw_val'][:90] + '…') if len(_r['raw_val']) > 90 else _r['raw_val']
            _dem_preview = (_r['demorphed_val'][:90] + '…') if len(_r['demorphed_val']) > 90 else _r['demorphed_val']
            print(f"    [{_r['game']}]  conf={_r['confidence']:.3f}  "
                  f"wc={_r['word_count']:>2}  thr={_r['threshold']}  "
                  f"key={_r['key']}")
            print(f"      raw      : {_raw_preview}")
            print(f"      demorphed: {_dem_preview}")
        if len(removed_entries) > 10:
            print(f"    ... ({len(removed_entries) - 10} more — see {removed_csv})")

    return corpus_map


# Cache: lang -> fully-expanded standard-dic forms set (avoids re-expansion per session)
_std_dic_forms_cache: Dict[str, Set[str]] = {}


def build_std_dic_forms(lang: str, num_threads: int = 4) -> Set[str]:
    """
    Return every surface form (lowercase) that the standard Hunspell dictionary for *lang*
    recognises, including all affix-expanded inflections.

    The result is cached per lang in ``_std_dic_forms_cache`` so repeated calls within
    the same kernel session are essentially free.

    Replicates the expansion logic used by ``filter_tokens_by_dictionary_with_affixes``,
    delegating to the already-defined ``_expand_chunk`` thread worker.
    """
    global _std_dic_forms_cache
    if lang in _std_dic_forms_cache:
        print(f"  Standard dic forms ({lang}) : {len(_std_dic_forms_cache[lang]):,}  [cached]")
        return _std_dic_forms_cache[lang]

    t0 = _time_now()
    std_dic_path = HUNSPELL_PATHS.get(lang, "")
    if not std_dic_path:
        raise ValueError(f"No HUNSPELL_PATHS entry for lang={lang!r}")
    aff_path = std_dic_path.replace(".dic", ".aff")

    print(f"  Expanding standard dic ({lang})...")
    affixes = parse_aff_file(aff_path)

    _dic_enc = affixes.get('encoding', 'utf-8')
    if isinstance(_dic_enc, str) and _dic_enc.lower() in ('utf-8', 'utf8'):
        _dic_enc = 'utf-8-sig'          # handle optional BOM

    forbidden: Set[str] = set()
    entries: List[tuple] = []           # (base_lower, flags_str)

    try:
        with open(std_dic_path, 'r', encoding=_dic_enc, errors='replace') as fh:
            raw_lines = fh.readlines()
    except Exception as e:
        print(f"    Warning: could not read {std_dic_path!r}: {e}")
        return set()

    for ln in raw_lines[1:]:            # skip count header
        ln = ln.strip()
        if not ln:
            continue
        parts = ln.split('/', 1)
        base  = parts[0].split('\t')[0].rstrip('.')
        flags = parts[1].split('\t')[0] if len(parts) > 1 else ''
        base_lower = base.lower()
        if not base_lower:
            continue
        # Collect FORBIDDENWORD entries to exclude them later
        if 'FORBIDDENWORD' in affixes and flags:
            fw_flag = affixes.get('FORBIDDENWORD', '')
            if fw_flag and fw_flag in flags:
                forbidden.add(base_lower)
                continue
        entries.append((base_lower, flags))

    # Parallel AFF expansion via the already-defined _expand_chunk worker
    chunk_size = max(1, len(entries) // (num_threads * 4))
    chunks = [entries[i:i + chunk_size] for i in range(0, len(entries), chunk_size)]

    all_forms: Set[str] = set()
    with ThreadPoolExecutor(max_workers=num_threads) as pool:
        futures = [pool.submit(_expand_chunk, chunk, affixes) for chunk in chunks]
        for fut in as_completed(futures):
            all_forms.update(fut.result())

    # Remove any forms that belong to FORBIDDENWORD headwords
    all_forms -= forbidden
    # Add plain headwords (some headwords have no flags)
    all_forms.update(base for base, _ in entries)

    # CHECKSHARPS: ß <-> ss variants
    if affixes.get('CHECKSHARPS'):
        all_forms.update({f.replace('ß', 'ss') for f in all_forms if 'ß' in f})
        all_forms.update({f.replace('ss', 'ß') for f in all_forms if 'ss' in f})

    elapsed = _time_now() - t0
    print(f"  Standard dic forms ({lang}) : {len(all_forms):,}  ({elapsed:.1f}s)")
    _std_dic_forms_cache[lang] = all_forms
    return all_forms


def _wordform_match_worker(
    batch: List[str],
    affixes: Dict,
    corpus_map: Dict[str, Set[str]],
    all_flags: List[str],
    propernoun_lower: frozenset = frozenset(),
    lang: str = "es",
    candidate_flags: List[str] | None = None,
    quorum: float = 0.5,
) -> List[dict]:
    """
    Thread worker — brute-forces all SFX+PFX flags against each base word,
    then intersects the generated (lowercased) forms with corpus_map keys.

    For base words in propernoun_lower, also generates gender "ghost" forms
    via _generate_gender_ghosts() and checks them against the corpus.  Ghost
    hits are tracked separately so the munch step can distinguish AFF-derived
    forms from gender ghosts.

    Flag attribution (munch pre-computation)
    -----------------------------------------
    When ``candidate_flags`` is provided, also tests each flag individually
    and records which flags pass the corpus quorum.  This avoids repeating
    the same generate_word_forms() + corpus-lookup work in a separate munch
    pass.  ``validated_flags`` is a list of flag strings whose generated
    forms met the quorum threshold against the corpus.

    True-case forms: all original-case variants stored in corpus_map[key] are
    collected, so 'jalatines' AND 'Jalatines' (if both appear in corpus) are
    both returned.

    Exclusion: any generated form whose lowercase equals base_lower is excluded
    (the base word is already in the custom dic).

    Returns:
        List of dicts {base_word, found_forms, count, ghost_forms, ghost_count,
        validated_flags}  -- only entries with >=1 hit (AFF or ghost).
        Words with 0 AFF/ghost hits but >=1 validated_flag are also included.
    """
    # Build a set of all lowercased corpus keys for fast quorum checking
    _corpus_keys: frozenset = frozenset(corpus_map.keys()) if candidate_flags else frozenset()

    results_batch = []
    for base_word in batch:
        base_lower = base_word.lower()

        # Targeted Spanish noise stems: skip brute-force generation entirely
        # so they do not create false verb-like families in outputs.
        if lang == 'es' and base_lower in SPANISH_GHOST_NOISE_BASES:
            continue
        try:
            forms_lower = generate_word_forms(base_lower, all_flags, affixes)
        except Exception:
            forms_lower = set()

        # Collect all true-case variants for each matching lowercase form,
        # excluding the base word itself.
        true_case_hits: Set[str] = set()
        for form_lower in forms_lower:
            if form_lower == base_lower:
                continue
            if form_lower in corpus_map:
                true_case_hits.update(corpus_map[form_lower])

        # ── Gender ghost forms for proper nouns ───────────────────────────
        ghost_true_case_hits: Set[str] = set()
        if base_lower in propernoun_lower:
            ghost_forms_lower = _generate_gender_ghosts(base_lower, lang)
            # Exclude forms already found via AFF (avoid double-counting)
            ghost_forms_lower -= forms_lower
            ghost_forms_lower.discard(base_lower)
            for gf in ghost_forms_lower:
                if gf in corpus_map:
                    ghost_true_case_hits.update(corpus_map[gf])
            # Also remove any ghost hits that were already AFF hits
            ghost_true_case_hits -= true_case_hits

        # ── Per-flag attribution for munch ────────────────────────────────
        validated: List[str] = []
        if candidate_flags:
            for flag in candidate_flags:
                try:
                    flag_forms = generate_word_forms(base_lower, [flag], affixes)
                except Exception:
                    continue
                derived = flag_forms - {base_lower}
                if not derived:
                    continue
                hits = sum(1 for f in derived if f in _corpus_keys)
                if hits > 0 and hits / len(derived) >= quorum:
                    validated.append(flag)

        if true_case_hits or ghost_true_case_hits or validated:
            results_batch.append({
                'base_word'       : base_word,
                'found_forms'     : sorted(true_case_hits, key=str.lower),
                'count'           : len(true_case_hits),
                'ghost_forms'     : sorted(ghost_true_case_hits, key=str.lower),
                'ghost_count'     : len(ghost_true_case_hits),
                'validated_flags' : validated,
            })
    return results_batch


def find_corpus_wordforms(
    dic_path   : str,
    lang       : str,
    games      : List[str],
    source_type: str  = 'i18n',
    sample     : int  = 800,
    workers    : int  = 8,
    batch_size : int  = 50,
    propernoun_sidecar: str | None = None,
    add_verb_flags: bool = False,
    quorum: float = 0.5,
) -> List[dict]:
    """
    For each token in the custom .dic, generate every Hunspell word form via
    brute-force application of ALL SFX+PFX flags, then check which generated
    forms appear in the i18n corpus.

    For tokens that belong to any category in PROPER_NOUN_KEY_PATTERNS (loaded
    from a sidecar JSON produced by load_and_tokenize_terminology_base), gender "ghost"
    forms are also generated and checked, even when the word's ending is
    incompatible with standard AFF conditions (e.g. bwork → bworka).

    Flag attribution (munch pre-computation)
    -----------------------------------------
    When MUNCH_FLAG_CONFIG defines validation/verb flags for *lang*, each
    candidate flag is tested per word inside the worker threads.  A flag is
    "validated" when >= *quorum* fraction of its generated forms appear in
    the corpus.  Verb flags are only tested when *add_verb_flags=True*.
    The result dict includes ``validated_flags`` per word so that the
    downstream ``munch_to_compressed_dic()`` no longer needs to repeat the
    generate + corpus-lookup work.

    True-case
    ---------
    Corpus tokens are stored with their original case; the CSV reports the
    exact form(s) as they appear in the i18n file.  If both 'Abella' and
    'abella' appear, both are listed.

    New-form filtering
    ------------------
    After collecting all corpus hits, each form is checked against:
      1. The custom .dic (dic_path)  -- ALL entries, not just the sample
      2. The standard Hunspell .dic  -- HUNSPELL_PATHS[lang]
    Forms already present in either dic are flagged as "known".

    CSV columns
    -----------
    base_word | new_found_forms | new_count | found_forms | count |
    ghost_forms | ghost_count | new_ghost_forms | new_ghost_count |
    validated_flags

    Args:
        dic_path   : Path to the custom filtered_tokens.dic file.
        lang       : 2-letter language code ('es', 'pt', 'en', 'de').
        games      : Games to source the i18n corpus from (e.g. ['WAKFU']).
        source_type: 'i18n' (only mode currently implemented).
        sample     : Maximum dic words to process (0 = all words).
        workers    : ThreadPoolExecutor max_workers.
        batch_size : Words fed to each thread per batch.
        propernoun_sidecar: Path to the JSON sidecar. If None, auto-resolves
            from {INTERMEDIARY_DIR}/{games[0]}_{lang}_propernoun_tokens.json.
        add_verb_flags: If True, include verb conjugation flags as candidates.
        quorum    : Minimum fraction of flag-generated forms that must be
            corpus-confirmed for the flag to be assigned (default 0.5).

    Returns:
        List of result dicts sorted by base_word.
    """
    t0 = _time_now()

    # ── Resolve AFF file ──────────────────────────────────────────────────────
    aff_template = HUNSPELL_PATHS.get(lang, "")
    if not aff_template:
        raise ValueError(f"No HUNSPELL_PATHS entry for lang={lang!r}")
    aff_path = aff_template.replace(".dic", ".aff")
    if not os.path.exists(aff_path):
        raise FileNotFoundError(f"AFF file not found: {aff_path!r}")
    # ── Parse AFF ─────────────────────────────────────────────────────────────
    std_dic_path = HUNSPELL_PATHS[lang]
    affixes   = parse_aff_file(aff_path)
    all_flags = sorted(set(affixes.get('SFX', {}).keys()) | set(affixes.get('PFX', {}).keys()))

    print("─" * 62)
    # ── Build candidate flags for munch pre-computation ───────────────────────
    flag_cfg = MUNCH_FLAG_CONFIG.get(lang, {"mandatory": [], "validation": [], "verb": []})
    _validation_flags = set(flag_cfg['validation'])
    _verb_flags       = set(flag_cfg['verb']) if add_verb_flags else set()
    _candidate_flags  = _validation_flags | _verb_flags
    _available_flags  = set(affixes['SFX'].keys()) | set(affixes['PFX'].keys())
    _candidate_flags &= _available_flags
    _candidate_flags_list = sorted(_candidate_flags) if _candidate_flags else None
    if _candidate_flags_list:
        print(f"  Candidate flags for munch: {_candidate_flags_list}  (quorum={quorum})")
        if _verb_flags:
            print(f"  Verb flags included      : {sorted(_verb_flags & _available_flags)}")
    else:
        print(f"  No candidate flags configured for lang={lang} — flag attribution skipped")

    # ── Load proper-noun sidecar ──────────────────────────────────────────────
    _propernoun_lower: frozenset = frozenset()
    if propernoun_sidecar is None:
        propernoun_sidecar = os.path.join(
            INTERMEDIARY_DIR, f"{games[0]}_{lang}_propernoun_tokens.json"
        )
    if os.path.exists(propernoun_sidecar):
        with open(propernoun_sidecar, 'r', encoding='utf-8') as _fh:
            _pn_data = json.load(_fh)
        # Merge all key_types from the sidecar
        # (every category present was already filtered by PROPER_NOUN_KEY_PATTERNS at tokenize time)
        _pn_tokens: Set[str] = set()
        for tok_list in _pn_data.values():
            _pn_tokens.update(tok_list)
        _propernoun_lower = frozenset(_pn_tokens)
        print(f"  [ghost] Proper-noun tokens loaded: {len(_propernoun_lower):,}"
              f"  (from {propernoun_sidecar})")
        _ghost_cfg = GENDER_GHOST_SUFFIXES.get(lang, {})
        if _ghost_cfg and _ghost_cfg.get("suffixes"):
            print(f"  [ghost] Suffixes ({lang}): {_ghost_cfg['suffixes']}")
        else:
            print(f"  [ghost] No gender ghost suffixes defined for lang={lang}")
    else:
        print(f"  [ghost] No propernoun sidecar found at {propernoun_sidecar}")

    # ── Load .dic words ───────────────────────────────────────────────────────
    with open(dic_path, 'r', encoding='utf-8') as fh:
        raw_lines = fh.readlines()
    dic_words_all = [ln.strip().split('/')[0] for ln in raw_lines[1:] if ln.strip()]
    if sample and sample < len(dic_words_all):
        dic_words = dic_words_all[:sample]
        print(f"  Dic words (sample) : {len(dic_words):,} of {len(dic_words_all):,} total")
    else:
        dic_words = dic_words_all
        print(f"  Dic words (full)   : {len(dic_words):,}")
    print("\nLoading i18n corpus...")
    # ── Build known-word sets (for new_found_forms filtering) ─────────────────
    print("\nBuilding known-word sets for filtering...")
    # Custom dic: ALL entries (not just the sample), lowercased
    custom_dic_lowers: Set[str] = {w.lower() for w in dic_words_all}
    print(f"  Custom dic entries (lowercase) : {len(custom_dic_lowers):,}")

    # Standard Hunspell dic -- full AFF expansion (all inflected forms, cached per session)
    std_dic_lowers = build_std_dic_forms(lang)

    corpus_map = load_i18n_corpus(lang, games, source_type)

    all_results: List[dict] = []
    if not corpus_map:
        print("\nNo i18n corpus tokens found for this pair — skipping wordform worker dispatch.")
    else:
        # ── Dispatch threads ──────────────────────────────────────────────────
        print(f"\nDispatching {len(dic_words):,} words across {workers} workers ...")
        batches = [dic_words[i:i + batch_size] for i in range(0, len(dic_words), batch_size)]
        done_count = 0
        with ThreadPoolExecutor(max_workers=workers) as pool:
            future_to_idx = {
                pool.submit(_wordform_match_worker, b, affixes, corpus_map,
                            all_flags, _propernoun_lower, lang,
                            _candidate_flags_list, quorum): i
                for i, b in enumerate(batches)
            }
            for fut in as_completed(future_to_idx):
                batch_res = fut.result()
                all_results.extend(batch_res)
                done_count += 1
                if done_count % 5 == 0 or done_count == len(batches):
                    print(
                        f"  {done_count:>4}/{len(batches)} batches done   ({len(all_results)} words with hits so far)",
                        end='\r',
                        flush=True,
                    )

    print()

    # ── Enrich results with new_found_forms and new_ghost_forms ─────────────
    for r in all_results:
        new_forms = [
            f for f in r['found_forms']
            if f.lower() not in custom_dic_lowers
            and f.lower() not in std_dic_lowers
        ]
        r['new_found_forms'] = new_forms
        r['new_count']       = len(new_forms)

        # Ghost forms enrichment
        new_ghosts = [
            f for f in r.get('ghost_forms', [])
            if f.lower() not in custom_dic_lowers
            and f.lower() not in std_dic_lowers
        ]
        r['new_ghost_forms'] = new_ghosts
        r['new_ghost_count'] = len(new_ghosts)

    # ── Sort + summarise ──────────────────────────────────────────────────────
    all_results.sort(key=lambda r: r['base_word'].lower())
    words_with_hits     = len(all_results)
    words_with_new      = sum(1 for r in all_results if r['new_count'] > 0)
    total_form_hits     = sum(r['count']     for r in all_results)
    total_new_form_hits = sum(r['new_count'] for r in all_results)
    words_with_ghost    = sum(1 for r in all_results if r.get('ghost_count', 0) > 0)
    total_ghost_hits    = sum(r.get('ghost_count', 0)     for r in all_results)
    total_new_ghosts    = sum(r.get('new_ghost_count', 0) for r in all_results)
    words_with_flags    = sum(1 for r in all_results if r.get('validated_flags'))
    total_flags_assigned = sum(len(r.get('validated_flags', [])) for r in all_results)
    elapsed = _time_now() - t0

    print(f"\n{'─'*62}")
    print(f"  Words processed           : {len(dic_words):,}")
    print(f"  Words with >=1 corpus hit : {words_with_hits:,}")
    print(f"  Total corpus form matches : {total_form_hits:,}")
    print(f"  Words with >=1 NEW form   : {words_with_new:,}")
    print(f"  Total NEW form matches    : {total_new_form_hits:,}  (not in custom dic nor std dic)")
    if _propernoun_lower:
        print(f"  Words with ghost hits     : {words_with_ghost:,}")
        print(f"  Total ghost corpus hits   : {total_ghost_hits:,}")
        print(f"  Total NEW ghost forms     : {total_new_ghosts:,}")
    if _candidate_flags_list:
        print(f"  Words with validated flags: {words_with_flags:,}")
        print(f"  Total flags assigned      : {total_flags_assigned:,}")
    print(f"  Elapsed                   : {elapsed:.1f}s")
    print(f"{'─'*62}")

    # ── Save CSV ──────────────────────────────────────────────────────────────
    games_tag  = "_".join(games)
    out_csv = os.path.join(INTERMEDIARY_DIR, f"{games_tag}_{lang}_corpus_wordforms.csv")
    fieldnames = [
        'base_word', 'new_found_forms', 'new_count',
        'found_forms', 'count',
        'ghost_forms', 'ghost_count',
        'new_ghost_forms', 'new_ghost_count',
        'validated_flags',
    ]
    with open(out_csv, 'w', newline='', encoding='utf-8-sig') as fh:
        writer = csv.DictWriter(fh, fieldnames=fieldnames)
        writer.writeheader()
        for r in all_results:
            writer.writerow({
                'base_word'       : r['base_word'],
                'new_found_forms' : ' | '.join(r['new_found_forms']),
                'new_count'       : r['new_count'],
                'found_forms'     : ' | '.join(r['found_forms']),
                'count'           : r['count'],
                'new_ghost_forms' : ' | '.join(r.get('new_ghost_forms', [])),
                'new_ghost_count' : r.get('new_ghost_count', 0),
                'ghost_forms'     : ' | '.join(r.get('ghost_forms', [])),
                'ghost_count'     : r.get('ghost_count', 0),
                'validated_flags' : ' | '.join(r.get('validated_flags', [])),
            })
    print(f"\nSaved -> {out_csv}  ({words_with_hits} rows)")

    # ── Top-20 preview by new_count ───────────────────────────────────────────
    top20 = sorted(all_results, key=lambda r: -r['new_count'])[:20]
    if top20:
        print("\nTop-20 by new_count (forms not yet in any dic):")
        for r in top20:
            if r['new_count'] == 0:
                break
            preview = " | ".join(r['new_found_forms'][:6])
            if r['new_count'] > 6:
                preview += f"  ... (+{r['new_count'] - 6} more)"
            print(f"  {r['base_word']:<30}  {r['new_count']:>3} new   ->  {preview}")

    # ── Top-20 preview by new_ghost_count ─────────────────────────────────────
    if _propernoun_lower:
        top20g = sorted(all_results, key=lambda r: -r.get('new_ghost_count', 0))[:20]
        if top20g and top20g[0].get('new_ghost_count', 0) > 0:
            print("\nTop-20 by new_ghost_count (gender ghosts not in any dic):")
            for r in top20g:
                gc = r.get('new_ghost_count', 0)
                if gc == 0:
                    break
                gpreview = " | ".join(r.get('new_ghost_forms', [])[:6])
                if gc > 6:
                    gpreview += f"  ... (+{gc - 6} more)"
                print(f"  {r['base_word']:<30}  {gc:>3} ghost ->  {gpreview}")

    return all_results


## 5. Munching & Export

In [6]:
# ==============================================================================
# 5.  MUNCH — Assemble flat + compressed dic from wordform results
# ==============================================================================
# Flag attribution (validation/verb) is done inside _wordform_match_worker()
# during find_corpus_wordforms().  This cell only does:
#   - mandatory flag assignment (no corpus check)
#   - casing inference + CSV export
#   - file I/O: flat .dic, compressed .dic + .aff


# ── Helper: extract raw AFF text blocks ──────────────────────────────────────

def _extract_raw_aff_blocks(aff_path: str) -> Dict:
    """
    Read a Hunspell .aff file and return:
      { 'header_lines': [str, ...],
        'SFX': {flag: [raw_lines]},
        'PFX': {flag: [raw_lines]},
        'encoding': str }
    Header lines are everything that is NOT a PFX/SFX line.
    """
    # Detect encoding
    _enc = 'utf-8'
    try:
        with open(aff_path, 'rb') as fb:
            for raw_line in fb:
                text = raw_line.decode('ascii', errors='ignore').strip()
                if text.upper().startswith('SET '):
                    _enc_name = text.split()[1].strip()
                    _enc = _enc_name.replace('ISO8859-', 'iso-8859-')
                    break
    except Exception:
        pass

    header_lines: List[str] = []
    sfx_blocks: Dict[str, List[str]] = {}
    pfx_blocks: Dict[str, List[str]] = {}

    with open(aff_path, 'r', encoding=_enc, errors='replace') as fh:
        for line in fh:
            stripped = line.rstrip('\n\r')
            parts = stripped.split()
            if len(parts) >= 2 and parts[0] in ('SFX', 'PFX'):
                rule_type = parts[0]
                flag = parts[1]
                bucket = sfx_blocks if rule_type == 'SFX' else pfx_blocks
                bucket.setdefault(flag, []).append(stripped)
            else:
                header_lines.append(stripped)

    return {
        'header_lines': header_lines,
        'SFX': sfx_blocks,
        'PFX': pfx_blocks,
        'encoding': _enc,
    }


def _normalize_aff_header_lines(header_lines: List[str]) -> List[str]:
    """
    Normalize AFF header directives used by exported game dictionaries:
      - WORDCHARS must include apostrophe, hyphen, and digits 0-9
      - TRY must include apostrophe and hyphen

    Existing directive content and order are preserved; only missing required
    characters are appended.
    """
    required_wordchars = "'-0123456789"
    required_try_chars = "-'"

    normalized: List[str] = []
    seen_wordchars = False
    seen_try = False

    for line in header_lines:
        stripped = line.strip()
        if not stripped:
            normalized.append(line)
            continue

        parts = stripped.split(None, 1)
        directive = parts[0].upper() if parts else ""

        if directive == 'WORDCHARS':
            seen_wordchars = True
            existing_chars = parts[1] if len(parts) > 1 else ""
            merged = existing_chars
            for ch in required_wordchars:
                if ch not in merged:
                    merged += ch
            normalized.append(f"WORDCHARS {merged}")
            continue

        if directive == 'TRY':
            seen_try = True
            existing_chars = parts[1] if len(parts) > 1 else ""
            merged = existing_chars
            for ch in required_try_chars:
                if ch not in merged:
                    merged += ch
            normalized.append(f"TRY {merged}")
            continue

        normalized.append(line)

    if not seen_wordchars:
        normalized.append(f"WORDCHARS {required_wordchars}")

    if not seen_try:
        normalized.append(f"TRY {required_try_chars}")

    return normalized


def _build_custom_aff(std_aff_path: str, used_flags: Set[str], lang: str,
                      aff_blocks: Dict = None) -> str:
    """
    Build a custom .aff file string containing:
      - A normalized header from the standard .aff
      - Only the SFX/PFX blocks whose flag appears in `used_flags`.

    Header normalization keeps source directives intact while ensuring
    WORDCHARS and TRY include required punctuation for game terms.
    """
    if aff_blocks is None:
        aff_blocks = _extract_raw_aff_blocks(std_aff_path)

    parts: List[str] = []

    # ── Header ────────────────────────────────────────────────────────────────
    for hl in _normalize_aff_header_lines(aff_blocks['header_lines']):
        parts.append(hl)

    # ── SFX blocks (in original order) ────────────────────────────────────────
    for flag, lines in aff_blocks['SFX'].items():
        if flag in used_flags:
            for ln in lines:
                parts.append(ln)

    # ── PFX blocks ────────────────────────────────────────────────────────────
    for flag, lines in aff_blocks['PFX'].items():
        if flag in used_flags:
            for ln in lines:
                parts.append(ln)

    return '\n'.join(parts) + '\n'


# ── Helper: casing inference ─────────────────────────────────────────────────

def _infer_casing_variants(
    base_words: Set[str],
    corpus_exact_lowers: Set[str],
    propernoun_tokens: Set[str],
    use_propernoun_tokens: bool = True,
) -> List[Dict]:
    """
    For each Capitalised base word, check if its lowercase version appears
    in ``corpus_exact_lowers`` (exact lowercase evidence) and optionally in
    ``propernoun_tokens``.  Returns a list of dicts
    ``[{ 'original': 'Bwork', 'inferred_lower': 'bwork', 'source': '...' }, ...]``
    for export to a review CSV.
    """
    inferences: List[Dict] = []
    propernoun_lower = {t.lower() for t in propernoun_tokens}

    for bw in sorted(base_words):
        if not bw or not bw[0].isupper():
            continue
        lw = bw.lower()
        if lw == bw:
            continue  # already all-lower (e.g. single char)
        if lw in base_words:
            continue  # lowercase version already in dic explicitly

        source_parts = []
        if lw in corpus_exact_lowers:
            source_parts.append('corpus_exact_lower')
        if use_propernoun_tokens and lw in propernoun_lower:
            source_parts.append('propernoun_sidecar')
        if source_parts:
            inferences.append({
                'original': bw,
                'inferred_lower': lw,
                'source': ' + '.join(source_parts),
            })

    return inferences


# ══════════════════════════════════════════════════════════════════════════════
# MAIN FUNCTION — Assembly-only (flag attribution done in find_corpus_wordforms)
# ══════════════════════════════════════════════════════════════════════════════

def munch_to_compressed_dic(
    dic_path: str,
    lang: str,
    games: List[str],
    wordform_results: List[dict],
    propernoun_sidecar: str | None = None,
) -> Dict:
    """
    Assemble filtered_tokens.dic + corpus-confirmed wordform results into:
      1. A flat consolidated .dic  (all words, one per line)
      2. A compressed .dic + .aff  (base words with affix flags)
      3. A casing-inference CSV     (capitalised → lowercase review)

    Flag attribution (validation/verb flags) has already been computed by
    ``find_corpus_wordforms()`` → ``_wordform_match_worker()`` and is
    available in ``wordform_results[i]['validated_flags']``.  This function
    only adds **mandatory** flags (from ``MUNCH_FLAG_CONFIG``), prunes
    redundant explicit forms covered by flagged entries, and writes outputs.

    Args:
        dic_path:           Path to filtered_tokens.dic (base words)
        lang:               Language code (es, de, en, pt, fr)
        games:              List of game tags (for file naming)
        wordform_results:   The list returned by find_corpus_wordforms()
        propernoun_sidecar: Path to propernoun sidecar JSON (for casing context)

    Returns:
        Dict with keys: flat_dic_path, compressed_dic_path, compressed_aff_path,
                        casing_csv_path, stats
    """
    import csv as _csv
    import json as _json
    import shutil as _shutil
    t0 = _time_now()
    games_tag = "_".join(games)
    game_lang_key = f"{games_tag}_{lang}"
    lang_game_key = f"{lang}_{games_tag}"

    # ── 1. Load base dic words ────────────────────────────────────────────────
    with open(dic_path, 'r', encoding='utf-8-sig') as fh:
        lines = fh.read().splitlines()
    base_words_raw: List[str] = []
    for ln in lines[1:]:
        w = ln.strip().split('/')[0].strip()
        if w:
            base_words_raw.append(w)
    base_words_set = set(base_words_raw)
    print(f"  Loaded {len(base_words_set):,} base words from {os.path.basename(dic_path)}")

    # ── 2. Build confirmed-forms set from wordform_results ────────────────────
    # Also build a per-word validated_flags map
    family_map: Dict[str, Set[str]] = {}
    validated_flags_map: Dict[str, List[str]] = {}
    corpus_exact_lowers: Set[str] = set()
    for r in wordform_results:
        bw = r['base_word']
        forms: Set[str] = set()
        for col in ('found_forms', 'new_found_forms',
                     'ghost_forms', 'new_ghost_forms'):
            for f in r.get(col, []):
                if isinstance(f, str) and f.strip():
                    clean_f = f.strip()
                    forms.add(clean_f)
                    if clean_f == clean_f.lower():
                        corpus_exact_lowers.add(clean_f)
        if forms:
            family_map.setdefault(bw, set()).update(forms)
        vf = r.get('validated_flags', [])
        if vf:
            validated_flags_map[bw] = list(vf)
    print(f"  Wordform results : {len(wordform_results):,} words, "
          f"{len(validated_flags_map):,} with validated flags")
    print(f"  Exact lowercase corpus forms: {len(corpus_exact_lowers):,}")

    # ── 3. Load propernoun sidecar (for casing context) ───────────────────────
    propernoun_tokens: Set[str] = set()
    if propernoun_sidecar and os.path.isfile(propernoun_sidecar):
        with open(propernoun_sidecar, 'r', encoding='utf-8') as fh:
            sidecar_data = _json.load(fh)
        for key_type, tokens in sidecar_data.items():
            propernoun_tokens.update(tokens)
        print(f"  Loaded {len(propernoun_tokens):,} propernoun tokens from sidecar")

    # ── 4. Build the flat confirmed-forms set ─────────────────────────────────
    all_confirmed: Set[str] = set(base_words_set)
    for forms in family_map.values():
        all_confirmed.update(forms)
    print(f"  Total confirmed forms (base + inflected): {len(all_confirmed):,}")

    # ── 5. Casing inference ───────────────────────────────────────────────────
    _use_sidecar_for_casing = False
    casing_inferences = _infer_casing_variants(
        base_words_set, corpus_exact_lowers, propernoun_tokens,
        use_propernoun_tokens=_use_sidecar_for_casing
    )
    for inf in casing_inferences:
        all_confirmed.add(inf['inferred_lower'])
        base_words_set.add(inf['inferred_lower'])
    print(f"  Casing inferences: {len(casing_inferences):,} "
          f"(Capitalised → lowercase)")

    # Re-apply ANKANIMATION ES custom overrides at assembly time so corpus-found
    # capitalized variants do not leak back into Flatlists exports.
    if lang == 'es' and any(str(g).strip().upper() == 'ANKANIMATION' for g in games):
        _before_base = len(base_words_set)
        _before_all = len(all_confirmed)
        base_words_set = _apply_ankanimation_token_overrides(base_words_set, 'es')
        all_confirmed = _apply_ankanimation_token_overrides(all_confirmed, 'es')
        all_confirmed.update(base_words_set)
        print(
            f"  [ankassembly-overrides] base={_before_base:,}->{len(base_words_set):,} "
            f"all={_before_all:,}->{len(all_confirmed):,}"
        )

    # ── 6. Parse standard AFF (for file structure only) ──────────────────────
    std_dic_path = HUNSPELL_PATHS.get(lang)
    if not std_dic_path:
        print(f"  ⚠ No HUNSPELL_PATHS entry for '{lang}' — skipping munch")
        return {}
    std_aff_path = std_dic_path.rsplit('.', 1)[0] + '.aff'
    if not os.path.isfile(std_aff_path):
        print(f"  ⚠ AFF file not found: {std_aff_path}")
        return {}
    affixes = parse_aff_file(std_aff_path)
    aff_blocks = _extract_raw_aff_blocks(std_aff_path)
    print(f"  Parsed {std_aff_path}  (for flag_mode + AFF block extraction)")

    # ── 7. Assemble per-word flags: mandatory + validated ─────────────────────
    flag_cfg = MUNCH_FLAG_CONFIG.get(lang, {"mandatory": [], "validation": [], "verb": []})
    available_flags = set(affixes['SFX'].keys()) | set(affixes['PFX'].keys())
    mandatory_flags = set(flag_cfg['mandatory']) & available_flags

    compressed_entries: Dict[str, Set[str]] = {}
    flag_usage_count: Dict[str, int] = {}

    for bw in sorted(base_words_set):
        assigned: Set[str] = set(mandatory_flags)

        # Add validated flags from corpus results (already quorum-checked)
        # Check both the exact key and case variants
        for key_variant in (bw, bw.lower(), bw[0].upper() + bw[1:] if bw else ''):
            for fl in validated_flags_map.get(key_variant, []):
                assigned.add(fl)

        if assigned:
            compressed_entries[bw] = assigned
            for fl in assigned:
                flag_usage_count[fl] = flag_usage_count.get(fl, 0) + 1

    used_flags_all = set()
    for flags in compressed_entries.values():
        used_flags_all.update(flags)

    _total_words = len(base_words_set)
    print(f"\n  Munch complete (assembly-only — no re-generation):")
    print(f"    Mandatory flags : {sorted(mandatory_flags) or '(none)'}")
    print(f"    Words with flags: {sum(1 for f in compressed_entries.values() if f):,} "
          f"/ {_total_words:,}")
    print(f"    Flags used      : {sorted(used_flags_all)}")
    for fl in sorted(used_flags_all):
        print(f"      {fl!r:>6}  →  {flag_usage_count.get(fl, 0):>6,} words")

    # ── 8. Output files ──────────────────────────────────────────────────────
    os.makedirs(OUTPUT_FLATLISTS_DIR, exist_ok=True)
    os.makedirs(OUTPUT_COMPRESSED_DIR, exist_ok=True)
    os.makedirs(OUTPUT_FULL_DIR, exist_ok=True)
    os.makedirs(INTERMEDIARY_DIR, exist_ok=True)

    compressed_bundle_dir = os.path.join(OUTPUT_COMPRESSED_DIR, lang_game_key)
    full_bundle_dir = os.path.join(OUTPUT_FULL_DIR, lang_game_key)
    os.makedirs(compressed_bundle_dir, exist_ok=True)
    os.makedirs(full_bundle_dir, exist_ok=True)

    # 8a. Flat consolidated .DIC (Word-compatible by default)
    flat_words = sorted(all_confirmed, key=str.lower)
    flat_dic_path = os.path.join(OUTPUT_FLATLISTS_DIR, f"{game_lang_key}.DIC")
    _flat_lines = [str(w).strip() for w in flat_words if str(w).strip()]
    _flat_text = "\r\n".join(_flat_lines).rstrip("\r\n") + "\r\n"
    with open(flat_dic_path, 'wb') as fh:
        fh.write(b"\xff\xfe")
        fh.write(_flat_text.encode('utf-16-le'))
    print(f"\n  Flat dic     → {flat_dic_path}  ({len(flat_words):,} words)")

    # 8b. Compressed .dic
    compressed_dic_path = os.path.join(compressed_bundle_dir,
                                       f"{lang_game_key}.dic")
    flag_mode = affixes.get('flag_mode', 'single')

    def _flags_to_str(flags: Set[str]) -> str:
        if not flags:
            return ''
        if flag_mode == 'long':
            return '/' + ''.join(sorted(flags))
        elif flag_mode == 'num':
            return '/' + ','.join(sorted(flags))
        else:  # 'single' or 'utf8'
            return '/' + ''.join(sorted(flags))

    # Keep compressed/full aligned with flat-confirmed forms (base + corpus forms).
    compressed_output_words: Set[str] = set(all_confirmed)

    # Prune explicit no-flag entries that are already generated by another
    # flagged entry (e.g., base/GS already covers explicit plural/feminine forms).
    generated_by_flagged: Dict[str, Set[str]] = {}
    _prune_errors: List[str] = []
    for src_word in sorted(compressed_output_words, key=str.lower):
        src_flags = compressed_entries.get(src_word, set())
        if not src_flags:
            continue
        try:
            generated_forms = generate_word_forms(src_word, sorted(src_flags), affixes)
        except Exception as err:
            _prune_errors.append(f"{src_word}: {type(err).__name__}: {err}")
            continue
        for form in generated_forms:
            form_txt = str(form).strip()
            if not form_txt or form_txt == src_word:
                continue
            if form_txt in compressed_output_words:
                generated_by_flagged.setdefault(form_txt, set()).add(src_word)

    dropped_redundant_words: Set[str] = set()
    for cand_word in generated_by_flagged:
        if not compressed_entries.get(cand_word, set()):
            dropped_redundant_words.add(cand_word)
    if dropped_redundant_words:
        compressed_output_words -= dropped_redundant_words
        print(f"  Pruned redundant explicit forms: {len(dropped_redundant_words):,}")
    if _prune_errors:
        print(f"  ⚠ Prune generation errors: {len(_prune_errors):,}")
    compressed_lines: List[str] = []
    for bw in sorted(compressed_output_words, key=str.lower):
        flags = compressed_entries.get(bw, set())
        compressed_lines.append(bw + _flags_to_str(flags))

    with open(compressed_dic_path, 'w', encoding='utf-8') as fh:
        fh.write(f"{len(compressed_lines)}\n")
        for ln in compressed_lines:
            fh.write(ln + '\n')
    print(f"  Compressed dic → {compressed_dic_path}  "
          f"({len(compressed_lines):,} entries)")

    # 8c. Compressed .aff (only used flag blocks)
    compressed_aff_path = os.path.join(compressed_bundle_dir,
                                       f"{lang_game_key}.aff")
    custom_aff_text = _build_custom_aff(std_aff_path, used_flags_all, lang,
                                         aff_blocks)
    with open(compressed_aff_path, 'w',
              encoding=aff_blocks['encoding']) as fh:
        fh.write(custom_aff_text)
    print(f"  Compressed aff → {compressed_aff_path}")

    # 8d. Full dictionary bundle: copy Hunspell reference folder + add generated pair
    def _copy_tree_no_overwrite(src_dir: str, dst_dir: str) -> int:
        copied = 0
        for root, _, files in os.walk(src_dir):
            rel_root = os.path.relpath(root, src_dir)
            target_root = dst_dir if rel_root == '.' else os.path.join(dst_dir, rel_root)
            os.makedirs(target_root, exist_ok=True)
            for name in files:
                src_file = os.path.join(root, name)
                dst_file = os.path.join(target_root, name)
                if os.path.exists(dst_file):
                    continue
                _shutil.copy2(src_file, dst_file)
                copied += 1
        return copied

    # Copy only the direct Hunspell language folder (e.g. .../es),
    # not every sibling language variant under *_dic.
    ref_src_dir = os.path.dirname(std_dic_path)
    legacy_ref_dir = os.path.join(full_bundle_dir, 'hunspell_ref')
    if os.path.isdir(legacy_ref_dir):
        _shutil.rmtree(legacy_ref_dir)
        print(f"    Removed legacy subfolder: {legacy_ref_dir}")
    copied_ref_files = _copy_tree_no_overwrite(ref_src_dir, full_bundle_dir)
    full_generated_dic_path = os.path.join(full_bundle_dir, f"{lang_game_key}.dic")
    full_generated_aff_path = os.path.join(full_bundle_dir, f"{lang_game_key}.aff")
    _shutil.copy2(compressed_dic_path, full_generated_dic_path)
    _shutil.copy2(compressed_aff_path, full_generated_aff_path)
    print(f"  Full bundle    → {full_bundle_dir}")
    print(f"    Hunspell ref copied (new files only): {copied_ref_files}")
    print(f"    Added generated dic: {full_generated_dic_path}")
    print(f"    Added generated aff: {full_generated_aff_path}")

    # 8e. Casing-inference CSV
    casing_csv_path = os.path.join(INTERMEDIARY_DIR,
                                    f"{games_tag}_{lang}_casing_inference.csv")
    with open(casing_csv_path, 'w', newline='', encoding='utf-8-sig') as fh:
        writer = _csv.DictWriter(fh,
                                fieldnames=['original', 'inferred_lower', 'source'])
        writer.writeheader()
        writer.writerows(casing_inferences)
    print(f"  Casing CSV   → {casing_csv_path}  "
          f"({len(casing_inferences):,} rows)")

    elapsed = _time_now() - t0
    print(f"\n  ⏱ Munch elapsed: {elapsed:.1f}s")

    return {
        'game_lang_key':      game_lang_key,
        'lang_game_key':      lang_game_key,
        'flat_dic_path':       flat_dic_path,
        'compressed_dic_path': compressed_dic_path,
        'compressed_aff_path': compressed_aff_path,
        'full_dic_dir':        full_bundle_dir,
        'full_dic_path':       full_generated_dic_path,
        'full_aff_path':       full_generated_aff_path,
        'casing_csv_path':     casing_csv_path,
        'stats': {
            'base_words':        len(base_words_set),
            'flat_words':        len(flat_words),
            'compressed_entries': len(compressed_lines),
            'compressed_pruned_redundant': len(dropped_redundant_words),
            'flags_used':        sorted(used_flags_all),
            'casing_inferences': len(casing_inferences),
            'elapsed':           elapsed,
        },
    }

# X. Batch processing

In [7]:
# Batch orchestrator for Steps 1-5 (tokenize/filter -> wordforms -> munch)
import os
import glob
import time
from typing import Any, Dict, List, Union


# Map language codes to available tokenizer modes used by load_and_tokenize_terminology_base().
_TOKENIZE_LANGUAGE_BY_CODE = {
    "en": "english",
    "pt": "portuguese",
    "pt-br": "portuguese",
    "pt-pt": "portuguese",
    "es": "default",
    "fr": "french",
    "fr-fr": "french",
    "de": "default",
    "it": "default",
}

# Session-level prewarm guard: (game, lang) pairs already warmed in this kernel.
_PIPELINE_PREWARM_DONE = set()


def _resolve_tokenize_language_mode(lang_code: str) -> str:
    """Return tokenizer mode for a language code, defaulting to 'default'."""
    code = str(lang_code or "").strip().lower()
    return _TOKENIZE_LANGUAGE_BY_CODE.get(code, _TOKENIZE_LANGUAGE_BY_CODE.get(normalize_language_code(code), "default"))


def _build_pair_paths(game: str, lang: str, work_dir: str) -> Dict[str, str]:
    """Build canonical intermediary file paths for one game/lang pair."""
    return {
        "token_txt": os.path.join(work_dir, f"{game}_{lang}_tokens.txt"),
        "filtered_dic": os.path.join(work_dir, f"{game}_{lang}_filtered_tokens.dic"),
        "propernoun_sidecar": os.path.join(work_dir, f"{game}_{lang}_propernoun_tokens.json"),
    }


def _cleanup_pair_outputs(game: str, lang: str, work_dir: str) -> List[str]:
    """Remove stale intermediary files for deterministic reruns."""
    removed: List[str] = []
    patterns = [
        os.path.join(work_dir, f"{game}_{lang}_filtered_tokens*.dic"),
        os.path.join(work_dir, f"{game}_{lang}_tokens*.txt"),
    ]
    for pattern in patterns:
        for path in glob.glob(pattern):
            try:
                os.remove(path)
                removed.append(path)
            except OSError:
                pass
    return removed


def run_pipeline_batch(
    languages: Union[List[str], str],  # list[str] ou 'all'
    games: List[str],                  # list[str] des jeux a traiter (cles TB_PATHS)
    sample: int = 0,                   # 0 = full run, >0 = echantillon de mots
    workers: int = 8,                  # nb de workers pour Step 4
    batch_size: int = 50,              # taille des lots envoyes aux workers
    add_verb_flags: bool = False,      # True = inclure flags verbaux candidats
    quorum: float = 0.5,               # seuil de validation des flags (0.0-1.0)
    cleanup_stale: bool = True,        # supprime anciens outputs intermediaires
    strict_mode: bool = False,         # stop au 1er echec si True
    output_folder: str | None = None,  # dossier de travail (None => INTERMEDIARY_DIR)
    prewarm_i18n: bool = True,         # precharge i18n filtre seulement si pas deja warm
) -> Dict[str, Any]:
    """
    Run the full Steps 1-5 pipeline for each (game, language) pair.

    Steps reused from TESTS:
      1-3) Tokenization and dictionary filtering
      4)   find_corpus_wordforms()
      5)   munch_to_compressed_dic()

    Required core params:
      - languages: list[str] or 'all'
      - games: list[str]

    Returns:
      Dict with keys:
        - runs: list of per-pair structured results
        - summary: aggregate totals and elapsed time
    """
    t_start = time.time()

    if isinstance(languages, str):
        _lang_token = languages.strip().lower()
        if _lang_token == 'all':
            languages_to_run = sorted({code.lower() for code in LANG_CODES.values() if code})
        elif _lang_token:
            languages_to_run = [_lang_token]
        else:
            languages_to_run = []
    else:
        languages_to_run = [str(l).strip() for l in languages if str(l).strip()]

    if not languages_to_run:
        raise ValueError("languages must contain at least one language code (or 'all')")
    if not games:
        raise ValueError("games must contain at least one game tag")

    work_dir = output_folder or INTERMEDIARY_DIR
    os.makedirs(work_dir, exist_ok=True)

    runs: List[Dict[str, Any]] = []

    print("=" * 80)
    print("BATCH PIPELINE (STEPS 1-5)")
    print("=" * 80)
    print(f"Games     : {games}")
    print(f"Languages : {languages_to_run}")
    print(f"Work dir  : {work_dir}")
    print(
        f"Options   : sample={sample}, workers={workers}, batch_size={batch_size}, "
        f"add_verb_flags={add_verb_flags}, quorum={quorum}, "
        f"cleanup_stale={cleanup_stale}, strict_mode={strict_mode}"
    )
    print("=" * 80)

    # Preflight dictionary resolution once for all requested languages.
    preflight_dicts: Dict[str, Dict[str, Any]] = {}
    for _lang_input in languages_to_run:
        try:
            _lang_norm = normalize_language_code(_lang_input)
            preflight_dicts[_lang_norm] = resolve_hunspell_paths(_lang_norm, dic_folder=DIC_FOLDER)
        except Exception as preflight_err:
            preflight_dicts[str(_lang_input).strip().lower()] = {
                "language": str(_lang_input),
                "ok": False,
                "dic": "",
                "aff": "",
                "checks": [],
                "error": f"{type(preflight_err).__name__}: {preflight_err}"
            }

    print("Dictionary preflight:")
    for _lang_key, _info in preflight_dicts.items():
        if _info.get("ok"):
            print(f"  ✅ {_lang_key}: {_info['dic']}")
        else:
            print(f"  ❌ {_lang_key}: no valid dic+aff pair")
            if _info.get("error"):
                print(f"     {_info['error']}")
            for _chk in _info.get("checks", []):
                print(
                    f"     - DIC {'OK' if _chk['dic_exists'] else 'MISSING'} | "
                    f"AFF {'OK' if _chk['aff_exists'] else 'MISSING'} :: {_chk['dic']}"
                )

    for game in games:
        for lang_input in languages_to_run:
            run_t0 = time.time()
            run_result: Dict[str, Any] = {
                "game": game,
                "language_input": lang_input,
                "language": None,
                "status": "ok",
                "error": "",
                "timings": {},
                "artifacts": {},
                "metrics": {},
            }

            print("\n" + "-" * 80)
            print(f"▶ Processing pair: game={game} | language={lang_input}")
            print("-" * 80)

            try:
                if game not in TB_PATHS:
                    raise KeyError(f"Game '{game}' is not configured in TB_PATHS")

                lang = normalize_language_code(lang_input)
                run_result["language"] = lang

                dict_status = preflight_dicts.get(lang, {"ok": False, "checks": []})
                run_result["metrics"]["dictionary_preflight"] = dict_status
                if not dict_status.get("ok"):
                    checks = dict_status.get("checks", [])
                    checked_paths = [c.get("dic", "") for c in checks]
                    raise FileNotFoundError(
                        f"No valid Hunspell dic+aff pair for '{lang}'. Checked: {checked_paths}"
                    )

                paths = _build_pair_paths(game=game, lang=lang, work_dir=work_dir)
                tokenize_language = _resolve_tokenize_language_mode(lang_input)
                tb_path = TB_PATHS[game]

                if cleanup_stale:
                    removed = _cleanup_pair_outputs(game=game, lang=lang, work_dir=work_dir)
                    if removed:
                        print(f"Removed stale outputs: {len(removed)}")
                    run_result["metrics"]["stale_removed"] = len(removed)

                # Optional i18n prewarm so ES-first-lower heuristics have local evidence.
                if prewarm_i18n and "load_i18n_corpus" in globals():
                    pair_key = (game, lang)
                    game_i18n_entry = i18n_PATHS.get(game, {})
                    if isinstance(game_i18n_entry, dict):
                        current_i18n = game_i18n_entry.get(lang, "")
                    else:
                        current_i18n = ""
                        run_result["metrics"]["i18n_status"] = "missing_config"
                        print(f"Prewarm skipped for {game}/{lang} [no i18n config]")

                    filtered_json = os.path.join(INTERMEDIARY_DIR, f"{game}_{lang}_i18n_filtered.json")
                    filtered_props = os.path.join(INTERMEDIARY_DIR, f"{game}_{lang}_i18n_filtered.properties")

                    # If a filtered i18n file already exists on disk, reuse it directly.
                    existing_filtered = ""
                    if os.path.exists(filtered_json):
                        existing_filtered = filtered_json
                    elif os.path.exists(filtered_props):
                        existing_filtered = filtered_props
                    if existing_filtered:
                        if isinstance(i18n_PATHS.get(game), dict):
                            i18n_PATHS[game][lang] = existing_filtered
                        current_i18n = existing_filtered

                    already_filtered = bool(current_i18n) and os.path.exists(current_i18n) and (
                        os.path.basename(current_i18n) == f"{game}_{lang}_i18n_filtered.json"
                        or os.path.basename(current_i18n) == f"{game}_{lang}_i18n_filtered.properties"
                    )
                    if pair_key in _PIPELINE_PREWARM_DONE or already_filtered:
                        if already_filtered:
                            _PIPELINE_PREWARM_DONE.add(pair_key)
                        run_result["metrics"]["i18n_status"] = "cached_filtered"
                        print(f"Prewarm skipped for {game}/{lang} [cached]")
                    elif not current_i18n:
                        if run_result["metrics"].get("i18n_status") != "missing_config":
                            run_result["metrics"]["i18n_status"] = "missing_file"
                            print(f"Prewarm skipped for {game}/{lang} [missing i18n file]")
                    else:
                        try:
                            print(f"Prewarming filtered i18n for {game}/{lang}...")
                            load_i18n_corpus(lang, [game], source_type="i18n", lang_detect=True)
                            _PIPELINE_PREWARM_DONE.add(pair_key)
                            run_result["metrics"]["i18n_status"] = "prewarmed"
                        except Exception as prewarm_err:
                            run_result["metrics"]["i18n_status"] = "prewarm_failed"
                            print(f"WARNING: Prewarm failed for {game}/{lang}: {prewarm_err}")
                elif not prewarm_i18n:
                    run_result["metrics"]["i18n_status"] = "prewarm_disabled"
                else:
                    run_result["metrics"]["i18n_status"] = "prewarm_loader_missing"

                # Steps 1-3: tokenize TB and dictionary-filter against Hunspell
                step_t0 = time.time()
                tokens_list, _ = load_and_tokenize_terminology_base(
                    excel_file_path=tb_path,
                    language_code=lang,
                    tokenize_language=tokenize_language,
                    output_file_path=paths["token_txt"],
                    save_propernoun_sidecar=True,
                    game_tag=game,
                    allow_language_fallback=False,
                )
                run_result["timings"]["step_1_tokenize"] = round(time.time() - step_t0, 3)
                run_result["metrics"]["token_count"] = len(tokens_list)

                step_t0 = time.time()
                batch_results = batch_filter_tokens_by_dictionary(
                    input_folder=work_dir,
                    target_languages=[lang],
                    dic_folder=DIC_FOLDER,
                    output_folder=work_dir,
                )
                run_result["timings"]["step_2_3_filter"] = round(time.time() - step_t0, 3)

                if not os.path.isfile(paths["filtered_dic"]):
                    raise FileNotFoundError(
                        f"Expected filtered dic not found after batch filtering: {paths['filtered_dic']}"
                    )

                run_result["artifacts"]["token_txt"] = paths["token_txt"]
                run_result["artifacts"]["filtered_dic"] = paths["filtered_dic"]
                run_result["artifacts"]["propernoun_sidecar"] = (
                    paths["propernoun_sidecar"] if os.path.isfile(paths["propernoun_sidecar"]) else None
                )
                run_result["metrics"]["batch_filter_results"] = batch_results

                # Step 4: find corpus wordforms from filtered .dic into i18n corpus
                step_t0 = time.time()
                wordform_results = find_corpus_wordforms(
                    dic_path=paths["filtered_dic"],
                    lang=lang,
                    games=[game],
                    source_type="i18n",
                    sample=sample,
                    workers=workers,
                    batch_size=batch_size,
                    propernoun_sidecar=run_result["artifacts"]["propernoun_sidecar"],
                    add_verb_flags=add_verb_flags,
                    quorum=quorum,
                )
                run_result["timings"]["step_4_wordforms"] = round(time.time() - step_t0, 3)
                run_result["metrics"]["wordform_rows"] = len(wordform_results)

                # Step 5: munch outputs (flat + compressed dic)
                step_t0 = time.time()
                munch_result = munch_to_compressed_dic(
                    dic_path=paths["filtered_dic"],
                    lang=lang,
                    games=[game],
                    wordform_results=wordform_results,
                    propernoun_sidecar=run_result["artifacts"]["propernoun_sidecar"],
                )
                run_result["timings"]["step_5_munch"] = round(time.time() - step_t0, 3)

                run_result["artifacts"].update({
                    "wordforms_csv": os.path.join(work_dir, f"{game}_{lang}_missing_wordforms.csv"),
                    "flat_dic": munch_result.get("flat_dic_path"),
                    "compressed_dic": munch_result.get("compressed_dic_path"),
                    "compressed_aff": munch_result.get("compressed_aff_path"),
                    "full_dic_dir": munch_result.get("full_dic_dir"),
                    "full_dic": munch_result.get("full_dic_path"),
                    "full_aff": munch_result.get("full_aff_path"),
                    "casing_csv": munch_result.get("casing_csv_path"),
                })
                run_result["metrics"]["munch_stats"] = munch_result.get("stats", {})

            except Exception as err:
                run_result["status"] = "failed"
                run_result["error"] = f"{type(err).__name__}: {err}"
                print(f"❌ Failed pair {game}/{lang_input}: {run_result['error']}")

                if strict_mode:
                    run_result["timings"]["total"] = round(time.time() - run_t0, 3)
                    runs.append(run_result)
                    total_elapsed = round(time.time() - t_start, 3)
                    return {
                        "runs": runs,
                        "summary": {
                            "status": "failed",
                            "reason": "strict_mode abort on first error",
                            "total_pairs": len(games) * len(languages_to_run),
                            "processed_pairs": len(runs),
                            "ok_pairs": sum(1 for r in runs if r["status"] == "ok"),
                            "failed_pairs": [
                                f"{r['game']}/{r['language_input']}"
                                for r in runs
                                if r["status"] == "failed"
                            ],
                            "failed_pairs_count": sum(1 for r in runs if r["status"] == "failed"),
                            "elapsed_seconds": total_elapsed,
                        },
                    }

            run_result["timings"]["total"] = round(time.time() - run_t0, 3)
            runs.append(run_result)

    total_elapsed = round(time.time() - t_start, 3)
    ok_pairs = sum(1 for r in runs if r["status"] == "ok")
    failed_pairs = [
        f"{r['game']}/{r['language_input']}"
        for r in runs
        if r["status"] == "failed"
    ]
    failed_pairs_count = len(failed_pairs)

    summary = {
        "status": "ok" if failed_pairs_count == 0 else "partial",
        "total_pairs": len(games) * len(languages_to_run),
        "processed_pairs": len(runs),
        "ok_pairs": ok_pairs,
        "failed_pairs": failed_pairs,
        "failed_pairs_count": failed_pairs_count,
        "elapsed_seconds": total_elapsed,
    }

    print("\n" + "=" * 80)
    print("BATCH PIPELINE COMPLETE")
    print(f"Status        : {summary['status']}")
    print(f"Processed     : {summary['processed_pairs']}/{summary['total_pairs']}")
    print(f"OK / Failed   : {summary['ok_pairs']} / {summary['failed_pairs_count']}")
    if summary['failed_pairs']:
        print(f"Failed pairs  : {summary['failed_pairs']}")
    print(f"Elapsed (sec) : {summary['elapsed_seconds']}")
    print("=" * 80)

    return {"runs": runs, "summary": summary}


def run_pipeline_single(
    language: str,
    game: str,
    **kwargs,
) -> Dict[str, Any]:
    """Convenience wrapper for one (game, language) pair."""
    result = run_pipeline_batch(languages=[language], games=[game], **kwargs)
    return result["runs"][0] if result.get("runs") else {}


# Example:
# batch_report = run_pipeline_batch(
#     languages=["es", "pt"],
#     games=["TOUCH", "DOFUS"],
#     sample=0,
#     workers=8,
#     batch_size=50,
#     strict_mode=False,
# )
# batch_report["summary"]

In [8]:
# ANK consolidation from existing exports only (no TB/i18n recomputation)
import os
import glob
import shutil
import codecs
from typing import Any, Dict, List, Set, Tuple, Union


def _write_flatlist_msword_unicode(flat_path: str, words: List[str]) -> str:
    """Write a Word-compatible flatlist (.DIC UTF-16 LE BOM, no count header)."""
    root, _ = os.path.splitext(flat_path)
    dst_path = root + ".DIC"
    os.makedirs(os.path.dirname(dst_path), exist_ok=True)

    lines = [str(w).strip() for w in words if str(w).strip()]
    out_text = "\r\n".join(lines).rstrip("\r\n") + "\r\n"

    with open(dst_path, "wb") as fh:
        fh.write(codecs.BOM_UTF16_LE)
        fh.write(out_text.encode("utf-16-le"))

    return dst_path


def _supported_lang_codes() -> Set[str]:
    """Return normalized language codes supported by this notebook config."""
    codes: Set[str] = set()
    for v in LANG_CODES.values():
        try:
            codes.add(normalize_language_code(v))
        except Exception:
            pass
    for k in HUNSPELL_PATHS.keys():
        try:
            codes.add(normalize_language_code(k))
        except Exception:
            pass
    return codes


def _parse_lang_game_token(token: str) -> Tuple[str, str] | None:
    """Parse names like 'es_DOFUS' or 'DOFUS_es' into (lang, game)."""
    base = str(token or "").strip()
    if not base:
        return None

    stem = os.path.splitext(base)[0]
    if "_" not in stem:
        return None

    parts = [p for p in stem.split("_") if p]
    if len(parts) < 2:
        return None

    known_langs = _supported_lang_codes()
    known_games = {str(g).upper() for g in GAMES}
    known_games.add("ANK")

    # Orientation 1: lang_game
    try:
        lang_first = normalize_language_code(parts[0])
    except Exception:
        lang_first = ""
    game_rest_1 = "_".join(parts[1:]).upper()
    if lang_first in known_langs and game_rest_1 in known_games:
        return lang_first, game_rest_1

    # Orientation 2: game_lang
    try:
        lang_last = normalize_language_code(parts[-1])
    except Exception:
        lang_last = ""
    game_rest_2 = "_".join(parts[:-1]).upper()
    if lang_last in known_langs and game_rest_2 in known_games:
        return lang_last, game_rest_2

    return None


def _discover_available_export_pairs(output_dir: str = OUTPUT_DIR) -> Dict[str, Any]:
    """
    Discover available (lang, game) pairs from existing exports.
    Source of truth: OUTPUT_DIR/Compressed_dics, with Flatlists as fallback signal.
    """
    compressed_root = os.path.join(output_dir, "Compressed_dics")
    flat_root = os.path.join(output_dir, "Flatlists")

    discovered_pairs: Set[Tuple[str, str]] = set()
    compressed_pairs: Set[Tuple[str, str]] = set()
    flat_pairs: Set[Tuple[str, str]] = set()

    # 1) Discover from compressed bundles (preferred)
    if os.path.isdir(compressed_root):
        for entry in os.listdir(compressed_root):
            full_dir = os.path.join(compressed_root, entry)
            if not os.path.isdir(full_dir):
                continue

            parsed = _parse_lang_game_token(entry)
            if not parsed:
                continue
            lang, game = parsed

            # Accept either file orientation in the bundle folder.
            dic_candidates = [
                os.path.join(full_dir, f"{lang}_{game}.dic"),
                os.path.join(full_dir, f"{game}_{lang}.dic"),
            ]
            if any(os.path.isfile(p) for p in dic_candidates):
                compressed_pairs.add((lang, game))
                discovered_pairs.add((lang, game))

    # 2) Add flatlist-only pairs as fallback
    if os.path.isdir(flat_root):
        for pattern in ("*.DIC", "*.dic"):
            for path in glob.glob(os.path.join(flat_root, pattern)):
                parsed = _parse_lang_game_token(os.path.basename(path))
                if not parsed:
                    continue
                flat_pairs.add(parsed)
                discovered_pairs.add(parsed)

    langs = sorted({lang for lang, _ in discovered_pairs})
    games = sorted({game for _, game in discovered_pairs})

    return {
        "pairs": sorted(discovered_pairs),
        "languages": langs,
        "games": games,
        "from_compressed": sorted(compressed_pairs),
        "from_flatlists": sorted(flat_pairs),
        "compressed_root": compressed_root,
        "flat_root": flat_root,
    }


def _resolve_source_export_paths(lang: str, game: str, output_dir: str = OUTPUT_DIR) -> Dict[str, str | None]:
    """Resolve existing source files for one (lang, game) export pair."""
    compressed_root = os.path.join(output_dir, "Compressed_dics")
    flat_root = os.path.join(output_dir, "Flatlists")

    bundle_candidates = [
        os.path.join(compressed_root, f"{lang}_{game}"),
        os.path.join(compressed_root, f"{game}_{lang}"),
    ]

    dic_candidates: List[str] = []
    aff_candidates: List[str] = []
    for bdir in bundle_candidates:
        dic_candidates.extend([
            os.path.join(bdir, f"{lang}_{game}.dic"),
            os.path.join(bdir, f"{game}_{lang}.dic"),
        ])
        aff_candidates.extend([
            os.path.join(bdir, f"{lang}_{game}.aff"),
            os.path.join(bdir, f"{game}_{lang}.aff"),
        ])

    flat_candidates = [
        os.path.join(flat_root, f"{game}_{lang}.DIC"),
        os.path.join(flat_root, f"{lang}_{game}.DIC"),
        os.path.join(flat_root, f"{game}_{lang}.dic"),
        os.path.join(flat_root, f"{lang}_{game}.dic"),
    ]

    dic_path = next((p for p in dic_candidates if os.path.isfile(p)), None)
    aff_path = next((p for p in aff_candidates if os.path.isfile(p)), None)
    flat_path = next((p for p in flat_candidates if os.path.isfile(p)), None)

    return {
        "compressed_dic": dic_path,
        "compressed_aff": aff_path,
        "flat_dic": flat_path,
    }


def _read_hunspell_dic_entries(dic_path: str) -> List[Tuple[str, Set[str]]]:
    """Read .dic/.DIC lines as (word, flags_set), skipping count header if present."""
    entries: List[Tuple[str, Set[str]]] = []
    if not dic_path or not os.path.isfile(dic_path):
        return entries

    with open(dic_path, "rb") as fh:
        raw = fh.read()

    text = None
    if raw.startswith(codecs.BOM_UTF16_LE):
        try:
            text = raw[len(codecs.BOM_UTF16_LE):].decode("utf-16-le")
        except UnicodeDecodeError:
            text = None

    if text is None:
        for enc in ("utf-8-sig", "utf-8", "latin-1"):
            try:
                text = raw.decode(enc)
                break
            except UnicodeDecodeError:
                continue

    if text is None:
        text = raw.decode("latin-1", errors="replace")

    raw_lines = [ln.strip() for ln in text.splitlines() if ln.strip()]

    if raw_lines and raw_lines[0].isdigit():
        data_lines = raw_lines[1:]
    else:
        data_lines = raw_lines

    for line in data_lines:
        base, _, tail = line.partition("/")
        word = base.strip()
        if not word:
            continue

        flags: Set[str] = set()
        tail = tail.strip()
        if tail:
            if "," in tail:
                flags.update([p.strip() for p in tail.split(",") if p.strip()])
            else:
                flags.update(list(tail))

        entries.append((word, flags))

    return entries


def _choose_surface(existing: str, candidate: str) -> str:
    """Deterministically choose a representative surface for one lowercase lemma."""
    if not existing:
        return candidate
    if not candidate:
        return existing
    if existing == candidate:
        return existing

    ex_cap = existing[0].isupper()
    ca_cap = candidate[0].isupper()
    if ex_cap != ca_cap:
        return candidate if ca_cap else existing

    # Stable tie-breaker: shorter first, then case-insensitive lexical.
    ex_key = (len(existing), existing.casefold(), existing)
    ca_key = (len(candidate), candidate.casefold(), candidate)
    return candidate if ca_key < ex_key else existing


def _merge_source_game_exports(
    lang: str,
    games: List[str],
    output_dir: str = OUTPUT_DIR,
    strict_mode: bool = False,
    source_type: str = "compressed",
) -> Dict[str, Any]:
    """Merge all selected source game exports for one language."""
    merged: Dict[str, Dict[str, Any]] = {}
    source_files: List[str] = []
    missing_pairs: List[str] = []

    if source_type not in {"compressed", "flat"}:
        raise ValueError("source_type must be 'compressed' or 'flat'")

    for game in games:
        if game == "ANK":
            continue

        paths = _resolve_source_export_paths(lang=lang, game=game, output_dir=output_dir)

        chosen = paths["compressed_dic"] if source_type == "compressed" else paths["flat_dic"]
        if chosen:
            source_files.append(chosen)
            source_entries = _read_hunspell_dic_entries(chosen)
        else:
            missing_pairs.append(f"{game}/{lang}")
            if strict_mode:
                raise FileNotFoundError(
                    f"No {source_type} export dictionary found for source pair {game}/{lang}"
                )
            continue

        for word, flags in source_entries:
            key = word
            if key not in merged:
                merged[key] = {
                    "surface": word,
                    "flags": set(flags),
                    "sources": {game},
                }
            else:
                merged[key]["flags"].update(flags)
                merged[key]["sources"].add(game)

    used_flags: Set[str] = set()
    for row in merged.values():
        used_flags.update(row["flags"])

    return {
        "lang": lang,
        "entries": merged,
        "source_files": sorted(set(source_files)),
        "missing_pairs": missing_pairs,
        "used_flags": used_flags,
        "entry_count": len(merged),
    }


def _copy_tree_no_overwrite(src_dir: str, dst_dir: str) -> int:
    """Copy files recursively, preserving existing destination files."""
    copied = 0
    for root, _, files in os.walk(src_dir):
        rel_root = os.path.relpath(root, src_dir)
        target_root = dst_dir if rel_root == "." else os.path.join(dst_dir, rel_root)
        os.makedirs(target_root, exist_ok=True)
        for name in files:
            src_file = os.path.join(root, name)
            dst_file = os.path.join(target_root, name)
            if os.path.exists(dst_file):
                continue
            shutil.copy2(src_file, dst_file)
            copied += 1
    return copied


def _write_ank_language_exports(
    lang: str,
    merged_rows: Dict[str, Dict[str, Any]],
    used_flags: Set[str],
    output_dir: str = OUTPUT_DIR,
) -> Dict[str, Any]:
    """Write ANK consolidated outputs for one language."""
    if not merged_rows:
        raise ValueError(f"No merged rows to export for language '{lang}'")

    out_flat = os.path.join(output_dir, "Flatlists")
    out_comp = os.path.join(output_dir, "Compressed_dics")
    out_full = os.path.join(output_dir, "Full_dics")
    os.makedirs(out_flat, exist_ok=True)
    os.makedirs(out_comp, exist_ok=True)
    os.makedirs(out_full, exist_ok=True)

    game_tag = "ANK"
    flat_key = f"{game_tag}_{lang}"   # keep current flat naming convention
    pair_key = f"{lang}_{game_tag}"   # keep current compressed/full naming convention

    sorted_items = sorted(merged_rows.values(), key=lambda r: r["surface"].casefold())

    # Flatlist (Word-compatible by default)
    flat_path = _write_flatlist_msword_unicode(
        os.path.join(out_flat, f"{flat_key}.DIC"),
        [row["surface"] for row in sorted_items],
    )

    # Resolve reference Hunspell dictionary for this language.
    resolved = resolve_hunspell_paths(lang, dic_folder=DIC_FOLDER)
    if not resolved.get("ok"):
        raise FileNotFoundError(
            f"Cannot resolve Hunspell dic+aff for '{lang}' while writing ANK export"
        )
    std_dic_path = resolved["dic"]
    std_aff_path = resolved["aff"]

    affixes = parse_aff_file(std_aff_path)
    aff_blocks = _extract_raw_aff_blocks(std_aff_path)
    flag_mode = affixes.get("flag_mode", "single")

    def _flags_to_str(flags: Set[str]) -> str:
        if not flags:
            return ""
        if flag_mode == "num":
            return "/" + ",".join(sorted(flags))
        return "/" + "".join(sorted(flags))

    # Compressed bundle
    comp_dir = os.path.join(out_comp, pair_key)
    os.makedirs(comp_dir, exist_ok=True)
    comp_dic_path = os.path.join(comp_dir, f"{pair_key}.dic")
    comp_aff_path = os.path.join(comp_dir, f"{pair_key}.aff")

    with open(comp_dic_path, "w", encoding="utf-8") as fh:
        fh.write(f"{len(sorted_items)}\n")
        for row in sorted_items:
            fh.write(row["surface"] + _flags_to_str(set(row["flags"])) + "\n")

    custom_aff_text = _build_custom_aff(std_aff_path, set(used_flags), lang, aff_blocks)
    with open(comp_aff_path, "w", encoding=aff_blocks["encoding"]) as fh:
        fh.write(custom_aff_text)

    # Full bundle: copy reference files + generated pair
    full_dir = os.path.join(out_full, pair_key)
    os.makedirs(full_dir, exist_ok=True)
    copied_ref_files = _copy_tree_no_overwrite(os.path.dirname(std_dic_path), full_dir)

    full_dic_path = os.path.join(full_dir, f"{pair_key}.dic")
    full_aff_path = os.path.join(full_dir, f"{pair_key}.aff")
    shutil.copy2(comp_dic_path, full_dic_path)
    shutil.copy2(comp_aff_path, full_aff_path)

    return {
        "flat_dic_path": flat_path,
        "compressed_dic_path": comp_dic_path,
        "compressed_aff_path": comp_aff_path,
        "full_dic_dir": full_dir,
        "full_dic_path": full_dic_path,
        "full_aff_path": full_aff_path,
        "copied_ref_files": copied_ref_files,
        "stats": {
            "entries": len(sorted_items),
            "flags_used": sorted(used_flags),
        },
    }


def _write_ank_flat_only(
    lang: str,
    merged_rows: Dict[str, Dict[str, Any]],
    output_dir: str = OUTPUT_DIR,
) -> str:
    """Write ANK flatlist from pre-merged rows only (Word-compatible)."""
    out_flat = os.path.join(output_dir, "Flatlists")
    os.makedirs(out_flat, exist_ok=True)
    sorted_items = sorted(merged_rows.values(), key=lambda r: r["surface"].casefold())
    return _write_flatlist_msword_unicode(
        os.path.join(out_flat, f"ANK_{lang}.DIC"),
        [row["surface"] for row in sorted_items],
    )


def _normalize_selector_games(games: Union[List[str], str], discovered_games: List[str]) -> List[str]:
    """Normalize games selector to an uppercase list, with all support."""
    if isinstance(games, str):
        token = games.strip().lower()
        if token == "all":
            return [g for g in discovered_games if g != "ANK"]
        if not token:
            return []
        return [games.strip().upper()]

    cleaned = [str(g).strip().upper() for g in games if str(g).strip()]
    return [g for g in cleaned if g != "ANK"]


def _normalize_selector_languages(
    languages: Union[List[str], str],
    discovered_languages: List[str],
) -> List[str]:
    """Normalize language selector to canonical codes, with all support."""
    if isinstance(languages, str):
        token = languages.strip().lower()
        if token == "all":
            return discovered_languages
        if not token:
            return []
        return [normalize_language_code(token)]

    normalized: List[str] = []
    for lang in languages:
        txt = str(lang).strip()
        if not txt:
            continue
        normalized.append(normalize_language_code(txt))

    # Preserve order and deduplicate.
    return list(dict.fromkeys(normalized))


def consolidate_ank_exports_by_language(
    games: Union[List[str], str] = "all",
    languages: Union[List[str], str] = "all",
    output_dir: str = OUTPUT_DIR,
    strict_mode: bool = False,
) -> Dict[str, Any]:
    """
    Consolidate existing game exports by language into synthetic ANK exports.

    Input scope restriction:
      - Reads only existing exports under output_dir (Compressed_dics/Flatlists).
      - Does not read TB, i18n corpus, or intermediary tokenization files.

    Args:
      games: source games list or 'all' discovered from exports
      languages: language code, list of codes, or 'all' discovered from exports
      output_dir: root output folder (default OUTPUT_DIR)
      strict_mode: fail fast on missing source pairs when True

    Returns:
      Dict with runs[] and summary, similar to run_pipeline_batch().
    """
    t0 = time.time()

    discovery = _discover_available_export_pairs(output_dir=output_dir)
    discovered_games = discovery["games"]
    discovered_languages = discovery["languages"]

    games_to_run = _normalize_selector_games(games, discovered_games)
    langs_to_run = _normalize_selector_languages(languages, discovered_languages)

    if not games_to_run:
        raise ValueError("No source games selected/found in exports")
    if not langs_to_run:
        raise ValueError("No languages selected/found in exports")

    print("=" * 80)
    print("ANK CONSOLIDATION FROM EXISTING EXPORTS")
    print("ANK CONSOLIDATION FROM EXISTING EXPORTS")
    print(f"Output dir        : {output_dir}")
    print(f"Available games   : {discovered_games}")
    print(f"Available langs   : {discovered_languages}")
    print(f"Selected games    : {games_to_run}")
    print(f"Selected languages: {langs_to_run}")
    print(f"Strict mode       : {strict_mode}")
    print("=" * 80)

    runs: List[Dict[str, Any]] = []
    runs: List[Dict[str, Any]] = []
    for lang in langs_to_run:
        run_t0 = time.time()
        run_result: Dict[str, Any] = {
            "language": lang,
            "target_game": "ANK",
            "status": "ok",
            "error": "",
            "source_games": games_to_run,
            "source_files": [],
            "missing_pairs": [],
            "artifacts": {},
            "metrics": {},
            "timings": {},
        }

        print("\n" + "-" * 80)
        print(f"▶ Consolidating language={lang} into ANK")
        print("-" * 80)

        try:
            merged_comp = _merge_source_game_exports(
                lang=lang,
                games=games_to_run,
                output_dir=output_dir,
                strict_mode=strict_mode,
                source_type="compressed",
            )
            merged_flat = _merge_source_game_exports(
                lang=lang,
                games=games_to_run,
                output_dir=output_dir,
                strict_mode=False,
                source_type="flat",
            )

            if merged_comp["entry_count"] == 0:
                raise ValueError(
                    f"No compressed source entries found for language '{lang}' with selected games"
                )

            write_result = _write_ank_language_exports(
                lang=lang,
                merged_rows=merged_comp["entries"],
                used_flags=merged_comp["used_flags"],
                output_dir=output_dir,
            )
            if merged_flat["entry_count"] > 0:
                write_result["flat_dic_path"] = _write_ank_flat_only(
                    lang=lang,
                    merged_rows=merged_flat["entries"],
                    output_dir=output_dir,
                )

            run_result["source_files"] = sorted(set(merged_comp["source_files"] + merged_flat["source_files"]))
            run_result["missing_pairs"] = sorted(set(merged_comp["missing_pairs"] + merged_flat["missing_pairs"]))
            run_result["artifacts"].update(write_result)
            run_result["metrics"]["merged_entries_compressed"] = merged_comp["entry_count"]
            run_result["metrics"]["merged_entries_flat"] = merged_flat["entry_count"]
            run_result["metrics"]["flags_used"] = sorted(merged_comp["used_flags"])
            run_result["metrics"]["source_files_count"] = len(run_result["source_files"])
            run_result["metrics"]["missing_pairs_count"] = len(run_result["missing_pairs"])

            print(f"  Source files used (all): {len(run_result['source_files'])}")
            if run_result["missing_pairs"]:
                print(f"  Missing source pairs: {len(run_result['missing_pairs'])}")
            print(f"  Merged entries (compressed): {merged_comp['entry_count']:,}")
            print(f"  Merged entries (flat)      : {merged_flat['entry_count']:,}")
            print(f"  Flat dic     -> {write_result['flat_dic_path']}")
            print(f"  Compressed   -> {write_result['compressed_dic_path']}")
            print(f"  Full bundle  -> {write_result['full_dic_dir']}")

        except Exception as err:
            run_result["status"] = "failed"
            run_result["error"] = f"{type(err).__name__}: {err}"
            print(f"❌ Failed consolidation for lang={lang}: {run_result['error']}")
            if strict_mode:
                run_result["timings"]["total"] = round(time.time() - run_t0, 3)
                runs.append(run_result)
                elapsed = round(time.time() - t0, 3)
                return {
                    "runs": runs,
                    "summary": {
                        "status": "failed",
                        "reason": "strict_mode abort on first error",
                        "processed_languages": len(runs),
                        "ok_languages": sum(1 for r in runs if r["status"] == "ok"),
                        "failed_languages": sum(1 for r in runs if r["status"] == "failed"),
                        "elapsed_seconds": elapsed,
                    },
                }

        run_result["timings"]["total"] = round(time.time() - run_t0, 3)
        runs.append(run_result)

    elapsed = round(time.time() - t0, 3)
    ok_languages = sum(1 for r in runs if r["status"] == "ok")
    failed_languages = sum(1 for r in runs if r["status"] == "failed")

    summary = {
        "status": "ok" if failed_languages == 0 else "partial",
        "selected_games": games_to_run,
        "selected_languages": langs_to_run,
        "processed_languages": len(runs),
        "ok_languages": ok_languages,
        "failed_languages": failed_languages,
        "elapsed_seconds": elapsed,
    }

    print("\n" + "=" * 80)
    print("ANK CONSOLIDATION COMPLETE")
    print(f"Status        : {summary['status']}")
    print(f"Processed     : {summary['processed_languages']}")
    print(f"OK / Failed   : {summary['ok_languages']} / {summary['failed_languages']}")
    print(f"Elapsed (sec) : {summary['elapsed_seconds']}")
    print("=" * 80)

    return {"runs": runs, "summary": summary}
    return {"runs": runs, "summary": summary}

# Examples:
# ank_report = consolidate_ank_exports_by_language(games="all", languages="all")
# ank_report = consolidate_ank_exports_by_language(games=["DOFUS", "WAKFU"], languages=["es", "pt"])

In [9]:
# Hotfix: prune generated duplicates from compressed/full ANK exports
# Keeps flatlist unchanged (all merged surfaces), but removes explicit entries that
# are derivable from another flagged base (e.g., plural listed separately).

def _prune_generated_redundant_entries(lang: str, merged_rows: Dict[str, Dict[str, Any]]) -> Dict[str, Any]:
    """
    Remove explicit entries that are generated by another flagged entry.

    Conservative rule:
      - Drop only rows with NO flags when that exact surface is generated by
        at least one other row's flags.
      - Keep rows that carry explicit flags to avoid over-pruning valid lemmas.
    """
    if not merged_rows:
        return {
            "rows": {},
            "dropped_keys": set(),
            "generated_hits": 0,
            "errors": [],
        }

    resolved = resolve_hunspell_paths(lang, dic_folder=DIC_FOLDER)
    if not resolved.get("ok"):
        return {
            "rows": dict(merged_rows),
            "dropped_keys": set(),
            "generated_hits": 0,
            "errors": [f"Hunspell paths not resolved for {lang}"],
        }

    try:
        affixes = parse_aff_file(resolved["aff"])
    except Exception as err:
        return {
            "rows": dict(merged_rows),
            "dropped_keys": set(),
            "generated_hits": 0,
            "errors": [f"AFF parse failed: {type(err).__name__}: {err}"],
        }

    all_keys = set(merged_rows.keys())
    generated_by: Dict[str, Set[str]] = {}
    errors: List[str] = []

    for src_key, row in merged_rows.items():
        src_surface = str(row.get("surface", "")).strip()
        src_flags = set(row.get("flags", set()))
        if not src_surface or not src_flags:
            continue

        try:
            forms = generate_word_forms(src_surface, sorted(src_flags), affixes)
        except Exception as err:
            errors.append(f"{src_surface}: {type(err).__name__}: {err}")
            continue

        for form in forms:
            form_key = str(form).casefold()
            if form_key == src_key:
                continue
            if form_key in all_keys:
                generated_by.setdefault(form_key, set()).add(src_key)

    drop_keys: Set[str] = set()
    for cand_key, _sources in generated_by.items():
        cand_flags = set(merged_rows[cand_key].get("flags", set()))
        if not cand_flags:
            drop_keys.add(cand_key)

    pruned_rows = {k: v for k, v in merged_rows.items() if k not in drop_keys}

    return {
        "rows": pruned_rows,
        "dropped_keys": drop_keys,
        "generated_hits": len(generated_by),
        "errors": errors,
    }


def _write_ank_language_exports(
    lang: str,
    merged_rows: Dict[str, Dict[str, Any]],
    used_flags: Set[str],
    output_dir: str = OUTPUT_DIR,
) -> Dict[str, Any]:
    """Write ANK consolidated outputs for one language."""
    if not merged_rows:
        raise ValueError(f"No merged rows to export for language '{lang}'")

    out_flat = os.path.join(output_dir, "Flatlists")
    out_comp = os.path.join(output_dir, "Compressed_dics")
    out_full = os.path.join(output_dir, "Full_dics")
    os.makedirs(out_flat, exist_ok=True)
    os.makedirs(out_comp, exist_ok=True)
    os.makedirs(out_full, exist_ok=True)

    game_tag = "ANK"
    flat_key = f"{game_tag}_{lang}"   # keep current flat naming convention
    pair_key = f"{lang}_{game_tag}"   # keep current compressed/full naming convention

    # Flatlist: keep all merged forms as requested.
    flat_items = sorted(merged_rows.values(), key=lambda r: r["surface"].casefold())

    # Compressed/full: prune redundant explicit generated forms.
    prune_info = _prune_generated_redundant_entries(lang=lang, merged_rows=merged_rows)
    compressed_rows = prune_info["rows"]
    compressed_items = sorted(compressed_rows.values(), key=lambda r: r["surface"].casefold())

    # Flatlist (Word-compatible by default)
    flat_path = _write_flatlist_msword_unicode(
        os.path.join(out_flat, f"{flat_key}.DIC"),
        [row["surface"] for row in flat_items],
    )

    # Resolve reference Hunspell dictionary for this language.
    resolved = resolve_hunspell_paths(lang, dic_folder=DIC_FOLDER)
    if not resolved.get("ok"):
        raise FileNotFoundError(
            f"Cannot resolve Hunspell dic+aff for '{lang}' while writing ANK export"
        )
    std_dic_path = resolved["dic"]
    std_aff_path = resolved["aff"]

    affixes = parse_aff_file(std_aff_path)
    aff_blocks = _extract_raw_aff_blocks(std_aff_path)
    flag_mode = affixes.get("flag_mode", "single")

    def _flags_to_str(flags: Set[str]) -> str:
        if not flags:
            return ""
        if flag_mode == "num":
            return "/" + ",".join(sorted(flags))
        return "/" + "".join(sorted(flags))

    # Keep AFF minimal: only flags used by compressed rows.
    used_flags_final: Set[str] = set()
    for row in compressed_items:
        used_flags_final.update(set(row.get("flags", set())))

    # Compressed bundle
    comp_dir = os.path.join(out_comp, pair_key)
    os.makedirs(comp_dir, exist_ok=True)
    comp_dic_path = os.path.join(comp_dir, f"{pair_key}.dic")
    comp_aff_path = os.path.join(comp_dir, f"{pair_key}.aff")

    with open(comp_dic_path, "w", encoding="utf-8") as fh:
        fh.write(f"{len(compressed_items)}\n")
        for row in compressed_items:
            fh.write(row["surface"] + _flags_to_str(set(row["flags"])) + "\n")

    custom_aff_text = _build_custom_aff(std_aff_path, used_flags_final, lang, aff_blocks)
    with open(comp_aff_path, "w", encoding=aff_blocks["encoding"]) as fh:
        fh.write(custom_aff_text)

    # Full bundle: copy reference files + generated pair
    full_dir = os.path.join(out_full, pair_key)
    os.makedirs(full_dir, exist_ok=True)
    copied_ref_files = _copy_tree_no_overwrite(os.path.dirname(std_dic_path), full_dir)

    full_dic_path = os.path.join(full_dir, f"{pair_key}.dic")
    full_aff_path = os.path.join(full_dir, f"{pair_key}.aff")
    shutil.copy2(comp_dic_path, full_dic_path)
    shutil.copy2(comp_aff_path, full_aff_path)

    return {
        "flat_dic_path": flat_path,
        "compressed_dic_path": comp_dic_path,
        "compressed_aff_path": comp_aff_path,
        "full_dic_dir": full_dir,
        "full_dic_path": full_dic_path,
        "full_aff_path": full_aff_path,
        "copied_ref_files": copied_ref_files,
        "stats": {
            "flat_entries": len(flat_items),
            "compressed_entries": len(compressed_items),
            "pruned_generated_entries": len(prune_info["dropped_keys"]),
            "generated_match_candidates": prune_info["generated_hits"],
            "flags_used": sorted(used_flags_final),
            "prune_errors": prune_info["errors"],
        },
    }

In [10]:
# Exemple d'appel annoté: chaque paramètre est expliqué à droite
batch_report = run_pipeline_batch(
    languages="all",                     # langues à traiter (codes ISO, ex: ["es", "pt"]) 
    games=GAMES,                        # jeux à traiter (clés présentes dans TB_PATHS)
    sample=0,                           # 0 = tout le dictionnaire, >0 = taille d'échantillon
    workers=8,                          # nombre de threads pour la recherche de wordforms
    batch_size=50,                      # taille de lot envoyée à chaque worker
    add_verb_flags=False,               # True = tester aussi les flags verbaux
    quorum=0.5,                         # seuil de validation des flags (0.5 = 50%)
    cleanup_stale=True,                 # supprime les sorties intermédiaires précédentes
    strict_mode=False,                  # True = stop au 1er échec, False = continue
    output_folder=None,                 # None = utilise INTERMEDIARY_DIR
    prewarm_i18n=True,                  # précharge le corpus i18n filtré avant traitement
)

batch_report["summary"]

BATCH PIPELINE (STEPS 1-5)
Games     : ['DOFUS', 'WAKFU', 'TOUCH', 'WAVEN', 'RETRO', 'ONE_MORE_GATE', 'ANKANIMATION']
Languages : ['de', 'en', 'es', 'fr', 'pt']
Work dir  : processing_dics
Options   : sample=0, workers=8, batch_size=50, add_verb_flags=False, quorum=0.5, cleanup_stale=True, strict_mode=False
Dictionary preflight:
  ✅ de: dics\de_dic\de_DE_frami.dic
  ✅ en: dics\en_dic\en_GB.dic
  ✅ es: dics\es_dic\es\es_ES.dic
  ✅ fr: dics\fr_dic\fr.dic
  ✅ pt: dics\pt_dic\pt_BR\pt_BR.dic

--------------------------------------------------------------------------------
▶ Processing pair: game=DOFUS | language=de
--------------------------------------------------------------------------------
Prewarming filtered i18n for DOFUS/de...
  [corpus] Lingua detector  : target=DE  fallbacks=['FRENCH', 'ENGLISH', 'SPANISH', 'PORTUGUESE']
  [corpus] Thresholds       : short(<7w)=compare vs EN/FR  long(>=7w)=0.12
  [corpus] Loading DOFUS/de: i18n/DOFUS/de.json
  [corpus]   -> lang-filtered : 7,840 

{'status': 'partial',
 'total_pairs': 35,
 'processed_pairs': 35,
 'ok_pairs': 31,
 'failed_pairs': ['WAKFU/de',
  'RETRO/de',
  'ONE_MORE_GATE/pt',
  'ANKANIMATION/de'],
 'failed_pairs_count': 4,
 'elapsed_seconds': 11088.658}

In [11]:
# Example run
ank_es_report = consolidate_ank_exports_by_language(
    games="all",
    languages="all",
    output_dir=OUTPUT_DIR,
    strict_mode=False,
)
ank_es_report["summary"]

ANK CONSOLIDATION FROM EXISTING EXPORTS
ANK CONSOLIDATION FROM EXISTING EXPORTS
Output dir        : output_dics
Available games   : ['ANKANIMATION', 'DOFUS', 'ONE_MORE_GATE', 'RETRO', 'TOUCH', 'WAKFU', 'WAVEN']
Available langs   : ['de', 'en', 'es', 'fr', 'pt']
Selected games    : ['ANKANIMATION', 'DOFUS', 'ONE_MORE_GATE', 'RETRO', 'TOUCH', 'WAKFU', 'WAVEN']
Selected languages: ['de', 'en', 'es', 'fr', 'pt']
Strict mode       : False

--------------------------------------------------------------------------------
▶ Consolidating language=de into ANK
--------------------------------------------------------------------------------
  Source files used (all): 8
  Missing source pairs: 3
  Merged entries (compressed): 30,961
  Merged entries (flat)      : 33,037
  Flat dic     -> output_dics\Flatlists\ANK_de.DIC
  Compressed   -> output_dics\Compressed_dics\de_ANK\de_ANK.dic
  Full bundle  -> output_dics\Full_dics\de_ANK

--------------------------------------------------------------------

{'status': 'ok',
 'selected_games': ['ANKANIMATION',
  'DOFUS',
  'ONE_MORE_GATE',
  'RETRO',
  'TOUCH',
  'WAKFU',
  'WAVEN'],
 'selected_languages': ['de', 'en', 'es', 'fr', 'pt'],
 'processed_languages': 5,
 'ok_languages': 5,
 'failed_languages': 0,
 'elapsed_seconds': 56.162}

### Flatlists are now Word-compatible by default (no clone step)

In [ ]:
# Quick verification of default Flatlists output format
import os
import codecs

print("Legacy clone function still defined:", "clone_flatlists_to_msword_unicode" in globals())

flat_dir = OUTPUT_FLATLISTS_DIR
all_dic_like = sorted([f for f in os.listdir(flat_dir) if f.lower().endswith(".dic")])
upper_dic = [f for f in all_dic_like if f.endswith(".DIC")]
lower_dic = [f for f in all_dic_like if f.endswith(".dic")]

print(f"Flatlists folder: {flat_dir}")
print(f"Total *.dic/*.DIC files: {len(all_dic_like)}")
print(f"Uppercase .DIC files: {len(upper_dic)}")
print(f"Lowercase .dic files: {len(lower_dic)}")

# Prefer validating the fresh pipeline target first.
probe_names = ["WAKFU_es.DIC", "WAKFU_es.dic"] + all_dic_like
checked = []
for name in probe_names:
    if name in checked:
        continue
    checked.append(name)
    path = os.path.join(flat_dir, name)
    if not os.path.isfile(path):
        continue
    with open(path, "rb") as fh:
        head = fh.read(2)
    print(f"Probe file: {path}")
    print(f"Starts with UTF-16 LE BOM: {head == codecs.BOM_UTF16_LE}")
    break

upper_dic[:10], lower_dic[:10]

### Post-prod fix replace AFF files with custom headers

In [ ]:
# Smoke-check AFF header normalization for all configured languages
required_wordchars = set("'-0123456789")
required_try = set("'-")

for lang in sorted(HUNSPELL_PATHS.keys()):
    resolved = resolve_hunspell_paths(lang, dic_folder=DIC_FOLDER)
    if not resolved.get("ok"):
        print(f"{lang}: SKIP (no resolved Hunspell pair)")
        continue

    aff_path = resolved["aff"]
    aff_blocks = _extract_raw_aff_blocks(aff_path)
    custom_aff = _build_custom_aff(aff_path, used_flags=set(), lang=lang, aff_blocks=aff_blocks)

    wc_line = next((ln for ln in custom_aff.splitlines() if ln.upper().startswith("WORDCHARS ")), "")
    try_line = next((ln for ln in custom_aff.splitlines() if ln.upper().startswith("TRY ")), "")

    wc_chars = wc_line.split(None, 1)[1] if wc_line else ""
    try_chars = try_line.split(None, 1)[1] if try_line else ""

    wc_ok = required_wordchars.issubset(set(wc_chars))
    try_ok = required_try.issubset(set(try_chars))

    print(f"{lang}: WORDCHARS={'OK' if wc_ok else 'FAIL'} TRY={'OK' if try_ok else 'FAIL'}")
    if not wc_ok:
        print(f"  missing WORDCHARS: {''.join(sorted(required_wordchars - set(wc_chars)))}")
    if not try_ok:
        print(f"  missing TRY: {''.join(sorted(required_try - set(try_chars)))}")

    assert wc_ok, f"{lang}: WORDCHARS is missing required chars"
    assert try_ok, f"{lang}: TRY is missing required chars"

print("Header normalization smoke-check passed.")

In [ ]:
# Patch already-exported AFF files in OUTPUT_DIR (no pipeline rerun needed).
# Decisions applied:
#   - WORDCHARS must include apostrophe, hyphen, digits 0-9
#   - TRY must include hyphen and apostrophe
#   - No BREAK insertion
# Scope restriction:
#   - Only game-pair exported AFF names: <lang>_<GAME>.aff (e.g. de_WAVEN.aff)
#   - Excludes copied Hunspell source files like de_DE_frami.aff, es_ES.aff, es_CO.aff

import os
import re
import pandas as pd
from typing import List, Tuple, Dict, Any

APPLY_CHANGES = True   # False = preview/diff table only, True = write files
VERBOSE = False
SHOW_ONLY_CHANGED = False

required_wordchars = "'-0123456789"
required_try_chars = "-'"


# Valid game tokens for exported pair files.
_valid_games = {str(g).upper() for g in GAMES}
_valid_games.add("ANK")


def _is_game_pair_aff_name(filename: str) -> bool:
    """True only for exported pair files like de_WAKFU.aff / es_ANK.aff."""
    m = re.match(r"^([a-z]{2})_([A-Za-z0-9_]+)\.aff$", filename)
    if not m:
        return False
    game_token = m.group(2).upper()
    return game_token in _valid_games


def _detect_aff_encoding(aff_path: str) -> str:
    enc = "utf-8"
    try:
        with open(aff_path, "rb") as fb:
            for raw_line in fb:
                text = raw_line.decode("ascii", errors="ignore").strip()
                if text.upper().startswith("SET "):
                    parts = text.split()
                    if len(parts) >= 2:
                        set_name = parts[1].strip()
                        enc = set_name.replace("ISO8859-", "iso-8859-")
                    break
    except Exception:
        pass
    return enc


def _extract_try_wordchars_lines(header_lines: List[str]) -> Tuple[str, str]:
    try_line = ""
    wordchars_line = ""
    for line in header_lines:
        stripped = line.strip()
        if not stripped:
            continue
        parts = stripped.split(None, 1)
        key = parts[0].upper() if parts else ""
        if key == "TRY" and not try_line:
            try_line = line
        elif key == "WORDCHARS" and not wordchars_line:
            wordchars_line = line
    return try_line, wordchars_line


def _normalize_aff_header_lines_for_export(header_lines: List[str]) -> List[str]:
    """Normalize TRY/WORDCHARS in AFF header while preserving existing content order."""
    out: List[str] = []
    seen_try = False
    seen_wordchars = False

    for line in header_lines:
        stripped = line.strip()
        if not stripped:
            out.append(line)
            continue

        parts = stripped.split(None, 1)
        directive = parts[0].upper() if parts else ""

        if directive == "TRY":
            if seen_try:
                continue  # keep one effective TRY line
            seen_try = True
            existing = parts[1] if len(parts) > 1 else ""
            merged = existing
            for ch in required_try_chars:
                if ch not in merged:
                    merged += ch
            out.append(f"TRY {merged}")
            continue

        if directive == "WORDCHARS":
            if seen_wordchars:
                continue  # keep one effective WORDCHARS line
            seen_wordchars = True
            existing = parts[1] if len(parts) > 1 else ""
            merged = existing
            for ch in required_wordchars:
                if ch not in merged:
                    merged += ch
            out.append(f"WORDCHARS {merged}")
            continue

        out.append(line)

    # Insert TRY if missing
    if not seen_try:
        insert_idx = 0
        for i, line in enumerate(out):
            d = line.strip().split(None, 1)
            key = d[0].upper() if d else ""
            if key in ("SET", "FLAG"):
                insert_idx = i + 1
        out.insert(insert_idx, f"TRY {required_try_chars}")

    # Insert WORDCHARS if missing (right after TRY when possible)
    if not seen_wordchars:
        insert_idx = 0
        for i, line in enumerate(out):
            d = line.strip().split(None, 1)
            key = d[0].upper() if d else ""
            if key == "TRY":
                insert_idx = i + 1
                break
            if key in ("SET", "FLAG"):
                insert_idx = i + 1
        out.insert(insert_idx, f"WORDCHARS {required_wordchars}")

    return out


def _patch_exported_aff_file(aff_path: str, apply_changes: bool = True) -> Dict[str, Any]:
    """Patch one exported AFF file in-place and return a detailed diff record."""
    rec: Dict[str, Any] = {
        "file": aff_path,
        "status": "",
        "changed": False,
        "try_before": "",
        "try_after": "",
        "wordchars_before": "",
        "wordchars_after": "",
    }

    try:
        enc = _detect_aff_encoding(aff_path)
        with open(aff_path, "r", encoding=enc, errors="replace") as fh:
            lines = fh.read().splitlines()

        # Header is everything before first PFX/SFX directive.
        split_idx = len(lines)
        for i, line in enumerate(lines):
            parts = line.strip().split()
            if len(parts) >= 2 and parts[0] in ("PFX", "SFX"):
                split_idx = i
                break

        header = lines[:split_idx]
        body = lines[split_idx:]

        before_try, before_wc = _extract_try_wordchars_lines(header)
        new_header = _normalize_aff_header_lines_for_export(header)
        after_try, after_wc = _extract_try_wordchars_lines(new_header)

        rec["try_before"] = before_try
        rec["try_after"] = after_try
        rec["wordchars_before"] = before_wc
        rec["wordchars_after"] = after_wc

        new_lines = new_header + body
        rec["changed"] = new_lines != lines

        if rec["changed"] and apply_changes:
            with open(aff_path, "w", encoding=enc) as fh:
                fh.write("\n".join(new_lines) + "\n")
            rec["status"] = "patched"
        elif rec["changed"]:
            rec["status"] = "would_patch"
        else:
            rec["status"] = "no_change"

        return rec
    except Exception as err:
        rec["status"] = f"error: {type(err).__name__}: {err}"
        return rec


# Collect exported game-pair AFF files from current OUTPUT_DIR only.
output_root = os.path.abspath(OUTPUT_DIR)
source_root = os.path.abspath(DIC_FOLDER)

export_aff_files: List[str] = []
for root, _, files in os.walk(output_root):
    for name in files:
        if not name.lower().endswith(".aff"):
            continue
        if not _is_game_pair_aff_name(name):
            continue

        p = os.path.abspath(os.path.join(root, name))

        # Safety guard: never patch files under source dictionary root.
        try:
            if os.path.commonpath([p, source_root]) == source_root:
                continue
        except Exception:
            pass

        export_aff_files.append(p)

export_aff_files = sorted(set(export_aff_files))
print(f"Found {len(export_aff_files)} game-pair exported .aff file(s) under {output_root}")
print(f"Mode: {'APPLY' if APPLY_CHANGES else 'PREVIEW'}")

records: List[Dict[str, Any]] = []
for aff_path in export_aff_files:
    rec = _patch_exported_aff_file(aff_path, apply_changes=APPLY_CHANGES)
    records.append(rec)
    if VERBOSE:
        print(f"- {rec['status']:10} {aff_path}")

preview_df = pd.DataFrame(records)
if not preview_df.empty:
    preview_df.insert(0, "lang", preview_df["file"].str.extract(r"(?:^|[\\/])([a-z]{2})_([A-Za-z0-9_]+)\.aff$", expand=True)[0].fillna(""))
    preview_df.insert(1, "game", preview_df["file"].str.extract(r"(?:^|[\\/])([a-z]{2})_([A-Za-z0-9_]+)\.aff$", expand=True)[1].fillna(""))

if SHOW_ONLY_CHANGED and not preview_df.empty:
    display_df = preview_df[preview_df["changed"]].copy()
else:
    display_df = preview_df.copy()

changed = int((preview_df["status"].isin(["would_patch", "patched"]).sum()) if not preview_df.empty else 0)
unchanged = int((preview_df["status"] == "no_change").sum() if not preview_df.empty else 0)
errors = int(preview_df["status"].astype(str).str.startswith("error:").sum() if not preview_df.empty else 0)

print("\nSummary")
print(f"  changed  : {changed}")
print(f"  unchanged: {unchanged}")
print(f"  errors   : {errors}")

# Main visual diff table: TRY/WORDCHARS before vs after.
if display_df.empty:
    print("No files to display.")
else:
    display_cols = [
        "status", "lang", "game", "file",
        "try_before", "try_after",
        "wordchars_before", "wordchars_after",
    ]
    display(display_df[display_cols].sort_values(["status", "lang", "game", "file"]).reset_index(drop=True))

In [ ]:
# Sanity check: resolved Hunspell path candidates
fr_check = resolve_hunspell_paths("fr", dic_folder=DIC_FOLDER)
fr_check

# TESTS

## Language detection audit — v0 vs filtered diff

Loads `v0_WAKFU_es_missing_wordforms.csv` (pre-detection baseline) and the current
`WAKFU_es_missing_wordforms.csv` (post-detection), then surfaces every `new_found_form`
that was present in v0 but has disappeared from the new version.

Three possible causes of disappearance:
1. **Correct removal** — the entry's source string was a French/English fallback.
2. **False positive** — a valid Spanish entry was mis-classified (short names, neologisms, game-specific vocab).
3. **Base-word also gone** — the entire `base_word` row was removed (all its corpus strings were filtered out).

The diff is saved as `v0_vs_new_WAKFU_es_diff.csv` in `processing_dics/` for manual review.


In [ ]:
import pandas as pd
import os

# ── Paths ─────────────────────────────────────────────────────────────────────
_V0_CSV  = os.path.join(INTERMEDIARY_DIR, "v0_WAKFU_es_missing_wordforms.csv")
_NEW_CSV = os.path.join(INTERMEDIARY_DIR, "WAKFU_es_missing_wordforms.csv")
_DIFF_CSV = os.path.join(INTERMEDIARY_DIR, "v0_vs_new_WAKFU_es_diff.csv")


def _parse_pipe(val) -> set:
    """Split a pipe-separated cell into a set of non-empty stripped strings."""
    if pd.isna(val) or str(val).strip() == "":
        return set()
    return {f.strip() for f in str(val).split("|") if f.strip()}


# ── Load ──────────────────────────────────────────────────────────────────────
df_v0  = pd.read_csv(_V0_CSV,  encoding="utf-8-sig")
df_new = pd.read_csv(_NEW_CSV, encoding="utf-8-sig")

# Build lookup: base_word (lower) → set of new_found_forms
v0_map  = {row["base_word"].lower(): _parse_pipe(row["new_found_forms"])
           for _, row in df_v0.iterrows()}
new_map = {row["base_word"].lower(): _parse_pipe(row["new_found_forms"])
           for _, row in df_new.iterrows()}

# ── Compute diff ──────────────────────────────────────────────────────────────
diff_rows = []

for base_lower, v0_forms in sorted(v0_map.items()):
    if not v0_forms:          # v0 row had no new_found_forms — nothing to miss
        continue

    new_forms   = new_map.get(base_lower, set())
    lost_forms  = v0_forms - new_forms
    kept_forms  = v0_forms & new_forms

    if not lost_forms:
        continue              # nothing removed for this base_word

    base_orig    = df_v0.loc[df_v0["base_word"].str.lower() == base_lower,
                             "base_word"].iloc[0]
    base_gone    = base_lower not in new_map   # entire row absent from new file

    diff_rows.append({
        "base_word"       : base_orig,
        "base_row_gone"   : base_gone,
        "lost_forms"      : " | ".join(sorted(lost_forms, key=str.lower)),
        "lost_count"      : len(lost_forms),
        "kept_forms"      : " | ".join(sorted(kept_forms, key=str.lower)),
        "kept_count"      : len(kept_forms),
        "v0_new_count"    : len(v0_forms),
    })

df_diff = pd.DataFrame(diff_rows)

# ── Summary stats ─────────────────────────────────────────────────────────────
total_v0_forms  = sum(len(f) for f in v0_map.values())
total_new_forms = sum(len(f) for f in new_map.values())
total_lost      = df_diff["lost_count"].sum()  if len(df_diff) else 0
base_gone_count = df_diff["base_row_gone"].sum() if len(df_diff) else 0

print("═" * 62)
print("  v0  → baseline (pre language-detection)")
print(f"       rows         : {len(df_v0):,}  base_words with new_found_forms")
print(f"       total forms  : {total_v0_forms:,}")
print("  new → post language-detection")
print(f"       rows         : {len(df_new):,}  base_words with new_found_forms")
print(f"       total forms  : {total_new_forms:,}")
print("─" * 62)
print(f"  Base-words with any lost form  : {len(df_diff):,}")
print(f"  Base-words whose row is gone   : {int(base_gone_count):,}  (all corpus strings filtered)")
print(f"  Total lost new_found_forms     : {int(total_lost):,}")
print(f"  Total retained new_found_forms : {total_new_forms:,}")
if total_v0_forms:
    pct_lost = total_lost / total_v0_forms * 100
    print(f"  Lost / v0 total                : {pct_lost:.1f}%")
print("═" * 62)

# ── Save diff CSV ─────────────────────────────────────────────────────────────
if len(df_diff):
    df_diff.to_csv(_DIFF_CSV, index=False, encoding="utf-8-sig")
    print(f"\nDiff saved → {_DIFF_CSV}")

# ── Interactive table: top losses ─────────────────────────────────────────────
if len(df_diff):
    print(f"\nTop-30 base_words by lost_count (potential false positives first):")
    display(
        df_diff.sort_values("lost_count", ascending=False)
               .head(30)
               .reset_index(drop=True)
               [["base_word", "base_row_gone", "lost_count", "lost_forms",
                 "kept_count", "kept_forms"]]
    )
else:
    print("\n✔  No new_found_forms were lost — language detection had no impact on this column.")

df_diff

In [ ]:
print(demorph_string("Ocult{[>1]?a:o}s"))          # Ocultas Ocultos
print(demorph_string("Pagar [#1] esquirla{[>1]?s:}")) # Pagar [#1] esquirlas esquirla
print(demorph_string("derrota{[=1]?:s}"))             # derrota derrotas


# A. Verify on Hunspell wrapper
```
echo hara-kiri | hunspell -d "C:\Users\Nelso\Documents\MundoDoce\TB2dic\dics\es_dic\es_ANK\es_ANK"
```

hunspell -d "C:\Users\Nelso\Documents\MundoDoce\TB2dic\dics\de_dic\de_DE_frami" "C:\Users\Nelso\test_words.txt"

In [ ]:
import subprocess
import tempfile
import os
import pandas as pd
from concurrent.futures import ThreadPoolExecutor, as_completed

# ── Config ────────────────────────────────────────────────────────────────────
_CHECK_GAME = "DOFUS"
_CHECK_LANG = "es"
_CHECK_KEY  = f"{_CHECK_GAME}_{_CHECK_LANG}"
INPUT_DIC   = os.path.abspath(os.path.join(INTERMEDIARY_DIR, f"{_CHECK_KEY}_filtered_tokens.dic"))
OUTPUT_TXT  = os.path.abspath(os.path.join(INTERMEDIARY_DIR, f"{_CHECK_KEY}_hunspell_check.txt"))
OUTPUT_CSV  = os.path.abspath(os.path.join(INTERMEDIARY_DIR, f"{_CHECK_KEY}_hunspell_check.csv"))

ES_DIC_BASE = os.path.abspath(os.path.splitext(HUNSPELL_PATHS["es"])[0])  # …/es_dic/es/es_ES

SAMPLE_ONLY  = False  # ← Full run
SAMPLE_SIZE  = 400    # only used when SAMPLE_ONLY = True
NUM_WORKERS  = 32      # parallel hunspell processes
CHUNK_SIZE   = 500    # words per chunk

# ── Load words (skip first line = word count header) ─────────────────────────
with open(INPUT_DIC, "r", encoding="utf-8") as f:
    lines = f.readlines()

all_words = [ln.strip() for ln in lines[1:] if ln.strip()]
words = all_words[:SAMPLE_SIZE] if SAMPLE_ONLY else all_words
print(f"{'Sample' if SAMPLE_ONLY else 'Full'}: {len(words)} / {len(all_words)} words loaded")

# ── Helper: parse one blank-line-delimited block for a single word ─────────────
def parse_block(word: str, block_lines: list[str]) -> tuple:
    """
    Hunspell emits one blank-line-delimited block per input word.
    For plain words the block has one line; for hyphenated words it has
    one line *per sub-word*.  We check every line in the block:
      * / + / -  → sub-word correct
      & word n off: s1, s2, …  → sub-word misspelled, suggestions available
      #           → sub-word misspelled, no suggestions

    isCorrect = True  only when ALL sub-words are correct
    isCorrect = False when ANY sub-word is misspelled
    suggestions = comma-joined suggestions from all misspelled sub-words
    """
    all_correct = True
    all_suggs   = []
    for line in block_lines:
        r = line.strip()
        if not r:
            continue
        if r.startswith("*") or r.startswith("+") or r.startswith("-"):
            pass  # sub-word OK
        elif r.startswith("&"):
            all_correct = False
            parts = r.split(":", 1)
            if len(parts) > 1:
                all_suggs.append(parts[1].strip())
        elif r.startswith("#"):
            all_correct = False
        else:
            # Unexpected line – treat as unknown
            return (word, None, r)
    return (word, all_correct, " | ".join(all_suggs))

# ── Helper: run hunspell on a chunk; return list of (word, isCorrect, suggestions) ──
def run_hunspell_chunk(chunk: list[str]) -> list[tuple]:
    """
    Pipe a list of words to hunspell, split output into per-word blocks
    (delimited by blank lines), and parse each block.

    Hunspell output format:
      One block per input word, separated by a blank line.
      For hyphenated words, each component gets its own result line
      inside the same block.
    """
    with tempfile.NamedTemporaryFile(mode="w", encoding="utf-8",
                                     suffix=".txt", delete=False) as tmp:
        tmp_path = tmp.name
        tmp.write("\n".join(chunk))
    try:
        proc = subprocess.run(
            ["hunspell", "-d", ES_DIC_BASE],
            stdin=open(tmp_path, "r", encoding="utf-8"),
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE
        )
        try:
            raw = proc.stdout.decode("utf-8")
        except UnicodeDecodeError:
            raw = proc.stdout.decode("cp1252", errors="replace")
    finally:
        os.remove(tmp_path)

    # Drop leading banner line(s) (e.g. "Hunspell 1.7.x …")
    output_lines = raw.splitlines()
    while output_lines and (output_lines[0].startswith("Hunspell") or
                             not output_lines[0].strip()):
        output_lines.pop(0)

    # Split into blocks: each block = non-empty lines between blank lines
    blocks = []
    current = []
    for line in output_lines:
        if line.strip() == "":
            if current:
                blocks.append(current)
                current = []
        else:
            current.append(line)
    if current:
        blocks.append(current)

    # Pair each block with its input word
    parsed = []
    for word, block in zip(chunk, blocks):
        parsed.append(parse_block(word, block))

    # If hunspell returned fewer blocks than words (shouldn't happen but just in case)
    if len(blocks) < len(chunk):
        for word in chunk[len(blocks):]:
            parsed.append((word, None, "no_block"))

    return parsed

# ── Run (chunked + parallel) ──────────────────────────────────────────────────
chunks = [words[i:i+CHUNK_SIZE] for i in range(0, len(words), CHUNK_SIZE)]
print(f"Running {len(chunks)} chunk(s) with up to {NUM_WORKERS} parallel workers...")

chunk_results = [None] * len(chunks)
with ThreadPoolExecutor(max_workers=NUM_WORKERS) as pool:
    futures = {pool.submit(run_hunspell_chunk, chunk): idx
               for idx, chunk in enumerate(chunks)}
    for fut in as_completed(futures):
        idx = futures[fut]
        chunk_results[idx] = fut.result()

# Flatten
all_parsed = [entry for chunk in chunk_results for entry in chunk]

# ── Build DataFrame ───────────────────────────────────────────────────────────
df_spellcheck = pd.DataFrame(all_parsed, columns=["word", "isCorrect", "suggestions"])

# ── Save CSV ──────────────────────────────────────────────────────────────────
df_spellcheck.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")
print(f"Saved CSV: {OUTPUT_CSV}  ({len(df_spellcheck)} rows)")

# ── Also save the raw flat text ───────────────────────────────────────────────
full_output = "\n".join(
    f"{row.word}\t{'OK' if row.isCorrect else 'MISS'}\t{row.suggestions}"
    for row in df_spellcheck.itertuples()
)
with open(OUTPUT_TXT, "w", encoding="utf-8") as f:
    f.write(full_output)
print(f"Saved TXT: {OUTPUT_TXT}")

# ── Quick summary ─────────────────────────────────────────────────────────────
n_correct   = df_spellcheck["isCorrect"].eq(True).sum()
n_incorrect = df_spellcheck["isCorrect"].eq(False).sum()
n_unknown   = df_spellcheck["isCorrect"].isna().sum()
print(f"\n{'─'*40}")
print(f"  Correct   : {n_correct:>6} ({n_correct/len(df_spellcheck)*100:.1f}%)")
print(f"  Misspelled: {n_incorrect:>6} ({n_incorrect/len(df_spellcheck)*100:.1f}%)")
print(f"  Unknown   : {n_unknown:>6}")
print(f"{'─'*40}")


In [ ]:
# ── Glance at correct words ───────────────────────────────────────────────────
correct_words_df = df_spellcheck[df_spellcheck["isCorrect"] == True]
correct_words = df_spellcheck[df_spellcheck["isCorrect"] == True]["word"]
print(f"Correct words ({len(correct_words)} total) – first 50:")
print(", ".join(correct_words.head(50).tolist()))


In [ ]:
df_spellcheck

In [ ]:
correct_words_df

In [ ]:

# Search for lines starting with exactly "über/" and lines starting with "ladung"/"Ladung"
DIC_FILE = r"c:\Users\Nelso\Documents\MundoDoce\TB2dic\dics\de_dic\de_DE_frami.dic"

uber_slash_hits = []
ladung_hits = []

with open(DIC_FILE, encoding="iso-8859-1") as f:
    for lineno, line in enumerate(f, start=1):
        stripped = line.rstrip("\n")
        if stripped.startswith("über/"):
            uber_slash_hits.append((lineno, stripped))
        if stripped.startswith("ladung") or stripped.startswith("Ladung"):
            ladung_hits.append((lineno, stripped))

print(f"=== Lines starting with exactly 'über/' ({len(uber_slash_hits)} hits) ===")
for lineno, text in uber_slash_hits:
    print(f"  L{lineno:>7}: {text}")
if not uber_slash_hits:
    print("  (no matches)")

print()
print(f"=== Lines starting with 'ladung' or 'Ladung' ({len(ladung_hits)} hits) ===")
for lineno, text in ladung_hits:
    print(f"  L{lineno:>7}: {text}")
if not ladung_hits:
    print("  (no matches)")


# B. Verify ANK_dic with Hunspell exe
Use the filtered tokens as input and spellcheck them using Hunspell based on ANK_dic as custom dictionary. This would ensure that all the listed words and affix patterns work correctly in production.
Example: If Uk'Not'Allag is in the ANK_dic.dic, it should be flagged as correct when analyzing it with Hunspell. Some errors might happen.

In [ ]:
import subprocess
import tempfile
import os
import pandas as pd
from concurrent.futures import ThreadPoolExecutor, as_completed

es_ANK_dic_PATH = os.path.join(DIC_FOLDER, "es_dic", "es_ANK", "es_ANK")
# ↑ Points to the base name (no extension) — Hunspell will load es_ANK.dic + es_ANK.aff

# ── Config ────────────────────────────────────────────────────────────────────
_CHECK_GAME = "DOFUS"
_CHECK_LANG = "es"
_CHECK_KEY  = f"{_CHECK_GAME}_{_CHECK_LANG}"
INPUT_DIC   = os.path.abspath(os.path.join(INTERMEDIARY_DIR, f"{_CHECK_KEY}_filtered_tokens.dic"))
OUTPUT_TXT  = os.path.abspath(os.path.join(INTERMEDIARY_DIR, f"{_CHECK_KEY}_ANK_hunspell_check.txt"))
OUTPUT_CSV  = os.path.abspath(os.path.join(INTERMEDIARY_DIR, f"{_CHECK_KEY}_ANK_hunspell_check.csv"))

ES_ANK_DIC_BASE = os.path.abspath(es_ANK_dic_PATH)  # …/es_dic/es_ANK/es_ANK

print(f"Using ANK dictionary base: {ES_ANK_DIC_BASE}")
assert os.path.exists(ES_ANK_DIC_BASE + ".dic"), f"Missing: {ES_ANK_DIC_BASE}.dic"
assert os.path.exists(ES_ANK_DIC_BASE + ".aff"), f"Missing: {ES_ANK_DIC_BASE}.aff"

SAMPLE_ONLY  = False   # ← Full run
SAMPLE_SIZE  = 400     # only used when SAMPLE_ONLY = True
NUM_WORKERS  = 10      # parallel hunspell processes
CHUNK_SIZE   = 500     # words per chunk

# ── Load words (skip first line = word count header) ─────────────────────────
with open(INPUT_DIC, "r", encoding="utf-8") as f:
    lines = f.readlines()

all_words = [ln.strip() for ln in lines[1:] if ln.strip()]
words = all_words[:SAMPLE_SIZE] if SAMPLE_ONLY else all_words
print(f"{'Sample' if SAMPLE_ONLY else 'Full'}: {len(words)} / {len(all_words)} words loaded")

# ── Helper: parse one blank-line-delimited block for a single word ────────────
def parse_block(word: str, block_lines: list) -> tuple:
    all_correct = True
    all_suggs   = []
    for line in block_lines:
        r = line.strip()
        if not r:
            continue
        if r.startswith(("*", "+", "-")):
            pass  # sub-word OK
        elif r.startswith("&"):
            all_correct = False
            parts = r.split(":", 1)
            if len(parts) > 1:
                all_suggs.append(parts[1].strip())
        elif r.startswith("#"):
            all_correct = False
        else:
            return (word, None, r)
    return (word, all_correct, " | ".join(all_suggs))

# ── Helper: run hunspell on a chunk ──────────────────────────────────────────
def run_hunspell_chunk(chunk: list) -> list:
    with tempfile.NamedTemporaryFile(mode="w", encoding="utf-8",
                                     suffix=".txt", delete=False) as tmp:
        tmp_path = tmp.name
        tmp.write("\n".join(chunk))
    try:
        proc = subprocess.run(
            ["hunspell", "-d", ES_ANK_DIC_BASE],
            stdin=open(tmp_path, "r", encoding="utf-8"),
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE
        )
        try:
            raw = proc.stdout.decode("utf-8")
        except UnicodeDecodeError:
            raw = proc.stdout.decode("cp1252", errors="replace")
    finally:
        os.remove(tmp_path)

    output_lines = raw.splitlines()
    while output_lines and (output_lines[0].startswith("Hunspell") or
                             not output_lines[0].strip()):
        output_lines.pop(0)

    blocks, current = [], []
    for line in output_lines:
        if line.strip() == "":
            if current:
                blocks.append(current)
                current = []
        else:
            current.append(line)
    if current:
        blocks.append(current)

    parsed = [parse_block(word, block) for word, block in zip(chunk, blocks)]
    if len(blocks) < len(chunk):
        parsed += [(word, None, "no_block") for word in chunk[len(blocks):]]
    return parsed

# ── Run (chunked + parallel) ──────────────────────────────────────────────────
chunks = [words[i:i+CHUNK_SIZE] for i in range(0, len(words), CHUNK_SIZE)]
print(f"Running {len(chunks)} chunk(s) with up to {NUM_WORKERS} parallel workers...")

chunk_results = [None] * len(chunks)
with ThreadPoolExecutor(max_workers=NUM_WORKERS) as pool:
    futures = {pool.submit(run_hunspell_chunk, chunk): idx
               for idx, chunk in enumerate(chunks)}
    for fut in as_completed(futures):
        idx = futures[fut]
        chunk_results[idx] = fut.result()

all_parsed = [entry for chunk in chunk_results for entry in chunk]

# ── Build DataFrame ───────────────────────────────────────────────────────────
df_ank = pd.DataFrame(all_parsed, columns=["word", "isCorrect", "suggestions"])

df_ank.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")
print(f"Saved CSV: {OUTPUT_CSV}  ({len(df_ank)} rows)")

full_output = "\n".join(
    f"{row.word}\t{'OK' if row.isCorrect else 'MISS'}\t{row.suggestions}"
    for row in df_ank.itertuples()
)
with open(OUTPUT_TXT, "w", encoding="utf-8") as f:
    f.write(full_output)
print(f"Saved TXT: {OUTPUT_TXT}")

# ── Quick summary ─────────────────────────────────────────────────────────────
n_correct   = df_ank["isCorrect"].eq(True).sum()
n_incorrect = df_ank["isCorrect"].eq(False).sum()
n_unknown   = df_ank["isCorrect"].isna().sum()
print(f"\n{'─'*40}")
print(f"  Correct   : {n_correct:>6} ({n_correct/len(df_ank)*100:.1f}%)")
print(f"  Misspelled: {n_incorrect:>6} ({n_incorrect/len(df_ank)*100:.1f}%)")
print(f"  Unknown   : {n_unknown:>6}")
print(f"{'─'*40}")

In [ ]:
# df of misspelled
df_misspelled_es_ANK = (
    df_ank[df_ank["isCorrect"] == False]
    .reset_index(drop=True)
)

print(f"Misspelled words (es_ANK): {len(df_misspelled_es_ANK)} / {len(df_ank)} "
      f"({len(df_misspelled_es_ANK)/len(df_ank)*100:.1f}%)")
df_misspelled_es_ANK.head(20)
